In [ ]:
import os, json

# Kaggle credentials
kaggle_creds = {
    "username": "anuragtiwari575",   # yahan tera Kaggle username (profile se)
    "key": "KGAT_ddd848c4c2eec057ff66433714b9fe50"         # yahan nayi token (purani revoke karke)
}

os.makedirs("/root/.kaggle", exist_ok=True)
with open("/root/.kaggle/kaggle.json", "w") as f:
    json.dump(kaggle_creds, f)
os.chmod("/root/.kaggle/kaggle.json", 0o600)

print("✅ Kaggle credentials saved")

!pip install -q kaggle
!kaggle --version

In [2]:
import os

# Target folder
DOWNLOAD_DIR = "/workspace/liver_tumor_project/data/raw/LiTS"
os.makedirs(DOWNLOAD_DIR, exist_ok=True)
os.chdir(DOWNLOAD_DIR)

print(f"📁 Downloading to: {DOWNLOAD_DIR}")
print(f"📦 Dataset: javariatahir/litstrain-val (~53.66 GB)")
print(f"⏱️  Expected time: 20-40 min\n")
print("=" * 60)

# Download command
!kaggle datasets download -d javariatahir/litstrain-val

print("\n" + "=" * 60)
print("✅ Download complete!")
print("\n📊 Files in folder:")
!ls -lh

📁 Downloading to: /workspace/liver_tumor_project/data/raw/LiTS
📦 Dataset: javariatahir/litstrain-val (~53.66 GB)
⏱️  Expected time: 20-40 min

Dataset URL: https://www.kaggle.com/datasets/javariatahir/litstrain-val
License(s): unknown
^C
User cancelled operation

✅ Download complete!

📊 Files in folder:
total 16G
drwxr-xr-x 7 root root 4.0K Aug 12 09:00 'LiTS(train_test)'
-rw-r--r-- 1 root root  16G Jan 27  2023  litstrain-val.zip


In [4]:
import os
os.chdir("/workspace/liver_tumor_project/data/raw/LiTS")

print("💾 Space check BEFORE extraction:")
!df -h /workspace

print("\n" + "="*60)
print("📦 Extracting litstrain-val.zip...")
print("⏱️  Yeh 5-15 min lag sakta hai (bade zip files ke liye)")
print("="*60 + "\n")

# Extract silently
!unzip -q litstrain-val.zip

print("✅ Extraction complete!\n")

print("📁 Extracted content:")
!ls -la

print("\n💾 Space check AFTER extraction:")
!df -h /workspace

print("\n📊 Size of extracted content:")
!du -sh */ 2>/dev/null

💾 Space check BEFORE extraction:
Filesystem                    Size  Used Avail Use% Mounted on
172.16.225.216:/gaurav_goyal  501G  295G  206G  59% /workspace

📦 Extracting litstrain-val.zip...
⏱️  Yeh 5-15 min lag sakta hai (bade zip files ke liye)

replace LiTS(train_test)/test_CT/volume-111.nii? [y]es, [n]o, [A]ll, [N]one, [r]ename: ✅ Extraction complete!

📁 Extracted content:
total 8
drwxr-xr-x 2 root root 4096 Aug 13 13:47 .
drwxr-xr-x 3 root root 4096 Aug 11 15:25 ..

💾 Space check AFTER extraction:
Filesystem                    Size  Used Avail Use% Mounted on
172.16.225.216:/gaurav_goyal  501G  278G  223G  56% /workspace

📊 Size of extracted content:


In [ ]:
import os
import subprocess

DATA_DIR = "/workspace/liver_tumor_project/data/raw/LiTS"
os.chdir(DATA_DIR)

print("=" * 70)
print("📊 EXTRACTION VERIFICATION")
print("=" * 70)

print("\n📁 Current directory contents:")
!ls -la

print("\n💾 Total data size:")
!du -sh . 
!df -h /workspace

# Find kya extract hua
print("\n🔍 Searching for extracted structure...")
result = subprocess.run(['find', '.', '-maxdepth', '3', '-type', 'd'], 
                       capture_output=True, text=True)
print(result.stdout)

print("\n🔢 File counts by type:")
!find . -name "*.nii" 2>/dev/null | wc -l | xargs echo ".nii files:"
!find . -name "*.nii.gz" 2>/dev/null | wc -l | xargs echo ".nii.gz files:"
!find . -name "volume*" 2>/dev/null | wc -l | xargs echo "volume* files:"
!find . -name "segmentation*" 2>/dev/null | wc -l | xargs echo "segmentation* files:"

# Sample files check
print("\n📄 Sample files (first 5):")
!find . -name "*.nii*" 2>/dev/null | head -5

In [ ]:
import os

# Zip delete karke 15.6 GB free karo
os.chdir("/workspace/liver_tumor_project/data/raw/LiTS")

zip_size = os.path.getsize("litstrain-val.zip") / (1024**3)
print(f"🗑️  Deleting zip file ({zip_size:.1f} GB)...")

!rm litstrain-val.zip

print("✅ Zip deleted\n")
print("💾 Space now:")
!df -h /workspace

In [ ]:
import os
import glob

BASE = "/workspace/liver_tumor_project/data/raw/LiTS/LiTS(train_test)"

print("=" * 70)
print("📊 DETAILED VERIFICATION")
print("=" * 70)

# Har folder ka count
for folder in ["train_CT", "train_mask", "test_CT", "test_mask"]:
    path = os.path.join(BASE, folder)
    files = sorted(glob.glob(os.path.join(path, "*.nii")))
    print(f"\n📁 {folder}: {len(files)} files")
    if files:
        # Volume indices dekhkr range
        indices = [int(os.path.basename(f).split('-')[1].split('.')[0]) for f in files]
        print(f"   Range: volume-{min(indices)} to volume-{max(indices)}")
        print(f"   First 3: {[os.path.basename(f) for f in files[:3]]}")
        print(f"   Last 3:  {[os.path.basename(f) for f in files[-3:]]}")
        
        # Size stats
        sizes = [os.path.getsize(f) / (1024**2) for f in files[:5]]
        print(f"   Sample sizes: {[f'{s:.1f}MB' for s in sizes]}")

# Naming pattern check kar
print("\n" + "=" * 70)
print("🔍 NAMING PATTERN CHECK")
print("=" * 70)
train_ct = sorted(os.listdir(os.path.join(BASE, "train_CT")))[:3]
train_mask = sorted(os.listdir(os.path.join(BASE, "train_mask")))[:3]
print(f"\ntrain_CT samples:    {train_ct}")
print(f"train_mask samples:  {train_mask}")

test_ct = sorted(os.listdir(os.path.join(BASE, "test_CT")))[:3]
test_mask = sorted(os.listdir(os.path.join(BASE, "test_mask")))[:3]
print(f"\ntest_CT samples:     {test_ct}")
print(f"test_mask samples:   {test_mask}")

In [ ]:
# Install nibabel (agar nahi hai)
!pip install -q nibabel

import nibabel as nib
import numpy as np
import os

# Ek sample volume + mask load karke check karte hain
sample_vol = "/workspace/liver_tumor_project/data/raw/LiTS/LiTS(train_test)/train_CT/volume-0.nii"
sample_mask = "/workspace/liver_tumor_project/data/raw/LiTS/LiTS(train_test)/train_mask/segmentation-0.nii"

# Check kaunsa naming actually hai
if not os.path.exists(sample_mask):
    # Try alternative naming
    for f in sorted(os.listdir("/workspace/liver_tumor_project/data/raw/LiTS/LiTS(train_test)/train_mask/"))[:3]:
        print(f"Actual mask name: {f}")
    sample_mask = "/workspace/liver_tumor_project/data/raw/LiTS/LiTS(train_test)/train_mask/" + sorted(os.listdir("/workspace/liver_tumor_project/data/raw/LiTS/LiTS(train_test)/train_mask/"))[0]

print("=" * 70)
print(f"📥 Loading sample volume: {os.path.basename(sample_vol)}")
print("=" * 70)

# Volume load karo
img_vol = nib.load(sample_vol)
vol_data = img_vol.get_fdata()

print(f"\n📐 Volume shape:        {vol_data.shape}  (H x W x Slices)")
print(f"📏 Voxel spacing (mm):  {img_vol.header.get_zooms()}")
print(f"📊 Intensity range:     [{vol_data.min():.1f}, {vol_data.max():.1f}] HU")
print(f"📈 Mean intensity:      {vol_data.mean():.1f} HU")
print(f"💾 Memory usage:        {vol_data.nbytes / (1024**2):.1f} MB")

# Mask load karo
print(f"\n📥 Loading corresponding mask: {os.path.basename(sample_mask)}")
img_mask = nib.load(sample_mask)
mask_data = img_mask.get_fdata()

print(f"\n📐 Mask shape:          {mask_data.shape}")
unique_labels = np.unique(mask_data)
print(f"🏷️  Unique labels:       {unique_labels}")
print(f"   → 0 = background")
print(f"   → 1 = liver")
print(f"   → 2 = tumor")

# Class distribution
total_voxels = mask_data.size
for label in unique_labels:
    count = np.sum(mask_data == label)
    pct = count / total_voxels * 100
    name = {0: "Background", 1: "Liver", 2: "Tumor"}.get(int(label), f"Label {int(label)}")
    print(f"\n   {name:12s}: {count:>12,} voxels ({pct:5.2f}%)")

# Slices with tumor
if 2 in unique_labels:
    tumor_slices = [i for i in range(mask_data.shape[2]) if np.any(mask_data[:,:,i] == 2)]
    print(f"\n🎯 Slices with tumor: {len(tumor_slices)}/{mask_data.shape[2]}")
    print(f"   Range: slice {min(tumor_slices)} to {max(tumor_slices)}")

In [ ]:
import nibabel as nib
import numpy as np
import os
import glob
from tqdm import tqdm
import pandas as pd

BASE = "/workspace/liver_tumor_project/data/raw/LiTS/LiTS(train_test)"

print("=" * 70)
print("📊 COMPREHENSIVE DATASET STATISTICS (131 training volumes)")
print("=" * 70)
print("⏱️  Yeh ~2-3 min lag sakta hai...\n")

# Collect stats
stats = []
train_ct_files = sorted(glob.glob(os.path.join(BASE, "train_CT", "*.nii")))
train_mask_files = sorted(glob.glob(os.path.join(BASE, "train_mask", "*.nii")))

for vol_path, mask_path in tqdm(list(zip(train_ct_files, train_mask_files)), 
                                 desc="Scanning volumes"):
    try:
        # Load headers (fast, no data loading)
        vol_img = nib.load(vol_path)
        mask_img = nib.load(mask_path)
        
        shape = vol_img.shape
        spacing = vol_img.header.get_zooms()
        
        # Load mask only (smaller)
        mask_data = mask_img.get_fdata()
        
        # Class counts
        total = mask_data.size
        liver_count = np.sum(mask_data >= 1)  # liver + tumor
        tumor_count = np.sum(mask_data == 2)
        
        # Tumor-containing slices
        tumor_slices = np.sum(np.any(mask_data == 2, axis=(0, 1)))
        liver_slices = np.sum(np.any(mask_data >= 1, axis=(0, 1)))
        
        stats.append({
            'name': os.path.basename(vol_path),
            'shape_x': shape[0],
            'shape_y': shape[1],
            'shape_z': shape[2],
            'spacing_x': round(spacing[0], 3),
            'spacing_y': round(spacing[1], 3),
            'spacing_z': round(spacing[2], 3),
            'total_voxels_M': round(total / 1e6, 1),
            'liver_pct': round(liver_count / total * 100, 3),
            'tumor_pct': round(tumor_count / total * 100, 4),
            'liver_slices': int(liver_slices),
            'tumor_slices': int(tumor_slices),
            'has_tumor': tumor_count > 0
        })
    except Exception as e:
        print(f"❌ Error with {os.path.basename(vol_path)}: {e}")

df = pd.DataFrame(stats)

# Save CSV
csv_path = "/workspace/liver_tumor_project/data/splits/dataset_stats.csv"
os.makedirs(os.path.dirname(csv_path), exist_ok=True)
df.to_csv(csv_path, index=False)
print(f"\n💾 Saved to: {csv_path}")

# Summary statistics
print("\n" + "=" * 70)
print("📐 SHAPE STATISTICS")
print("=" * 70)
print(df[['shape_x', 'shape_y', 'shape_z']].describe().round(1))

print("\n" + "=" * 70)
print("📏 SPACING STATISTICS (mm)")
print("=" * 70)
print(df[['spacing_x', 'spacing_y', 'spacing_z']].describe().round(3))

print("\n" + "=" * 70)
print("🏷️  CLASS DISTRIBUTION")
print("=" * 70)
print(f"Volumes with tumor:      {df['has_tumor'].sum()}/{len(df)}")
print(f"Volumes without tumor:   {(~df['has_tumor']).sum()}/{len(df)}")
print(f"\nAvg liver %:  {df['liver_pct'].mean():.3f}%  (std: {df['liver_pct'].std():.3f})")
print(f"Avg tumor %:  {df['tumor_pct'].mean():.4f}%  (std: {df['tumor_pct'].std():.4f})")
print(f"\nAvg tumor slices per volume:  {df['tumor_slices'].mean():.1f}")
print(f"Avg total slices per volume:  {df['shape_z'].mean():.1f}")

print("\n" + "=" * 70)
print("🎯 RESAMPLING TARGET SUGGESTIONS")
print("=" * 70)
print(f"Median spacing:  ({df['spacing_x'].median():.3f}, {df['spacing_y'].median():.3f}, {df['spacing_z'].median():.3f})")
print(f"Mean spacing:    ({df['spacing_x'].mean():.3f}, {df['spacing_y'].mean():.3f}, {df['spacing_z'].mean():.3f})")
print(f"Suggested:       (1.5, 1.5, 2.0) mm")

In [ ]:
import os
import glob
import nibabel as nib

BASE = "/workspace/liver_tumor_project/data/raw/LiTS/LiTS(train_test)"

print("=" * 70)
print("🔍 FULL DATASET AUDIT — 4 FOLDERS + MASK VERIFICATION")
print("=" * 70)

# Sab 4 folders check karo
folders = ["train_CT", "train_mask", "test_CT", "test_mask"]

folder_data = {}
for folder in folders:
    path = os.path.join(BASE, folder)
    if not os.path.exists(path):
        print(f"\n❌ {folder}: FOLDER MISSING!")
        continue
    
    files = sorted(glob.glob(os.path.join(path, "*.nii")))
    total_size_gb = sum(os.path.getsize(f) for f in files) / (1024**3)
    
    # Extract indices from filenames
    indices = []
    for f in files:
        name = os.path.basename(f)
        # volume-N.nii or segmentation-N.nii
        try:
            idx = int(name.split('-')[1].split('.')[0])
            indices.append(idx)
        except:
            pass
    
    folder_data[folder] = {
        'count': len(files),
        'size_gb': total_size_gb,
        'indices': sorted(indices),
        'first_3': [os.path.basename(f) for f in files[:3]],
        'last_3': [os.path.basename(f) for f in files[-3:]]
    }
    
    print(f"\n📁 {folder}")
    print(f"   Count:      {len(files)} files")
    print(f"   Total size: {total_size_gb:.2f} GB")
    print(f"   Index range: {min(indices) if indices else 'N/A'} to {max(indices) if indices else 'N/A'}")
    print(f"   First 3: {[os.path.basename(f) for f in files[:3]]}")
    print(f"   Last 3:  {[os.path.basename(f) for f in files[-3:]]}")

# Pairing verification
print("\n" + "=" * 70)
print("🔗 VOLUME ↔ MASK PAIRING CHECK")
print("=" * 70)

# Train pairing
train_ct_idx = set(folder_data['train_CT']['indices'])
train_mask_idx = set(folder_data['train_mask']['indices'])

print(f"\n🎓 TRAIN:")
print(f"   Volumes: {len(train_ct_idx)} unique indices")
print(f"   Masks:   {len(train_mask_idx)} unique indices")

missing_masks_train = train_ct_idx - train_mask_idx
missing_vols_train = train_mask_idx - train_ct_idx

if missing_masks_train:
    print(f"   ⚠️  Volumes without masks: {sorted(missing_masks_train)}")
else:
    print(f"   ✅ All train volumes have masks")

if missing_vols_train:
    print(f"   ⚠️  Masks without volumes: {sorted(missing_vols_train)}")

# Test pairing
test_ct_idx = set(folder_data['test_CT']['indices'])
test_mask_idx = set(folder_data['test_mask']['indices'])

print(f"\n🎯 TEST:")
print(f"   Volumes: {len(test_ct_idx)} unique indices")
print(f"   Masks:   {len(test_mask_idx)} unique indices")

missing_masks_test = test_ct_idx - test_mask_idx
missing_vols_test = test_mask_idx - test_ct_idx

if missing_masks_test:
    print(f"   ⚠️  Volumes without masks: {sorted(missing_masks_test)}")
else:
    print(f"   ✅ All test volumes have masks")

if missing_vols_test:
    print(f"   ⚠️  Masks without volumes: {sorted(missing_vols_test)}")

# Total summary
print("\n" + "=" * 70)
print("📊 GRAND TOTAL")
print("=" * 70)
total_files = sum(fd['count'] for fd in folder_data.values())
total_size = sum(fd['size_gb'] for fd in folder_data.values())
total_pairs = len(train_ct_idx & train_mask_idx) + len(test_ct_idx & test_mask_idx)

print(f"Total files:         {total_files}")
print(f"Total size:          {total_size:.2f} GB")
print(f"Total volume-mask pairs: {total_pairs}")
print(f"   ├─ Train pairs: {len(train_ct_idx & train_mask_idx)}")
print(f"   └─ Test pairs:  {len(test_ct_idx & test_mask_idx)}")

# Verify shape match for one sample from each folder
print("\n" + "=" * 70)
print("🔬 SHAPE VERIFICATION (sample pairs)")
print("=" * 70)

# One sample train pair
if train_ct_idx & train_mask_idx:
    idx = min(train_ct_idx & train_mask_idx)
    vol = nib.load(os.path.join(BASE, "train_CT", f"volume-{idx}.nii"))
    mask = nib.load(os.path.join(BASE, "train_mask", f"segmentation-{idx}.nii"))
    match = "✅ MATCH" if vol.shape == mask.shape else "❌ MISMATCH"
    print(f"\n🎓 Train volume-{idx}:")
    print(f"   CT shape:   {vol.shape}, spacing: {vol.header.get_zooms()}")
    print(f"   Mask shape: {mask.shape}, spacing: {mask.header.get_zooms()}")
    print(f"   {match}")

# One sample test pair
if test_ct_idx & test_mask_idx:
    idx = min(test_ct_idx & test_mask_idx)
    vol = nib.load(os.path.join(BASE, "test_CT", f"volume-{idx}.nii"))
    mask = nib.load(os.path.join(BASE, "test_mask", f"segmentation-{idx}.nii"))
    match = "✅ MATCH" if vol.shape == mask.shape else "❌ MISMATCH"
    print(f"\n🎯 Test volume-{idx}:")
    print(f"   CT shape:   {vol.shape}, spacing: {vol.header.get_zooms()}")
    print(f"   Mask shape: {mask.shape}, spacing: {mask.header.get_zooms()}")
    print(f"   {match}")

print("\n" + "=" * 70)
print("✅ AUDIT COMPLETE")
print("=" * 70)

In [ ]:
import numpy as np
import pandas as pd
import json
import os
import glob

# Fixed seed for reproducibility
SEED = 42
np.random.seed(SEED)

# Load stats
df = pd.read_csv("/workspace/liver_tumor_project/data/splits/dataset_stats.csv")

print("=" * 70)
print("📊 STRATIFIED SPLIT — 111 TRAINING POOL")
print("=" * 70)
print(f"\n📁 Training pool: {len(df)} volumes")
print(f"   ├─ With tumor:    {df['has_tumor'].sum()}")
print(f"   └─ Without tumor: {(~df['has_tumor']).sum()}")

# Separate tumor and no-tumor
tumor_vols = df[df['has_tumor']].copy().sample(frac=1, random_state=SEED).reset_index(drop=True)
no_tumor_vols = df[~df['has_tumor']].copy().sample(frac=1, random_state=SEED).reset_index(drop=True)

# 85/15 stratified split
# From 101 tumor: 86 train, 15 val
# From 10 no-tumor: 9 train, 1 val
n_train_tumor = int(round(len(tumor_vols) * 0.85))    # 86
n_train_no_tumor = int(round(len(no_tumor_vols) * 0.85))  # 8-9

train_vols = pd.concat([
    tumor_vols.iloc[:n_train_tumor],
    no_tumor_vols.iloc[:n_train_no_tumor]
]).sample(frac=1, random_state=SEED).reset_index(drop=True)

val_vols = pd.concat([
    tumor_vols.iloc[n_train_tumor:],
    no_tumor_vols.iloc[n_train_no_tumor:]
]).sample(frac=1, random_state=SEED).reset_index(drop=True)

# Kaggle ka pre-defined test set
BASE = "/workspace/liver_tumor_project/data/raw/LiTS/LiTS(train_test)"
test_ct_files = sorted(glob.glob(os.path.join(BASE, "test_CT", "*.nii")),
                       key=lambda x: int(os.path.basename(x).split('-')[1].split('.')[0]))
test_names = [os.path.basename(f) for f in test_ct_files]

# Final splits
print("\n" + "=" * 70)
print("📊 FINAL SPLITS")
print("=" * 70)
print(f"\n🎓 TRAIN: {len(train_vols)} volumes")
print(f"   ├─ With tumor:    {train_vols['has_tumor'].sum()}")
print(f"   └─ Without tumor: {(~train_vols['has_tumor']).sum()}")

print(f"\n✅ VAL: {len(val_vols)} volumes")
print(f"   ├─ With tumor:    {val_vols['has_tumor'].sum()}")
print(f"   └─ Without tumor: {(~val_vols['has_tumor']).sum()}")

print(f"\n🎯 TEST (Kaggle held-out): {len(test_names)} volumes")
print(f"   Range: {test_names[0]} to {test_names[-1]}")

# Save splits to JSON
splits = {
    "seed": SEED,
    "strategy": "stratified_by_tumor_presence",
    "split_ratio": "85/15 from 111 train pool + 20 Kaggle test",
    "train": {
        "count": int(len(train_vols)),
        "with_tumor": int(train_vols['has_tumor'].sum()),
        "without_tumor": int((~train_vols['has_tumor']).sum()),
        "volumes": train_vols['name'].tolist()
    },
    "val": {
        "count": int(len(val_vols)),
        "with_tumor": int(val_vols['has_tumor'].sum()),
        "without_tumor": int((~val_vols['has_tumor']).sum()),
        "volumes": val_vols['name'].tolist()
    },
    "test": {
        "count": int(len(test_names)),
        "source": "Kaggle pre-defined test set (test_CT/, test_mask/)",
        "volumes": test_names
    }
}

split_path = "/workspace/liver_tumor_project/data/splits/fold_splits.json"
with open(split_path, 'w') as f:
    json.dump(splits, f, indent=2)

print(f"\n💾 Splits saved to: {split_path}")

# Sample outputs
print(f"\n📄 Sample train volumes:  {splits['train']['volumes'][:5]}")
print(f"📄 Sample val volumes:    {splits['val']['volumes'][:5]}")
print(f"📄 Sample test volumes:   {splits['test']['volumes'][:5]}")

# Sanity check totals
total = splits['train']['count'] + splits['val']['count'] + splits['test']['count']
print(f"\n✅ Total volumes accounted for: {total}/131")

In [ ]:
print("=" * 70)
print("🧹 STEP 1: CLEAN UNINSTALL")
print("=" * 70)
print("⚠️  Purani buggy versions remove ki jaa rahi hain...\n")

# Remove broken PyTorch stack
print("Removing PyTorch and CUDA libraries...")
!pip uninstall -y torch torchvision torchaudio 2>&1 | tail -3

# Remove Mamba (need to reinstall matched to new torch)
print("\nRemoving Mamba packages...")
!pip uninstall -y mamba-ssm causal-conv1d 2>&1 | tail -3

# Remove ALL nvidia CUDA cu13 packages (broken)
print("\nRemoving broken CUDA 13 packages...")
!pip uninstall -y \
    nvidia-cublas nvidia-cuda-cupti nvidia-cuda-nvdisasm \
    nvidia-cuda-nvrtc nvidia-cuda-runtime nvidia-cudnn-cu13 \
    nvidia-cufft nvidia-cufile nvidia-nccl-cu13 \
    nvidia-nvshmem-cu13 nvidia-cusparselt-cu13 \
    cuda-bindings cuda-toolkit \
    2>&1 | tail -3

# Also remove CUDA 12 if broken
print("\nRemoving old CUDA 12 packages (will reinstall)...")
!pip uninstall -y \
    nvidia-cublas-cu12 nvidia-cuda-cupti-cu12 nvidia-cuda-nvrtc-cu12 \
    nvidia-cuda-runtime-cu12 nvidia-cudnn-cu12 nvidia-cufft-cu12 \
    2>&1 | tail -3

# Remove nvidia-cudnn-frontend
!pip uninstall -y nvidia-cudnn-frontend 2>&1 | tail -3

print("\n" + "=" * 70)
print("✅ Uninstall complete")
print("=" * 70)

# Check what remains
print("\n🔍 Checking remaining nvidia/torch packages:")
!pip list 2>&1 | grep -iE "torch|nvidia|nccl|mamba|cuda" | head -20

In [ ]:
print("=" * 70)
print("📦 STEP 2: INSTALL STABLE PYTORCH 2.4 + CUDA 12.1")
print("=" * 70)
print("⏱️  Yeh 10-15 min lagega — patience rakh\n")

# Stable PyTorch 2.4 with CUDA 12.1 (well-tested, mature)
!pip install torch==2.4.0 torchvision==0.19.0 torchaudio==2.4.0 \
    --index-url https://download.pytorch.org/whl/cu121 2>&1 | tail -10

print("\n✅ PyTorch installed")

# Verify immediately
print("\n" + "=" * 70)
print("🔍 IMMEDIATE VERIFICATION")
print("=" * 70)

import torch
print(f"PyTorch version:    {torch.__version__}")
print(f"CUDA available:     {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA version:       {torch.version.cuda}")
    print(f"cuDNN version:      {torch.backends.cudnn.version()}")
    print(f"GPU:                {torch.cuda.get_device_name(0)}")
    print(f"GPU memory:         {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
    
    # Quick GPU test
    print("\n🧪 GPU compute test:")
    x = torch.randn(1000, 1000).cuda()
    y = torch.matmul(x, x)
    print(f"   Matrix multiplication: ✅ ({y.shape})")
    print(f"   GPU memory used:       {torch.cuda.memory_allocated() / 1024**2:.1f} MB")

In [ ]:
print("=" * 70)
print("📦 STEP 1: CORE MEDICAL AI STACK")
print("=" * 70)
print("⚠️  NO Mamba install yet — sirf safe packages\n")

# Core packages — no torch conflicts
!pip install -q --no-deps \
    monai==1.4.0 \
    nibabel \
    SimpleITK \
    einops \
    timm

# Their deps (safe ones)
!pip install -q \
    tqdm pandas h5py \
    tensorboard wandb \
    matplotlib seaborn scikit-image \
    scipy scikit-learn

# Common medical utilities
!pip install -q \
    batchgenerators \
    medpy \
    dicom2nifti

print("\n✅ Core installed")

# Verify — CRITICAL step
print("\n" + "=" * 70)
print("🔍 VERIFY PYTORCH STILL WORKS")
print("=" * 70)

import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available:  {torch.cuda.is_available()}")

# Ensure it's still 2.4 (not upgraded)
if not torch.__version__.startswith("2.4"):
    print(f"\n🚨 WARNING: PyTorch changed to {torch.__version__}!")
    print("Kuch package ne torch upgrade kar diya. Stop karo!")
else:
    print("✅ PyTorch 2.4 intact\n")

# Verify MONAI
import monai
print(f"MONAI version: {monai.__version__}")

import nibabel as nib
print(f"Nibabel: {nib.__version__}")

import SimpleITK as sitk
print(f"SimpleITK: {sitk.__version__}")

import einops
print(f"Einops: {einops.__version__}")

print("\n✅ Core stack ready")

In [ ]:
import os
import shutil

OUTPUT_ROOT = "/workspace/liver_tumor_project/data/preprocessed"

# Check how much is there
print("🔍 BEFORE CLEANUP:")
!du -sh {OUTPUT_ROOT}/*/  2>/dev/null

# Count files
for subdir in ["imagesTr", "labelsTr", "imagesTs", "labelsTs"]:
    path = os.path.join(OUTPUT_ROOT, subdir)
    if os.path.exists(path):
        count = len([f for f in os.listdir(path) if f.endswith('.nii.gz')])
        print(f"  {subdir}: {count} files")

# Delete all preprocessed files (keep folder structure)
print("\n🗑️  Deleting old preprocessed files...")
for subdir in ["imagesTr", "labelsTr", "imagesTs", "labelsTs"]:
    path = os.path.join(OUTPUT_ROOT, subdir)
    if os.path.exists(path):
        for f in os.listdir(path):
            if f.endswith('.nii.gz') or f.endswith('.nii'):
                os.remove(os.path.join(path, f))
        print(f"  ✅ Cleaned {subdir}/")

# Also clean old visualizations
VIZ_DIR = "/workspace/liver_tumor_project/results/preprocessing_viz"
if os.path.exists(VIZ_DIR):
    for f in os.listdir(VIZ_DIR):
        os.remove(os.path.join(VIZ_DIR, f))
    print(f"  ✅ Cleaned visualizations")

print("\n🔍 AFTER CLEANUP:")
!du -sh {OUTPUT_ROOT}/*/ 2>/dev/null
print(f"\n💾 Free space:")
!df -h /workspace | tail -1

print("\n✅ Clean slate ready for V2 preprocessing")

In [ ]:
import os
import shutil

OUTPUT_ROOT = "/workspace/liver_tumor_project/data/preprocessed"

print("🧹 CLEANUP — Removing old preprocessed files")
print("=" * 60)

# Check before
print("\n📊 Before cleanup:")
!du -sh {OUTPUT_ROOT}/*/  2>/dev/null

total_deleted = 0
for subdir in ["imagesTr", "labelsTr", "imagesTs", "labelsTs"]:
    path = os.path.join(OUTPUT_ROOT, subdir)
    if os.path.exists(path):
        files = [f for f in os.listdir(path) if f.endswith(('.nii.gz', '.nii'))]
        for f in files:
            os.remove(os.path.join(path, f))
        total_deleted += len(files)
        print(f"  ✅ Cleaned {subdir}/ ({len(files)} files)")

# Clean visualizations
VIZ_DIR = "/workspace/liver_tumor_project/results/preprocessing_viz"
if os.path.exists(VIZ_DIR):
    viz_files = [f for f in os.listdir(VIZ_DIR) if f.endswith('.png')]
    for f in viz_files:
        os.remove(os.path.join(VIZ_DIR, f))
    print(f"  ✅ Cleaned visualizations ({len(viz_files)} PNGs)")

# Old stats files
LOG_DIR = "/workspace/liver_tumor_project/logs/preprocessing"
if os.path.exists(LOG_DIR):
    old_stats = [f for f in os.listdir(LOG_DIR) if f.startswith("preprocessing_stats")]
    for f in old_stats:
        os.remove(os.path.join(LOG_DIR, f))
    print(f"  ✅ Cleaned old stats files ({len(old_stats)} JSONs)")

print(f"\n📊 After cleanup:")
!du -sh {OUTPUT_ROOT}/*/ 2>/dev/null
print(f"\n💾 Free space:")
!df -h /workspace | tail -1

print(f"\n✅ Total {total_deleted} old files removed. Clean slate ready!")

In [ ]:
import os
import shutil
import torch

print("🔍 PRE-FLIGHT CHECK V2")
print("=" * 60)

# Data exists?
data_root = "/workspace/liver_tumor_project/data/raw/LiTS/LiTS(train_test)"
for folder in ["train_CT", "train_mask", "test_CT", "test_mask"]:
    path = os.path.join(data_root, folder)
    count = len(os.listdir(path)) if os.path.exists(path) else 0
    status = "✅" if count > 0 else "❌"
    print(f"  {status} {folder}: {count} files")

# Splits file
splits_path = "/workspace/liver_tumor_project/data/splits/fold_splits.json"
print(f"\n  {'✅' if os.path.exists(splits_path) else '❌'} Splits file")

# Output should be empty
output_root = "/workspace/liver_tumor_project/data/preprocessed"
for subdir in ["imagesTr", "labelsTr", "imagesTs", "labelsTs"]:
    path = os.path.join(output_root, subdir)
    if os.path.exists(path):
        count = len([f for f in os.listdir(path) if f.endswith(('.nii.gz', '.nii'))])
        status = "✅ empty" if count == 0 else f"⚠️  has {count} files!"
        print(f"  {status} {subdir}")

# Libraries
print(f"\n  Libraries:")
try:
    from scipy import ndimage
    print(f"    ✅ scipy.ndimage")
except: print(f"    ❌ scipy.ndimage")

try:
    from monai.transforms import Compose
    print(f"    ✅ monai.transforms")
except: print(f"    ❌ monai.transforms")

try:
    import nibabel
    print(f"    ✅ nibabel {nibabel.__version__}")
except: print(f"    ❌ nibabel")

try:
    import matplotlib
    print(f"    ✅ matplotlib")
except: print(f"    ❌ matplotlib")

# PyTorch
print(f"\n  PyTorch: {torch.__version__}")
print(f"  CUDA:    {torch.cuda.is_available()}")

# Disk space
free_gb = shutil.disk_usage("/workspace").free / (1024**3)
print(f"\n  Free space: {free_gb:.1f} GB")
print(f"  Needed:     ~15 GB")
print(f"  Status:     {'✅ OK' if free_gb > 20 else '⚠️  LOW'}")

print("\n" + "=" * 60)
print("Sab ✅ hai to next cell mein V2 script chala aur so ja! 😴")

In [ ]:
"""
=========================================================================
OVERNIGHT PREPROCESSING V2 — AGGRESSIVE BACKGROUND REMOVAL
=========================================================================
- 3-layer background removal (body mask + liver ROI + slice filter)
- Consistent V2 preprocessing for all 131 volumes
- Full logging + visualizations + stats
- Safe: warns if old files exist
=========================================================================
"""

import os
import sys
import json
import time
import traceback
from datetime import datetime

import numpy as np
import nibabel as nib
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from scipy import ndimage
from scipy.ndimage import binary_fill_holes, label as scipy_label

from monai.transforms import (
    Compose, LoadImaged, EnsureChannelFirstd, Orientationd,
    Spacingd, EnsureTyped
)

# ============================================================
# CONFIGURATION
# ============================================================
class Config:
    # Paths
    DATA_ROOT = "/workspace/liver_tumor_project/data/raw/LiTS/LiTS(train_test)"
    SPLIT_FILE = "/workspace/liver_tumor_project/data/splits/fold_splits.json"
    OUTPUT_ROOT = "/workspace/liver_tumor_project/data/preprocessed"
    LOG_DIR = "/workspace/liver_tumor_project/logs/preprocessing"
    VIZ_DIR = "/workspace/liver_tumor_project/results/preprocessing_viz"
    
    # HU windowing (Diff4MMLiTS standard for liver)
    HU_MIN = -21
    HU_MAX = 189
    
    # Body detection (before windowing, on raw HU)
    BODY_HU_THRESHOLD = -500  # Air is ~-1000, soft tissue > -500
    
    # Resampling target
    TARGET_SPACING = (1.0, 1.0, 1.5)  # mm
    
    # Liver ROI cropping
    LIVER_MARGIN = 20  # voxels around liver bbox
    
    # Empty slice filter
    KEEP_LIVER_SLICES_ONLY = True
    CONTEXT_SLICES = 5  # slices above/below liver for context
    
    # Visualizations
    VIZ_SAMPLE_COUNT = 20

cfg = Config()

# ============================================================
# SAFETY CHECK — prevent inconsistent preprocessing
# ============================================================
existing_count = 0
for subdir in ["imagesTr", "labelsTr", "imagesTs", "labelsTs"]:
    path = os.path.join(cfg.OUTPUT_ROOT, subdir)
    if os.path.exists(path):
        existing_count += len([f for f in os.listdir(path) if f.endswith(('.nii.gz', '.nii'))])

if existing_count > 0:
    print(f"⚠️  WARNING: {existing_count} preprocessed files already exist!")
    print(f"   These may be from old preprocessing method.")
    print(f"   Run cleanup cell FIRST, then re-run this script.")
    raise SystemExit("❌ EXITING - Old preprocessed files detected. Clean up first!")
else:
    print(f"✅ Clean state — proceeding with V2 preprocessing\n")

# Create output directories
for d in [cfg.OUTPUT_ROOT, f"{cfg.OUTPUT_ROOT}/imagesTr", 
          f"{cfg.OUTPUT_ROOT}/labelsTr", f"{cfg.OUTPUT_ROOT}/imagesTs",
          f"{cfg.OUTPUT_ROOT}/labelsTs", cfg.LOG_DIR, cfg.VIZ_DIR]:
    os.makedirs(d, exist_ok=True)

# ============================================================
# LOGGING
# ============================================================
LOG_FILE = os.path.join(cfg.LOG_DIR, f"preprocessing_v2_{datetime.now().strftime('%Y%m%d_%H%M%S')}.log")

def log(msg, print_console=True):
    timestamp = datetime.now().strftime('%H:%M:%S')
    line = f"[{timestamp}] {msg}"
    if print_console:
        print(line)
    with open(LOG_FILE, 'a') as f:
        f.write(line + "\n")

log("=" * 70)
log("🌙 OVERNIGHT PREPROCESSING V2 — AGGRESSIVE BG REMOVAL")
log("=" * 70)
log(f"Configuration:")
log(f"  HU range:           [{cfg.HU_MIN}, {cfg.HU_MAX}]")
log(f"  Body HU threshold:  > {cfg.BODY_HU_THRESHOLD}")
log(f"  Target spacing:     {cfg.TARGET_SPACING} mm")
log(f"  Liver margin:       {cfg.LIVER_MARGIN} voxels")
log(f"  Context slices:     {cfg.CONTEXT_SLICES}")
log(f"  Output:             {cfg.OUTPUT_ROOT}")
log(f"  Log:                {LOG_FILE}")

# ============================================================
# BACKGROUND REMOVAL FUNCTIONS
# ============================================================
def compute_body_mask(hu_volume, threshold=-500):
    """
    Layer 1: Body mask via HU thresholding + morphology.
    Removes air outside body and CT table.
    """
    body_mask = hu_volume > threshold
    
    # Fill holes slice-by-slice (lungs would otherwise be excluded)
    for z in range(body_mask.shape[2]):
        body_mask[:, :, z] = binary_fill_holes(body_mask[:, :, z])
    
    # Keep only largest connected component (main body, remove table)
    labeled, num_components = scipy_label(body_mask)
    if num_components > 1:
        sizes = ndimage.sum(body_mask, labeled, range(1, num_components + 1))
        largest = np.argmax(sizes) + 1
        body_mask = labeled == largest
    
    return body_mask


def compute_liver_bbox(label_volume, margin=20):
    """
    Layer 2: Bounding box around liver (label >= 1).
    Returns tuple of (x_min, x_max, y_min, y_max, z_min, z_max).
    """
    liver_region = label_volume >= 1
    
    if not np.any(liver_region):
        return None
    
    coords = np.argwhere(liver_region)
    x_min, y_min, z_min = coords.min(axis=0)
    x_max, y_max, z_max = coords.max(axis=0) + 1
    
    # Add margin, clip to volume bounds
    x_min = max(0, x_min - margin)
    y_min = max(0, y_min - margin)
    z_min = max(0, z_min - margin)
    x_max = min(label_volume.shape[0], x_max + margin)
    y_max = min(label_volume.shape[1], y_max + margin)
    z_max = min(label_volume.shape[2], z_max + margin)
    
    return (x_min, x_max, y_min, y_max, z_min, z_max)


def filter_empty_slices(image, label, context_slices=5):
    """
    Layer 3: Remove Z-slices with no liver, keep context slices around.
    """
    liver_present = np.any(label >= 1, axis=(0, 1))
    
    if not np.any(liver_present):
        return image, label, (0, image.shape[2])
    
    liver_slices = np.where(liver_present)[0]
    z_start = max(0, liver_slices[0] - context_slices)
    z_end = min(image.shape[2], liver_slices[-1] + 1 + context_slices)
    
    return image[:, :, z_start:z_end], label[:, :, z_start:z_end], (z_start, z_end)


# ============================================================
# BUILD FILE PATHS FROM SPLITS
# ============================================================
with open(cfg.SPLIT_FILE, 'r') as f:
    splits = json.load(f)

def build_paths(volume_names, subset):
    paths = []
    for vol_name in volume_names:
        idx = vol_name.replace('volume-', '').replace('.nii', '')
        if subset == "test":
            img_src = f"{cfg.DATA_ROOT}/test_CT/volume-{idx}.nii"
            lbl_src = f"{cfg.DATA_ROOT}/test_mask/segmentation-{idx}.nii"
            img_out_dir = f"{cfg.OUTPUT_ROOT}/imagesTs"
            lbl_out_dir = f"{cfg.OUTPUT_ROOT}/labelsTs"
        else:
            img_src = f"{cfg.DATA_ROOT}/train_CT/volume-{idx}.nii"
            lbl_src = f"{cfg.DATA_ROOT}/train_mask/segmentation-{idx}.nii"
            img_out_dir = f"{cfg.OUTPUT_ROOT}/imagesTr"
            lbl_out_dir = f"{cfg.OUTPUT_ROOT}/labelsTr"
        
        if os.path.exists(img_src) and os.path.exists(lbl_src):
            paths.append({
                "image": img_src,
                "label": lbl_src,
                "img_out": f"{img_out_dir}/volume-{idx}.nii.gz",
                "lbl_out": f"{lbl_out_dir}/segmentation-{idx}.nii.gz",
                "name": f"volume-{idx}",
                "subset": subset,
            })
    return paths

all_paths = (
    [(p, "train") for p in build_paths(splits["train"]["volumes"], "train")] +
    [(p, "val") for p in build_paths(splits["val"]["volumes"], "train")] +
    [(p, "test") for p in build_paths(splits["test"]["volumes"], "test")]
)

log(f"\n📊 Total volumes to process: {len(all_paths)}")
log(f"   Train: {sum(1 for _, s in all_paths if s == 'train')}")
log(f"   Val:   {sum(1 for _, s in all_paths if s == 'val')}")
log(f"   Test:  {sum(1 for _, s in all_paths if s == 'test')}")

# ============================================================
# LOAD + RESAMPLE TRANSFORM
# ============================================================
resample_transform = Compose([
    LoadImaged(keys=["image", "label"]),
    EnsureChannelFirstd(keys=["image", "label"]),
    Orientationd(keys=["image", "label"], axcodes="RAS"),
    Spacingd(
        keys=["image", "label"],
        pixdim=cfg.TARGET_SPACING,
        mode=("bilinear", "nearest"),
    ),
    EnsureTyped(keys=["image", "label"], data_type="numpy"),
])


# ============================================================
# VISUALIZATION
# ============================================================
def save_viz(img_data, lbl_data, name, viz_dir, stats_str=""):
    """Save 3-panel visualization: CT + label + overlay."""
    try:
        img = np.squeeze(img_data)
        lbl = np.squeeze(lbl_data)
        
        tumor_slices = np.where(np.any(lbl == 2, axis=(0, 1)))[0]
        if len(tumor_slices) > 0:
            slice_idx = tumor_slices[len(tumor_slices) // 2]
            slice_type = "tumor"
        else:
            liver_slices = np.where(np.any(lbl == 1, axis=(0, 1)))[0]
            slice_idx = liver_slices[len(liver_slices) // 2] if len(liver_slices) > 0 else img.shape[2] // 2
            slice_type = "liver" if len(liver_slices) > 0 else "middle"
        
        fig, axes = plt.subplots(1, 3, figsize=(15, 5))
        
        axes[0].imshow(img[:, :, slice_idx], cmap='gray')
        axes[0].set_title(f'CT ({slice_type} slice {slice_idx})')
        axes[0].axis('off')
        
        axes[1].imshow(lbl[:, :, slice_idx], cmap='viridis', vmin=0, vmax=2)
        axes[1].set_title('Ground truth')
        axes[1].axis('off')
        
        axes[2].imshow(img[:, :, slice_idx], cmap='gray')
        mask = np.ma.masked_where(lbl[:, :, slice_idx] == 0, lbl[:, :, slice_idx])
        axes[2].imshow(mask, cmap='autumn', alpha=0.5, vmin=1, vmax=2)
        axes[2].set_title('Overlay')
        axes[2].axis('off')
        
        plt.suptitle(f'{name} | Shape: {img.shape} | {stats_str}', fontsize=11)
        plt.tight_layout()
        plt.savefig(f"{viz_dir}/{name}.png", dpi=80, bbox_inches='tight')
        plt.close()
        return True
    except Exception as e:
        log(f"    Viz failed: {e}", print_console=False)
        return False


# ============================================================
# MAIN PROCESSING LOOP
# ============================================================
stats = {
    "total": len(all_paths),
    "processed": 0,
    "failed": 0,
    "skipped_exists": 0,
    "start_time": time.time(),
    "volumes": [],
    "failures": [],
    "reduction_stats": [],
}

viz_saved = 0
log("\n" + "=" * 70)
log("🚀 STARTING PROCESSING")
log("=" * 70)

for i, (path_dict, subset) in enumerate(all_paths, 1):
    name = path_dict["name"]
    t0 = time.time()
    
    try:
        # Skip if already exists (resume support - safe since we cleaned first)
        if os.path.exists(path_dict["img_out"]) and os.path.exists(path_dict["lbl_out"]):
            stats["skipped_exists"] += 1
            log(f"[{i}/{stats['total']}] {name} ({subset}) — SKIP (exists)")
            continue
        
        log(f"[{i}/{stats['total']}] {name} ({subset})...")
        
        # Original stats
        orig_img = nib.load(path_dict["image"])
        orig_shape = orig_img.shape
        orig_voxels = np.prod(orig_shape)
        orig_spacing = tuple(round(s, 3) for s in orig_img.header.get_zooms())
        
        # Step 1: Load + resample to target spacing
        data = resample_transform({
            "image": path_dict["image"],
            "label": path_dict["label"]
        })
        img_resampled = np.squeeze(data["image"])
        lbl_resampled = np.squeeze(data["label"]).astype(np.uint8)
        after_resample_shape = img_resampled.shape
        
        # LAYER 1: Body mask (on HU values)
        body_mask = compute_body_mask(img_resampled, cfg.BODY_HU_THRESHOLD)
        img_resampled = np.where(body_mask, img_resampled, -1024)
        
        # LAYER 2: Liver ROI crop
        bbox = compute_liver_bbox(lbl_resampled, cfg.LIVER_MARGIN)
        if bbox:
            x_min, x_max, y_min, y_max, z_min, z_max = bbox
            img_cropped = img_resampled[x_min:x_max, y_min:y_max, z_min:z_max]
            lbl_cropped = lbl_resampled[x_min:x_max, y_min:y_max, z_min:z_max]
        else:
            log(f"    ⚠️  No liver detected, using full volume")
            img_cropped = img_resampled
            lbl_cropped = lbl_resampled
        after_bbox_shape = img_cropped.shape
        
        # LAYER 3: Filter empty Z-slices
        img_final, lbl_final, (z_s, z_e) = filter_empty_slices(
            img_cropped, lbl_cropped, cfg.CONTEXT_SLICES
        )
        after_slice_shape = img_final.shape
        
        # HU windowing + normalize to [0, 1]
        img_final = np.clip(img_final, cfg.HU_MIN, cfg.HU_MAX)
        img_final = (img_final - cfg.HU_MIN) / (cfg.HU_MAX - cfg.HU_MIN)
        img_final = img_final.astype(np.float32)
        
        # Compute stats
        final_voxels = np.prod(after_slice_shape)
        reduction_pct = (1 - final_voxels / orig_voxels) * 100
        
        tumor_voxels = int(np.sum(lbl_final == 2))
        liver_voxels = int(np.sum(lbl_final == 1))
        total_voxels = int(lbl_final.size)
        bg_voxels = total_voxels - tumor_voxels - liver_voxels
        
        # Save with proper spacing affine
        affine = np.eye(4)
        affine[0, 0] = cfg.TARGET_SPACING[0]
        affine[1, 1] = cfg.TARGET_SPACING[1]
        affine[2, 2] = cfg.TARGET_SPACING[2]
        
        nib.save(nib.Nifti1Image(img_final, affine), path_dict["img_out"])
        nib.save(nib.Nifti1Image(lbl_final, affine), path_dict["lbl_out"])
        
        elapsed = time.time() - t0
        
        # Detailed log
        log(f"    Orig:      {orig_shape} @ {orig_spacing}")
        log(f"    Resampled: {after_resample_shape}")
        log(f"    Bbox crop: {after_bbox_shape}")
        log(f"    Sliced:    {after_slice_shape}")
        log(f"    Voxels: {orig_voxels/1e6:.1f}M → {final_voxels/1e6:.1f}M (-{reduction_pct:.1f}%)")
        log(f"    BG: {100*bg_voxels/total_voxels:.1f}% | Liver: {100*liver_voxels/total_voxels:.2f}% | Tumor: {100*tumor_voxels/total_voxels:.3f}%")
        log(f"    Time: {elapsed:.1f}s")
        
        # Save visualization
        if viz_saved < cfg.VIZ_SAMPLE_COUNT:
            stats_str = f"-{reduction_pct:.0f}% voxels | tumor: {tumor_voxels:,}"
            if save_viz(img_final, lbl_final, f"{i:03d}_{name}_{subset}", cfg.VIZ_DIR, stats_str):
                viz_saved += 1
        
        # Record stats
        stats["volumes"].append({
            "name": name,
            "subset": subset,
            "orig_shape": list(orig_shape),
            "orig_spacing": list(orig_spacing),
            "resampled_shape": list(after_resample_shape),
            "bbox_shape": list(after_bbox_shape),
            "final_shape": list(after_slice_shape),
            "reduction_pct": float(reduction_pct),
            "bg_pct": float(100*bg_voxels/total_voxels),
            "liver_pct": float(100*liver_voxels/total_voxels),
            "tumor_pct": float(100*tumor_voxels/total_voxels),
            "tumor_voxels": tumor_voxels,
            "liver_voxels": liver_voxels,
            "time_sec": float(elapsed),
        })
        stats["reduction_stats"].append(reduction_pct)
        stats["processed"] += 1
        
    except Exception as e:
        stats["failed"] += 1
        stats["failures"].append({
            "name": name,
            "subset": subset,
            "error": str(e),
            "traceback": traceback.format_exc(),
        })
        log(f"    ❌ FAILED: {str(e)[:100]}")
    
    # Progress every 10 volumes
    if i % 10 == 0:
        elapsed_total = time.time() - stats["start_time"]
        eta = (elapsed_total / i) * (stats["total"] - i)
        avg_reduction = np.mean(stats["reduction_stats"]) if stats["reduction_stats"] else 0
        log(f"\n📊 [{i}/{stats['total']}] {100*i/stats['total']:.0f}% | Elapsed: {elapsed_total/60:.1f}m | ETA: {eta/60:.1f}m | Avg reduction: {avg_reduction:.0f}%\n")

# ============================================================
# FINAL SUMMARY
# ============================================================
total_time = time.time() - stats["start_time"]

log("\n" + "=" * 70)
log("✅ PREPROCESSING COMPLETE")
log("=" * 70)
log(f"Total:          {stats['total']}")
log(f"✅ Processed:   {stats['processed']}")
log(f"⏭️  Skipped:     {stats['skipped_exists']}")
log(f"❌ Failed:      {stats['failed']}")
log(f"⏱️  Total time:  {total_time/60:.1f} minutes")
log(f"📸 Visualizations: {viz_saved}")

if stats["volumes"]:
    reductions = [v["reduction_pct"] for v in stats["volumes"]]
    tumor_pcts = [v["tumor_pct"] for v in stats["volumes"]]
    bg_pcts = [v["bg_pct"] for v in stats["volumes"]]
    liver_pcts = [v["liver_pct"] for v in stats["volumes"]]
    
    log(f"\n📊 AGGREGATE STATISTICS:")
    log(f"\n   VOXEL REDUCTION:")
    log(f"     Mean reduction: {np.mean(reductions):.1f}%")
    log(f"     Range:          {min(reductions):.1f}% - {max(reductions):.1f}%")
    
    log(f"\n   NEW CLASS DISTRIBUTION (post-preprocessing):")
    log(f"     Background: {np.mean(bg_pcts):.1f}% (was 97%+)")
    log(f"     Liver:      {np.mean(liver_pcts):.2f}% (was 2.8%)")
    log(f"     Tumor:      {np.mean(tumor_pcts):.4f}% (was 0.1%)")
    
    log(f"\n   IMBALANCE IMPROVEMENT:")
    mean_tumor = max(np.mean(tumor_pcts), 0.001)
    log(f"     Tumor:BG ratio: 1:{int(np.mean(bg_pcts) / mean_tumor)}")

if stats["failures"]:
    log(f"\n❌ FAILURES:")
    for f in stats["failures"][:10]:
        log(f"   - {f['name']}: {f['error'][:80]}")

# Save stats
stats_path = os.path.join(cfg.LOG_DIR, "preprocessing_stats_v2.json")
with open(stats_path, 'w') as f:
    json.dump(stats, f, indent=2, default=str)
log(f"\n💾 Stats: {stats_path}")

# Disk usage
log(f"\n💾 DISK USAGE:")
os.system(f"du -sh {cfg.OUTPUT_ROOT}/* 2>/dev/null | tee -a {LOG_FILE}")
os.system(f"df -h /workspace | tail -1 | tee -a {LOG_FILE}")

log("\n" + "=" * 70)
log("🌅 GOOD MORNING! Check outputs.")
log("=" * 70)

print("\n\n" + "=" * 70)
print("✅ SCRIPT COMPLETE")
print("=" * 70)
print(f"\n📁 Locations to check:")
print(f"   1. Preprocessed data:   {cfg.OUTPUT_ROOT}/")
print(f"   2. Visualizations:      {cfg.VIZ_DIR}/")
print(f"   3. Log file:            {LOG_FILE}")
print(f"   4. Stats JSON:          {stats_path}")

In [ ]:
# 1. Log summary check
!tail -50 /workspace/liver_tumor_project/logs/preprocessing/preprocessing_v2_*.log

# 2. Files count
!ls /workspace/liver_tumor_project/data/preprocessed/imagesTr/ | wc -l
!ls /workspace/liver_tumor_project/data/preprocessed/imagesTs/ | wc -l

# 3. Sample visualization
from IPython.display import Image
import os
viz_dir = "/workspace/liver_tumor_project/results/preprocessing_viz"
first_viz = sorted(os.listdir(viz_dir))[0]
Image(filename=f"{viz_dir}/{first_viz}")

In [ ]:
import os
import json

# Check output structure
OUTPUT_ROOT = "/workspace/liver_tumor_project/data/preprocessed"

print("=" * 70)
print("📊 PREPROCESSED DATASET SUMMARY")
print("=" * 70)

# File counts
for subdir in ["imagesTr", "labelsTr", "imagesTs", "labelsTs"]:
    path = os.path.join(OUTPUT_ROOT, subdir)
    if os.path.exists(path):
        files = [f for f in os.listdir(path) if f.endswith('.nii.gz')]
        size_gb = sum(os.path.getsize(os.path.join(path, f)) for f in files) / (1024**3)
        print(f"  {subdir:<15} {len(files):>4} files  ({size_gb:.2f} GB)")

# Total size
print(f"\n💾 Total preprocessed size:")
os.system(f"du -sh {OUTPUT_ROOT}")

# Disk space
print(f"\n💾 Available space:")
os.system(f"df -h /workspace | tail -1")

# Load and show aggregate stats from JSON
stats_path = "/workspace/liver_tumor_project/logs/preprocessing/preprocessing_stats_v2.json"
if os.path.exists(stats_path):
    with open(stats_path) as f:
        stats = json.load(f)
    
    print(f"\n" + "=" * 70)
    print("📊 AGGREGATE STATISTICS")
    print("=" * 70)
    
    volumes = stats.get("volumes", [])
    if volumes:
        import numpy as np
        
        reductions = [v["reduction_pct"] for v in volumes]
        tumor_pcts = [v["tumor_pct"] for v in volumes]
        bg_pcts = [v["bg_pct"] for v in volumes]
        liver_pcts = [v["liver_pct"] for v in volumes]
        times = [v["time_sec"] for v in volumes]
        
        print(f"\n  Total processed:  {stats['processed']}")
        print(f"  Failures:         {stats['failed']}")
        print(f"  Total time:       {stats.get('total_time_sec', 0)/60:.1f} min")
        
        print(f"\n  VOXEL REDUCTION:")
        print(f"    Mean:  {np.mean(reductions):.1f}%")
        print(f"    Range: {min(reductions):.1f}% - {max(reductions):.1f}%")
        
        print(f"\n  CLASS DISTRIBUTION (post-preprocessing):")
        print(f"    Background: {np.mean(bg_pcts):.1f}%  (was 97%+)")
        print(f"    Liver:      {np.mean(liver_pcts):.2f}%")
        print(f"    Tumor:      {np.mean(tumor_pcts):.4f}%")
        
        mean_tumor = max(np.mean(tumor_pcts), 0.001)
        old_ratio = 7375
        new_ratio = int(np.mean(bg_pcts) / mean_tumor)
        improvement = old_ratio / new_ratio
        
        print(f"\n  IMBALANCE IMPROVEMENT:")
        print(f"    Old ratio: 1:{old_ratio} (tumor:bg)")
        print(f"    New ratio: 1:{new_ratio}")
        print(f"    Improvement: {improvement:.1f}x better!")
        
        print(f"\n  TIMING:")
        print(f"    Mean per volume: {np.mean(times):.1f}s")
        print(f"    Max per volume:  {max(times):.1f}s")

In [ ]:
import os
from IPython.display import Image, display

viz_dir = "/workspace/liver_tumor_project/results/preprocessing_viz"
viz_files = sorted(os.listdir(viz_dir))

print(f"📸 Total visualizations: {len(viz_files)}")
print(f"\nSample files:")
for f in viz_files[:20]:
    size_kb = os.path.getsize(os.path.join(viz_dir, f)) / 1024
    print(f"  {f} ({size_kb:.0f} KB)")

# Show first 3 visualizations
print(f"\nShowing first 3 samples:")
for f in viz_files[:3]:
    display(Image(filename=os.path.join(viz_dir, f)))

In [ ]:
import nibabel as nib
import numpy as np
import os
import time

PREPROCESSED = "/workspace/liver_tumor_project/data/preprocessed"

print("=" * 70)
print("🔬 LOADABILITY CHECK — Random 5 preprocessed volumes")
print("=" * 70)

# Test 5 random preprocessed volumes
test_files = sorted(os.listdir(f"{PREPROCESSED}/imagesTr"))[:5]

for fname in test_files:
    idx = fname.replace('volume-', '').replace('.nii.gz', '')
    img_path = f"{PREPROCESSED}/imagesTr/{fname}"
    lbl_path = f"{PREPROCESSED}/labelsTr/segmentation-{idx}.nii.gz"
    
    t0 = time.time()
    
    # Load image
    img = nib.load(img_path).get_fdata()
    lbl = nib.load(lbl_path).get_fdata()
    
    load_time = time.time() - t0
    
    # Verify
    print(f"\n📄 volume-{idx}:")
    print(f"   Image shape:  {img.shape}")
    print(f"   Image range:  [{img.min():.3f}, {img.max():.3f}] (should be [0,1])")
    print(f"   Label shape:  {lbl.shape}")
    print(f"   Label unique: {sorted(np.unique(lbl).astype(int).tolist())}")
    print(f"   Tumor voxels: {int(np.sum(lbl == 2)):,}")
    print(f"   Load time:    {load_time*1000:.0f} ms")
    print(f"   Shape match:  {'✅' if img.shape == lbl.shape else '❌'}")

# Compression ratio
print(f"\n" + "=" * 70)
print(f"📊 STORAGE EFFICIENCY")
print("=" * 70)
os.system(f"du -sh {PREPROCESSED}/imagesTr/")
os.system(f"du -sh {PREPROCESSED}/labelsTr/")

print(f"\n🎯 Ready for training pipeline!")

In [ ]:
"""
=========================================================================
UNIVERSAL DATA LOADER FACTORY
For all 10 baselines + your novel method
Reads preprocessed .nii.gz volumes, extracts training patches
=========================================================================
"""

import os

script_content = '''"""
Universal Data Loader for LiTS Preprocessed Data
Used by all 10 baseline models + novel method
"""

import os
import json
import numpy as np
import torch
from monai.transforms import (
    Compose,
    LoadImaged,
    EnsureChannelFirstd,
    RandCropByPosNegLabeld,
    RandFlipd,
    RandRotate90d,
    RandShiftIntensityd,
    RandScaleIntensityd,
    RandGaussianNoised,
    RandAdjustContrastd,
    EnsureTyped,
    ToTensord,
)
from monai.data import CacheDataset, Dataset, DataLoader, list_data_collate
from monai.utils import set_determinism


# ========================================================================
# CONFIGURATION
# ========================================================================
class DataConfig:
    """Universal data config for all models."""
    # Paths (preprocessed data)
    DATA_ROOT = "/workspace/liver_tumor_project/data/preprocessed"
    SPLIT_FILE = "/workspace/liver_tumor_project/data/splits/fold_splits.json"
    
    # Training patch size (fits 80GB A100 with batch=2-4)
    PATCH_SIZE = (128, 128, 128)
    
    # Sampling ratio (tumor:non-tumor patches)
    # Since tumor is ~0.66%, we heavily weight positive samples
    POS_NEG_RATIO = 2.0   # 2 tumor patches per 1 background
    NUM_SAMPLES = 4        # 4 patches per volume per iteration
    
    # Batch sizes
    TRAIN_BATCH_SIZE = 2   # 2 volumes * 4 patches = effective batch of 8
    VAL_BATCH_SIZE = 1     # Full volumes for validation
    TEST_BATCH_SIZE = 1
    
    # Workers
    NUM_WORKERS = 8        # 256 CPU cores available!
    CACHE_RATE = 1.0       # Cache all in RAM (2TB available)
    
    # Reproducibility
    SEED = 42


# ========================================================================
# BUILD DATA DICTS FROM PREPROCESSED FOLDER
# ========================================================================
def build_data_dicts(cfg=DataConfig):
    """Build (image, label) dicts from split JSON."""
    with open(cfg.SPLIT_FILE, "r") as f:
        splits = json.load(f)
    
    def make_dicts(volume_names, subset):
        dicts = []
        for vol_name in volume_names:
            idx = vol_name.replace("volume-", "").replace(".nii", "")
            if subset == "test":
                img_path = f"{cfg.DATA_ROOT}/imagesTs/volume-{idx}.nii.gz"
                lbl_path = f"{cfg.DATA_ROOT}/labelsTs/segmentation-{idx}.nii.gz"
            else:
                img_path = f"{cfg.DATA_ROOT}/imagesTr/volume-{idx}.nii.gz"
                lbl_path = f"{cfg.DATA_ROOT}/labelsTr/segmentation-{idx}.nii.gz"
            
            if os.path.exists(img_path) and os.path.exists(lbl_path):
                dicts.append({
                    "image": img_path,
                    "label": lbl_path,
                    "name": vol_name,
                })
        return dicts
    
    train_dicts = make_dicts(splits["train"]["volumes"], "train")
    val_dicts = make_dicts(splits["val"]["volumes"], "train")
    test_dicts = make_dicts(splits["test"]["volumes"], "test")
    
    return train_dicts, val_dicts, test_dicts


# ========================================================================
# TRAINING TRANSFORMS - patch extraction + augmentation
# ========================================================================
def get_train_transforms(cfg=DataConfig):
    """Training pipeline: load + patch + augment."""
    return Compose([
        # 1. Load preprocessed data
        LoadImaged(keys=["image", "label"]),
        EnsureChannelFirstd(keys=["image", "label"]),
        
        # 2. Extract tumor-focused patches
        RandCropByPosNegLabeld(
            keys=["image", "label"],
            label_key="label",
            spatial_size=cfg.PATCH_SIZE,
            pos=cfg.POS_NEG_RATIO,
            neg=1.0,
            num_samples=cfg.NUM_SAMPLES,
            image_key="image",
            image_threshold=0,
        ),
        
        # 3. Spatial augmentations
        RandFlipd(keys=["image", "label"], spatial_axis=[0], prob=0.5),
        RandFlipd(keys=["image", "label"], spatial_axis=[1], prob=0.5),
        RandFlipd(keys=["image", "label"], spatial_axis=[2], prob=0.5),
        RandRotate90d(keys=["image", "label"], prob=0.5, max_k=3),
        
        # 4. Intensity augmentations
        RandShiftIntensityd(keys=["image"], offsets=0.10, prob=0.5),
        RandScaleIntensityd(keys=["image"], factors=0.10, prob=0.5),
        RandAdjustContrastd(keys=["image"], gamma=(0.7, 1.5), prob=0.3),
        RandGaussianNoised(keys=["image"], mean=0.0, std=0.01, prob=0.15),
        
        # 5. Tensor conversion
        EnsureTyped(keys=["image", "label"]),
    ])


# ========================================================================
# VAL/TEST TRANSFORMS - no augmentation, no patching
# ========================================================================
def get_val_transforms(cfg=DataConfig):
    """Validation: load full volume, no augmentation."""
    return Compose([
        LoadImaged(keys=["image", "label"]),
        EnsureChannelFirstd(keys=["image", "label"]),
        EnsureTyped(keys=["image", "label"]),
    ])


def get_test_transforms(cfg=DataConfig):
    """Test: same as validation."""
    return get_val_transforms(cfg)


# ========================================================================
# DATALOADER FACTORY
# ========================================================================
def get_dataloaders(cfg=DataConfig, verbose=True):
    """
    Returns (train_loader, val_loader, test_loader).
    Uses CacheDataset for train/val (2TB RAM available).
    """
    set_determinism(seed=cfg.SEED)
    
    train_dicts, val_dicts, test_dicts = build_data_dicts(cfg)
    
    if verbose:
        print(f"📊 Dataset sizes:")
        print(f"   Train: {len(train_dicts)} volumes")
        print(f"   Val:   {len(val_dicts)} volumes")
        print(f"   Test:  {len(test_dicts)} volumes")
    
    # Training dataset with caching
    train_ds = CacheDataset(
        data=train_dicts,
        transform=get_train_transforms(cfg),
        cache_rate=cfg.CACHE_RATE,
        num_workers=cfg.NUM_WORKERS,
        progress=verbose,
    )
    
    # Validation with caching (full volumes)
    val_ds = CacheDataset(
        data=val_dicts,
        transform=get_val_transforms(cfg),
        cache_rate=cfg.CACHE_RATE,
        num_workers=cfg.NUM_WORKERS,
        progress=verbose,
    )
    
    # Test - no caching needed (evaluated once)
    test_ds = Dataset(
        data=test_dicts,
        transform=get_test_transforms(cfg),
    )
    
    train_loader = DataLoader(
        train_ds,
        batch_size=cfg.TRAIN_BATCH_SIZE,
        shuffle=True,
        num_workers=cfg.NUM_WORKERS,
        pin_memory=True,
        collate_fn=list_data_collate,
    )
    
    val_loader = DataLoader(
        val_ds,
        batch_size=cfg.VAL_BATCH_SIZE,
        shuffle=False,
        num_workers=cfg.NUM_WORKERS,
        pin_memory=True,
    )
    
    test_loader = DataLoader(
        test_ds,
        batch_size=cfg.TEST_BATCH_SIZE,
        shuffle=False,
        num_workers=cfg.NUM_WORKERS,
        pin_memory=True,
    )
    
    return train_loader, val_loader, test_loader


# ========================================================================
# QUICK TEST
# ========================================================================
if __name__ == "__main__":
    train_loader, val_loader, test_loader = get_dataloaders()
    
    print("\\n🔍 Sample batch from train_loader:")
    batch = next(iter(train_loader))
    print(f"   Image shape: {batch['image'].shape}")
    print(f"   Label shape: {batch['label'].shape}")
    print(f"   Image range: [{batch['image'].min():.3f}, {batch['image'].max():.3f}]")
    print(f"   Label unique: {torch.unique(batch['label']).tolist()}")
'''

script_path = "/workspace/liver_tumor_project/scripts/dataloader.py"
os.makedirs(os.path.dirname(script_path), exist_ok=True)
with open(script_path, 'w') as f:
    f.write(script_content)

print(f"✅ Dataloader saved: {script_path}")
print(f"📏 File size: {os.path.getsize(script_path) / 1024:.1f} KB")

# Show
print("\n📁 Scripts folder:")
!ls -la /workspace/liver_tumor_project/scripts/

In [ ]:
"""
=========================================================================
UNIVERSAL DATA LOADER FACTORY - Reusable for all 10 baselines
=========================================================================
"""
import os

script_content = '''"""
Universal Data Loader for LiTS
Used by all 10 baselines + novel method
"""

import os
import json
import numpy as np
import torch
from monai.transforms import (
    Compose,
    LoadImaged,
    EnsureChannelFirstd,
    RandCropByPosNegLabeld,
    RandFlipd,
    RandRotate90d,
    RandShiftIntensityd,
    RandScaleIntensityd,
    RandGaussianNoised,
    RandAdjustContrastd,
    EnsureTyped,
)
from monai.data import CacheDataset, Dataset, DataLoader, list_data_collate
from monai.utils import set_determinism


class DataConfig:
    DATA_ROOT = "/workspace/liver_tumor_project/data/preprocessed"
    SPLIT_FILE = "/workspace/liver_tumor_project/data/splits/fold_splits.json"
    
    # Training patch size
    PATCH_SIZE = (128, 128, 128)
    
    # Tumor-focused sampling
    POS_NEG_RATIO = 2.0
    NUM_SAMPLES = 4
    
    # Batch sizes
    TRAIN_BATCH_SIZE = 2
    VAL_BATCH_SIZE = 1
    TEST_BATCH_SIZE = 1
    
    NUM_WORKERS = 8
    CACHE_RATE = 1.0
    SEED = 42


def build_data_dicts(cfg=DataConfig):
    with open(cfg.SPLIT_FILE, "r") as f:
        splits = json.load(f)
    
    def make_dicts(volume_names, subset):
        dicts = []
        for vol_name in volume_names:
            idx = vol_name.replace("volume-", "").replace(".nii", "")
            if subset == "test":
                img_path = f"{cfg.DATA_ROOT}/imagesTs/volume-{idx}.nii.gz"
                lbl_path = f"{cfg.DATA_ROOT}/labelsTs/segmentation-{idx}.nii.gz"
            else:
                img_path = f"{cfg.DATA_ROOT}/imagesTr/volume-{idx}.nii.gz"
                lbl_path = f"{cfg.DATA_ROOT}/labelsTr/segmentation-{idx}.nii.gz"
            
            if os.path.exists(img_path) and os.path.exists(lbl_path):
                dicts.append({
                    "image": img_path,
                    "label": lbl_path,
                    "name": vol_name,
                })
        return dicts
    
    return (
        make_dicts(splits["train"]["volumes"], "train"),
        make_dicts(splits["val"]["volumes"], "train"),
        make_dicts(splits["test"]["volumes"], "test"),
    )


def get_train_transforms(cfg=DataConfig):
    return Compose([
        LoadImaged(keys=["image", "label"]),
        EnsureChannelFirstd(keys=["image", "label"]),
        RandCropByPosNegLabeld(
            keys=["image", "label"],
            label_key="label",
            spatial_size=cfg.PATCH_SIZE,
            pos=cfg.POS_NEG_RATIO,
            neg=1.0,
            num_samples=cfg.NUM_SAMPLES,
            image_key="image",
            image_threshold=0,
        ),
        RandFlipd(keys=["image", "label"], spatial_axis=[0], prob=0.5),
        RandFlipd(keys=["image", "label"], spatial_axis=[1], prob=0.5),
        RandFlipd(keys=["image", "label"], spatial_axis=[2], prob=0.5),
        RandRotate90d(keys=["image", "label"], prob=0.5, max_k=3),
        RandShiftIntensityd(keys=["image"], offsets=0.10, prob=0.5),
        RandScaleIntensityd(keys=["image"], factors=0.10, prob=0.5),
        RandAdjustContrastd(keys=["image"], gamma=(0.7, 1.5), prob=0.3),
        RandGaussianNoised(keys=["image"], mean=0.0, std=0.01, prob=0.15),
        EnsureTyped(keys=["image", "label"]),
    ])


def get_val_transforms(cfg=DataConfig):
    return Compose([
        LoadImaged(keys=["image", "label"]),
        EnsureChannelFirstd(keys=["image", "label"]),
        EnsureTyped(keys=["image", "label"]),
    ])


def get_dataloaders(cfg=DataConfig, verbose=True):
    set_determinism(seed=cfg.SEED)
    train_dicts, val_dicts, test_dicts = build_data_dicts(cfg)
    
    if verbose:
        print(f"Dataset sizes: train={len(train_dicts)}, val={len(val_dicts)}, test={len(test_dicts)}")
    
    train_ds = CacheDataset(
        data=train_dicts,
        transform=get_train_transforms(cfg),
        cache_rate=cfg.CACHE_RATE,
        num_workers=cfg.NUM_WORKERS,
        progress=verbose,
    )
    val_ds = CacheDataset(
        data=val_dicts,
        transform=get_val_transforms(cfg),
        cache_rate=cfg.CACHE_RATE,
        num_workers=cfg.NUM_WORKERS,
        progress=verbose,
    )
    test_ds = Dataset(data=test_dicts, transform=get_val_transforms(cfg))
    
    train_loader = DataLoader(
        train_ds, batch_size=cfg.TRAIN_BATCH_SIZE, shuffle=True,
        num_workers=cfg.NUM_WORKERS, pin_memory=True,
        collate_fn=list_data_collate,
    )
    val_loader = DataLoader(
        val_ds, batch_size=cfg.VAL_BATCH_SIZE, shuffle=False,
        num_workers=cfg.NUM_WORKERS, pin_memory=True,
    )
    test_loader = DataLoader(
        test_ds, batch_size=cfg.TEST_BATCH_SIZE, shuffle=False,
        num_workers=cfg.NUM_WORKERS, pin_memory=True,
    )
    
    return train_loader, val_loader, test_loader
'''

script_path = "/workspace/liver_tumor_project/scripts/dataloader.py"
os.makedirs(os.path.dirname(script_path), exist_ok=True)
with open(script_path, 'w') as f:
    f.write(script_content)

print(f"✅ Dataloader saved: {script_path}")

In [ ]:
"""
=========================================================================
nnU-Net v2 SETUP FOR LiTS
Auto-configuring baseline - gold standard reference
=========================================================================
"""

import os
import json
import shutil

# nnU-Net v2 requires specific folder structure and env variables
print("=" * 70)
print("🚀 nnU-Net v2 SETUP")
print("=" * 70)

# Set up nnU-Net v2 environment variables (required)
NNUNET_BASE = "/workspace/liver_tumor_project/models/nnunet_v2"

os.environ["nnUNet_raw"] = f"{NNUNET_BASE}/nnUNet_raw"
os.environ["nnUNet_preprocessed"] = f"{NNUNET_BASE}/nnUNet_preprocessed"
os.environ["nnUNet_results"] = f"{NNUNET_BASE}/nnUNet_results"

# Create required directories
for var, path in [
    ("nnUNet_raw", os.environ["nnUNet_raw"]),
    ("nnUNet_preprocessed", os.environ["nnUNet_preprocessed"]),
    ("nnUNet_results", os.environ["nnUNet_results"]),
]:
    os.makedirs(path, exist_ok=True)
    print(f"  ✅ {var}: {path}")

# nnU-Net v2 expects dataset in specific format
# Dataset name: Dataset001_LiTS
DATASET_ID = 1
DATASET_NAME = f"Dataset{DATASET_ID:03d}_LiTS"
DATASET_DIR = os.path.join(os.environ["nnUNet_raw"], DATASET_NAME)

print(f"\n📁 Creating dataset structure: {DATASET_NAME}")
os.makedirs(f"{DATASET_DIR}/imagesTr", exist_ok=True)
os.makedirs(f"{DATASET_DIR}/labelsTr", exist_ok=True)
os.makedirs(f"{DATASET_DIR}/imagesTs", exist_ok=True)

# Verify nnU-Net installed
try:
    import nnunetv2
    print(f"\n✅ nnU-Net v2 installed: {nnunetv2.__file__}")
except ImportError:
    print("\n❌ nnU-Net v2 not installed! Installing now...")
    !pip install nnunetv2 -q

# Check GPU
import torch
print(f"\n🎮 GPU: {torch.cuda.get_device_name(0)}")
print(f"   Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

In [ ]:
"""
Convert preprocessed data to nnU-Net v2 format
nnU-Net expects:
  imagesTr/volume-0_0000.nii.gz  (note the _0000 suffix for modality)
  labelsTr/volume-0.nii.gz
"""

import os
import json

DATA_ROOT = "/workspace/liver_tumor_project/data/preprocessed"
SPLIT_FILE = "/workspace/liver_tumor_project/data/splits/fold_splits.json"

# nnU-Net raw directory
DATASET_DIR = os.path.join(os.environ["nnUNet_raw"], "Dataset001_LiTS")

# Load splits
with open(SPLIT_FILE) as f:
    splits = json.load(f)

# nnU-Net treats train+val as one training set (uses its own CV)
# We'll use test set as-is
print("=" * 70)
print("🔗 CREATING nnU-Net FORMAT SYMLINKS (no disk waste)")
print("=" * 70)

train_count = 0
test_count = 0

# Training + Validation → nnU-Net's "imagesTr"
for vol_name in splits["train"]["volumes"] + splits["val"]["volumes"]:
    idx = vol_name.replace("volume-", "").replace(".nii", "")
    
    src_img = f"{DATA_ROOT}/imagesTr/volume-{idx}.nii.gz"
    src_lbl = f"{DATA_ROOT}/labelsTr/segmentation-{idx}.nii.gz"
    
    # nnU-Net naming: {case_id}_{modality:04d}.nii.gz
    dst_img = f"{DATASET_DIR}/imagesTr/volume-{idx}_0000.nii.gz"
    dst_lbl = f"{DATASET_DIR}/labelsTr/volume-{idx}.nii.gz"
    
    if os.path.exists(src_img) and not os.path.exists(dst_img):
        os.symlink(src_img, dst_img)
    if os.path.exists(src_lbl) and not os.path.exists(dst_lbl):
        os.symlink(src_lbl, dst_lbl)
    
    train_count += 1

# Test set → nnU-Net's "imagesTs"
for vol_name in splits["test"]["volumes"]:
    idx = vol_name.replace("volume-", "").replace(".nii", "")
    
    src_img = f"{DATA_ROOT}/imagesTs/volume-{idx}.nii.gz"
    dst_img = f"{DATASET_DIR}/imagesTs/volume-{idx}_0000.nii.gz"
    
    if os.path.exists(src_img) and not os.path.exists(dst_img):
        os.symlink(src_img, dst_img)
    
    test_count += 1

print(f"\n✅ Train symlinks: {train_count} images + labels")
print(f"✅ Test symlinks:  {test_count} images")

# Create nnU-Net's required dataset.json
dataset_json = {
    "channel_names": {
        "0": "CT"
    },
    "labels": {
        "background": 0,
        "liver": 1,
        "tumor": 2
    },
    "numTraining": train_count,
    "file_ending": ".nii.gz",
    "name": "LiTS",
    "description": "Liver Tumor Segmentation Challenge - Preprocessed with aggressive BG removal",
    "reference": "https://competitions.codalab.org/competitions/17094",
    "licence": "CC-BY-SA 4.0",
    "release": "1.0",
    "dataset_name": "Dataset001_LiTS",
    "tensorImageSize": "3D",
}

json_path = f"{DATASET_DIR}/dataset.json"
with open(json_path, 'w') as f:
    json.dump(dataset_json, f, indent=2)

print(f"\n📝 dataset.json created: {json_path}")

# Verify structure
print(f"\n📁 nnU-Net dataset structure:")
!ls -la {DATASET_DIR}
print(f"\n📊 Files count:")
!ls {DATASET_DIR}/imagesTr/ | wc -l
!ls {DATASET_DIR}/labelsTr/ | wc -l
!ls {DATASET_DIR}/imagesTs/ | wc -l

In [ ]:
"""
=========================================================================
nnU-Net v2 SETUP FOR LiTS - Auto-configuring baseline
=========================================================================
"""

import os

print("=" * 70)
print("🚀 nnU-Net v2 SETUP")
print("=" * 70)

# nnU-Net v2 required environment variables
NNUNET_BASE = "/workspace/liver_tumor_project/models/nnunet_v2"

os.environ["nnUNet_raw"] = f"{NNUNET_BASE}/nnUNet_raw"
os.environ["nnUNet_preprocessed"] = f"{NNUNET_BASE}/nnUNet_preprocessed"
os.environ["nnUNet_results"] = f"{NNUNET_BASE}/nnUNet_results"

# Create required directories
print("\n📁 Setting up nnU-Net directories:")
for var, path in [
    ("nnUNet_raw", os.environ["nnUNet_raw"]),
    ("nnUNet_preprocessed", os.environ["nnUNet_preprocessed"]),
    ("nnUNet_results", os.environ["nnUNet_results"]),
]:
    os.makedirs(path, exist_ok=True)
    print(f"  ✅ {var}: {path}")

# Dataset name (nnU-Net convention: Dataset{ID:03d}_{Name})
DATASET_ID = 1
DATASET_NAME = f"Dataset{DATASET_ID:03d}_LiTS"
DATASET_DIR = os.path.join(os.environ["nnUNet_raw"], DATASET_NAME)

print(f"\n📁 Creating dataset structure: {DATASET_NAME}")
os.makedirs(f"{DATASET_DIR}/imagesTr", exist_ok=True)
os.makedirs(f"{DATASET_DIR}/labelsTr", exist_ok=True)
os.makedirs(f"{DATASET_DIR}/imagesTs", exist_ok=True)
print(f"  ✅ {DATASET_DIR}/imagesTr")
print(f"  ✅ {DATASET_DIR}/labelsTr")
print(f"  ✅ {DATASET_DIR}/imagesTs")

# Verify nnU-Net v2 installation
try:
    import nnunetv2
    print(f"\n✅ nnU-Net v2 imported successfully")
    print(f"   Location: {os.path.dirname(nnunetv2.__file__)}")
except ImportError:
    print("\n❌ nnU-Net v2 not installed! Installing...")
    !pip install nnunetv2 -q
    import nnunetv2
    print(f"✅ Installed: {nnunetv2.__file__}")

# GPU check
import torch
print(f"\n🎮 GPU: {torch.cuda.get_device_name(0)}")
print(f"   Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
print(f"   Compute: {torch.cuda.get_device_capability(0)}")

# Save env vars to a file so we can source them later
env_file = f"{NNUNET_BASE}/nnunet_env.sh"
with open(env_file, 'w') as f:
    f.write(f'export nnUNet_raw="{os.environ["nnUNet_raw"]}"\n')
    f.write(f'export nnUNet_preprocessed="{os.environ["nnUNet_preprocessed"]}"\n')
    f.write(f'export nnUNet_results="{os.environ["nnUNet_results"]}"\n')
print(f"\n💾 Env vars saved: {env_file}")
print("   Use: source " + env_file + "  (for terminal commands)")

print("\n" + "=" * 70)
print("✅ nnU-Net v2 SETUP COMPLETE")
print("=" * 70)

In [ ]:
"""
Convert preprocessed data to nnU-Net v2 format via symlinks
No disk waste - just creates references
"""

import os
import json

DATA_ROOT = "/workspace/liver_tumor_project/data/preprocessed"
SPLIT_FILE = "/workspace/liver_tumor_project/data/splits/fold_splits.json"

# Ensure env variables are set (in case of kernel restart)
if "nnUNet_raw" not in os.environ:
    NNUNET_BASE = "/workspace/liver_tumor_project/models/nnunet_v2"
    os.environ["nnUNet_raw"] = f"{NNUNET_BASE}/nnUNet_raw"
    os.environ["nnUNet_preprocessed"] = f"{NNUNET_BASE}/nnUNet_preprocessed"
    os.environ["nnUNet_results"] = f"{NNUNET_BASE}/nnUNet_results"

DATASET_DIR = os.path.join(os.environ["nnUNet_raw"], "Dataset001_LiTS")

# Load splits
with open(SPLIT_FILE) as f:
    splits = json.load(f)

print("=" * 70)
print("🔗 CREATING nnU-Net FORMAT SYMLINKS")
print("=" * 70)
print(f"\n📁 Source: {DATA_ROOT}")
print(f"📁 Target: {DATASET_DIR}")
print(f"\n💡 Using symlinks - no disk waste, ~1 minute\n")

train_count = 0
test_count = 0
skipped_train = 0
skipped_test = 0

# Training + Validation → nnU-Net's "imagesTr"
# nnU-Net does its own cross-validation, so combine train + val
print("Processing train + val volumes...")
for vol_name in splits["train"]["volumes"] + splits["val"]["volumes"]:
    idx = vol_name.replace("volume-", "").replace(".nii", "")
    
    src_img = f"{DATA_ROOT}/imagesTr/volume-{idx}.nii.gz"
    src_lbl = f"{DATA_ROOT}/labelsTr/segmentation-{idx}.nii.gz"
    
    # nnU-Net naming: {case_id}_{modality:04d}.nii.gz for images
    #                 {case_id}.nii.gz for labels
    dst_img = f"{DATASET_DIR}/imagesTr/volume-{idx}_0000.nii.gz"
    dst_lbl = f"{DATASET_DIR}/labelsTr/volume-{idx}.nii.gz"
    
    if not os.path.exists(src_img):
        print(f"  ⚠️  Source missing: {src_img}")
        skipped_train += 1
        continue
    
    # Create symlinks (skip if already exists)
    if not os.path.exists(dst_img):
        os.symlink(src_img, dst_img)
    if not os.path.exists(dst_lbl) and os.path.exists(src_lbl):
        os.symlink(src_lbl, dst_lbl)
    
    train_count += 1

print(f"  ✅ Train+val: {train_count} volumes linked ({skipped_train} skipped)")

# Test set → "imagesTs"
print("\nProcessing test volumes...")
for vol_name in splits["test"]["volumes"]:
    idx = vol_name.replace("volume-", "").replace(".nii", "")
    
    src_img = f"{DATA_ROOT}/imagesTs/volume-{idx}.nii.gz"
    dst_img = f"{DATASET_DIR}/imagesTs/volume-{idx}_0000.nii.gz"
    
    if not os.path.exists(src_img):
        print(f"  ⚠️  Source missing: {src_img}")
        skipped_test += 1
        continue
    
    if not os.path.exists(dst_img):
        os.symlink(src_img, dst_img)
    
    test_count += 1

print(f"  ✅ Test: {test_count} volumes linked ({skipped_test} skipped)")

# Create nnU-Net's required dataset.json
dataset_json = {
    "channel_names": {
        "0": "CT"
    },
    "labels": {
        "background": 0,
        "liver": 1,
        "tumor": 2
    },
    "numTraining": train_count,
    "file_ending": ".nii.gz",
    "name": "LiTS",
    "description": "Liver Tumor Segmentation - Preprocessed with aggressive background removal (V2 pipeline)",
    "reference": "https://competitions.codalab.org/competitions/17094",
    "licence": "CC-BY-SA 4.0",
    "release": "1.0",
    "tensorImageSize": "3D",
}

json_path = f"{DATASET_DIR}/dataset.json"
with open(json_path, 'w') as f:
    json.dump(dataset_json, f, indent=2)

print(f"\n📝 dataset.json created: {json_path}")

# Verify structure
print("\n" + "=" * 70)
print("📊 FINAL VERIFICATION")
print("=" * 70)

print(f"\n📁 Dataset directory:")
!ls -la {DATASET_DIR}

print(f"\n📊 File counts:")
!echo "imagesTr: $(ls {DATASET_DIR}/imagesTr/ | wc -l) files"
!echo "labelsTr: $(ls {DATASET_DIR}/labelsTr/ | wc -l) files"
!echo "imagesTs: $(ls {DATASET_DIR}/imagesTs/ | wc -l) files"

print(f"\n📄 Sample file names:")
!echo "imagesTr samples:" && ls {DATASET_DIR}/imagesTr/ | head -3
!echo "labelsTr samples:" && ls {DATASET_DIR}/labelsTr/ | head -3
!echo "imagesTs samples:" && ls {DATASET_DIR}/imagesTs/ | head -3

# Verify a symlink actually resolves
print(f"\n🔗 Symlink resolution test:")
sample_link = f"{DATASET_DIR}/imagesTr/{os.listdir(DATASET_DIR + '/imagesTr')[0]}"
print(f"   Link: {sample_link}")
print(f"   Target: {os.readlink(sample_link)}")
print(f"   Resolves: {'✅' if os.path.exists(sample_link) else '❌'}")

# Show dataset.json content
print(f"\n📝 dataset.json contents:")
with open(json_path) as f:
    print(f.read())

print("=" * 70)
print("✅ DATA CONVERSION COMPLETE - Ready for nnU-Net planning")
print("=" * 70)

In [ ]:
"""
=========================================================================
SHAPE MISMATCH REPAIR
Check all volumes, align mask shape to image shape via crop/pad
=========================================================================
"""

import os
import nibabel as nib
import numpy as np
from tqdm import tqdm

BASE = "/workspace/liver_tumor_project/data/preprocessed"

# Collect all pairs to check
pairs = []
# Train pairs
train_dir_img = f"{BASE}/imagesTr"
train_dir_lbl = f"{BASE}/labelsTr"
for f in sorted(os.listdir(train_dir_img)):
    idx = f.replace("volume-", "").replace(".nii.gz", "")
    pairs.append({
        "name": f"volume-{idx}",
        "img": f"{train_dir_img}/{f}",
        "lbl": f"{train_dir_lbl}/segmentation-{idx}.nii.gz",
        "set": "train"
    })
# Test pairs
test_dir_img = f"{BASE}/imagesTs"
test_dir_lbl = f"{BASE}/labelsTs"
for f in sorted(os.listdir(test_dir_img)):
    idx = f.replace("volume-", "").replace(".nii.gz", "")
    pairs.append({
        "name": f"volume-{idx}",
        "img": f"{test_dir_img}/{f}",
        "lbl": f"{test_dir_lbl}/segmentation-{idx}.nii.gz",
        "set": "test"
    })

print("=" * 70)
print(f"🔍 CHECKING {len(pairs)} PAIRS FOR SHAPE MISMATCH")
print("=" * 70)

mismatches = []
for pair in tqdm(pairs, desc="Scanning"):
    if not (os.path.exists(pair["img"]) and os.path.exists(pair["lbl"])):
        continue
    
    img_nii = nib.load(pair["img"])
    lbl_nii = nib.load(pair["lbl"])
    
    img_shape = img_nii.shape
    lbl_shape = lbl_nii.shape
    
    if img_shape != lbl_shape:
        mismatches.append({
            **pair,
            "img_shape": img_shape,
            "lbl_shape": lbl_shape,
            "diff": tuple(l - i for i, l in zip(img_shape, lbl_shape))
        })

print(f"\n📊 Results:")
print(f"   Total pairs:  {len(pairs)}")
print(f"   Mismatches:   {len(mismatches)}")
print(f"   OK pairs:     {len(pairs) - len(mismatches)}")

if not mismatches:
    print("\n✅ No mismatches found! Something else caused the error.")
else:
    print(f"\n⚠️  Mismatch details:")
    for m in mismatches:
        print(f"   {m['name']} ({m['set']}): img={m['img_shape']} vs lbl={m['lbl_shape']} diff={m['diff']}")
    
    print(f"\n🔧 FIXING mismatches — cropping/padding masks to match image shape...")
    
    fixed = 0
    for m in tqdm(mismatches, desc="Fixing"):
        img_nii = nib.load(m["img"])
        lbl_nii = nib.load(m["lbl"])
        
        img_data = img_nii.get_fdata()
        lbl_data = lbl_nii.get_fdata().astype(np.uint8)
        
        target_shape = img_data.shape
        current_shape = lbl_data.shape
        
        # Create new array with target shape
        new_lbl = np.zeros(target_shape, dtype=np.uint8)
        
        # Compute copy region (minimum of both shapes on each axis)
        min_shape = tuple(min(t, c) for t, c in zip(target_shape, current_shape))
        
        # Copy the overlapping region
        slices = tuple(slice(0, s) for s in min_shape)
        new_lbl[slices] = lbl_data[slices]
        
        # Save with same affine as image
        fixed_nii = nib.Nifti1Image(new_lbl, img_nii.affine, header=img_nii.header)
        nib.save(fixed_nii, m["lbl"])
        
        # Verify
        verify_nii = nib.load(m["lbl"])
        if verify_nii.shape == target_shape:
            fixed += 1
        else:
            print(f"   ❌ Failed to fix {m['name']}: got {verify_nii.shape}")
    
    print(f"\n✅ Fixed {fixed}/{len(mismatches)} mismatches")

# Final verification
print("\n" + "=" * 70)
print("🔬 RE-VERIFICATION")
print("=" * 70)

remaining_mismatches = 0
for pair in pairs:
    if not (os.path.exists(pair["img"]) and os.path.exists(pair["lbl"])):
        continue
    img_shape = nib.load(pair["img"]).shape
    lbl_shape = nib.load(pair["lbl"]).shape
    if img_shape != lbl_shape:
        remaining_mismatches += 1
        print(f"   ❌ Still mismatched: {pair['name']}")

if remaining_mismatches == 0:
    print("✅ All 131 pairs now have matching shapes!")
else:
    print(f"⚠️  {remaining_mismatches} pairs still have issues")

print("=" * 70)

In [ ]:
"""
=========================================================================
nnU-Net v2 — FINGERPRINTING + PLANNING + PREPROCESSING (Retry)
Shape mismatch fixed, ab clean run hoga
=========================================================================
"""

import os
import subprocess
import time
from datetime import datetime

# Ensure env variables
NNUNET_BASE = "/workspace/liver_tumor_project/models/nnunet_v2"
os.environ["nnUNet_raw"] = f"{NNUNET_BASE}/nnUNet_raw"
os.environ["nnUNet_preprocessed"] = f"{NNUNET_BASE}/nnUNet_preprocessed"
os.environ["nnUNet_results"] = f"{NNUNET_BASE}/nnUNet_results"

# Log file
LOG_DIR = "/workspace/liver_tumor_project/logs/nnunet"
os.makedirs(LOG_DIR, exist_ok=True)
LOG_FILE = f"{LOG_DIR}/planning_{datetime.now().strftime('%Y%m%d_%H%M%S')}.log"

def log(msg, also_file=True):
    ts = datetime.now().strftime('%H:%M:%S')
    line = f"[{ts}] {msg}"
    print(line)
    if also_file:
        with open(LOG_FILE, 'a') as f:
            f.write(line + "\n")

def run_command(cmd, description, timeout=None):
    """Run shell command with real-time output and logging."""
    log(f"\n{'='*70}")
    log(f"🚀 {description}")
    log(f"{'='*70}")
    log(f"Command: {' '.join(cmd) if isinstance(cmd, list) else cmd}")
    log(f"Started at: {datetime.now().strftime('%H:%M:%S')}")
    
    t0 = time.time()
    env = os.environ.copy()
    
    try:
        result = subprocess.run(
            cmd,
            shell=isinstance(cmd, str),
            env=env,
            capture_output=True,
            text=True,
            timeout=timeout,
        )
        
        elapsed = time.time() - t0
        
        if result.stdout:
            log("\n📤 STDOUT (last 30 lines):")
            for line in result.stdout.strip().split('\n')[-30:]:
                log(f"  {line}", also_file=True)
        
        if result.stderr:
            log("\n⚠️  STDERR (last 20 lines):")
            for line in result.stderr.strip().split('\n')[-20:]:
                log(f"  {line}", also_file=True)
        
        if result.returncode == 0:
            log(f"\n✅ SUCCESS in {elapsed:.1f}s ({elapsed/60:.1f} min)")
            return True
        else:
            log(f"\n❌ FAILED with code {result.returncode} after {elapsed:.1f}s")
            return False
            
    except subprocess.TimeoutExpired:
        log(f"\n⏱️  TIMEOUT after {timeout}s")
        return False
    except Exception as e:
        log(f"\n❌ ERROR: {e}")
        return False


log("=" * 70)
log("🌟 nnU-Net v2 PIPELINE — RETRY AFTER SHAPE FIX")
log("=" * 70)
log(f"Dataset ID: 1 (LiTS)")
log(f"Log file:   {LOG_FILE}")

# ============================================================
# STEP 1: Fingerprint Extraction (WITHOUT verify since we already checked)
# ============================================================
success = run_command(
    cmd=["nnUNetv2_extract_fingerprint", "-d", "1"],  # removed --verify_dataset_integrity
    description="STEP 1/3: Extract dataset fingerprint",
    timeout=1800,
)

if not success:
    log("\n❌ Fingerprint extraction failed even after shape fix.")
    log("Check nnU-Net installation and dataset paths.")
    raise SystemExit("Stopping pipeline")

# ============================================================
# STEP 2: Experiment Planning
# ============================================================
success = run_command(
    cmd=["nnUNetv2_plan_experiment", "-d", "1"],
    description="STEP 2/3: Plan experiment (auto-config architecture)",
    timeout=600,
)

if not success:
    log("\n❌ Experiment planning failed.")
    raise SystemExit("Stopping pipeline")

# ============================================================
# STEP 3: Full Preprocessing
# ============================================================
success = run_command(
    cmd=["nnUNetv2_preprocess", "-d", "1", "-c", "3d_fullres", "-np", "8"],
    description="STEP 3/3: Full preprocessing (3d_fullres, 8 workers)",
    timeout=7200,
)

if not success:
    log("\n❌ Preprocessing failed.")
    raise SystemExit("Stopping pipeline")


# ============================================================
# FINAL VERIFICATION + PLAN INSPECTION
# ============================================================
log("\n" + "=" * 70)
log("📊 FINAL VERIFICATION & AUTO-CONFIG SUMMARY")
log("=" * 70)

preprocessed_dir = f"{os.environ['nnUNet_preprocessed']}/Dataset001_LiTS"
if os.path.exists(preprocessed_dir):
    log(f"\n📁 Preprocessed directory: {preprocessed_dir}")
    
    subdirs = os.listdir(preprocessed_dir)
    log(f"   Contents: {subdirs}")
    
    # Show 3d_fullres files
    fullres_dir = f"{preprocessed_dir}/nnUNetPlans_3d_fullres"
    if os.path.exists(fullres_dir):
        all_files = os.listdir(fullres_dir)
        npz_files = [f for f in all_files if f.endswith('.npz')]
        pkl_files = [f for f in all_files if f.endswith('.pkl')]
        log(f"   ✅ 3d_fullres files:")
        log(f"      .npz (data):   {len(npz_files)} cases")
        log(f"      .pkl (props):  {len(pkl_files)} property files")
    
    # Inspect plans
    plans_file = f"{preprocessed_dir}/nnUNetPlans.json"
    if os.path.exists(plans_file):
        import json
        with open(plans_file) as f:
            plans = json.load(f)
        
        log(f"\n📋 nnU-Net AUTO-CONFIGURED PLAN:")
        log(f"   Dataset spacing:  {plans.get('original_median_spacing_after_transp')}")
        log(f"   Median shape:     {plans.get('original_median_shape_after_transp')}")
        
        if "configurations" in plans and "3d_fullres" in plans["configurations"]:
            config = plans["configurations"]["3d_fullres"]
            log(f"\n   🏗️  3d_fullres configuration:")
            log(f"      Patch size:    {config.get('patch_size')}")
            log(f"      Batch size:    {config.get('batch_size')}")
            log(f"      Spacing:       {config.get('spacing')}")
            log(f"      Network name:  {config.get('UNet_class_name', 'PlainConvUNet')}")
            log(f"      # stages:      {config.get('n_conv_per_stage_encoder', 'N/A')}")
            
            if "architecture" in config:
                arch = config["architecture"]
                log(f"      Architecture:  {arch.get('network_class_name', 'Standard UNet')}")

# Show dataset fingerprint
fingerprint_file = f"{preprocessed_dir}/dataset_fingerprint.json"
if os.path.exists(fingerprint_file):
    import json
    with open(fingerprint_file) as f:
        fp = json.load(f)
    log(f"\n🔬 DATASET FINGERPRINT:")
    if "spacings" in fp and len(fp["spacings"]) > 0:
        import numpy as np
        spacings = np.array(fp["spacings"])
        log(f"   Volumes analyzed: {len(spacings)}")
        log(f"   Median spacing:   {np.median(spacings, axis=0).tolist()}")
    if "shapes_after_crop" in fp:
        log(f"   Shape range:      min-max varying")

# Disk usage
log(f"\n💾 Disk usage after preprocessing:")
os.system(f"du -sh {NNUNET_BASE}/nnUNet_preprocessed 2>&1 | tee -a {LOG_FILE}")
os.system(f"df -h /workspace | tail -1 | tee -a {LOG_FILE}")

log("\n" + "=" * 70)
log("🎉 nnU-Net v2 SETUP COMPLETE — READY FOR TRAINING")
log("=" * 70)
log(f"\n📄 Full log: {LOG_FILE}")

In [ ]:
"""
=========================================================================
FIX — Corrected custom trainer for latest nnU-Net v2 API
=========================================================================
"""

import os
import subprocess
import nnunetv2

# Fix path
nnunet_dir = os.path.dirname(nnunetv2.__file__)
custom_trainer_path = f"{nnunet_dir}/training/nnUNetTrainer/variants/training_length/nnUNetTrainer_500epochs.py"

# CORRECTED trainer — uses **kwargs to accept any nnU-Net API version
corrected_trainer_code = '''"""
Custom nnU-Net trainer with 500 epochs
Compatible with latest nnU-Net v2 API (uses **kwargs for future-proofing)
"""
from nnunetv2.training.nnUNetTrainer.nnUNetTrainer import nnUNetTrainer


class nnUNetTrainer_500epochs(nnUNetTrainer):
    """nnU-Net trainer with 500 epochs (balanced middle ground for LiTS baseline)."""
    
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.num_epochs = 500
'''

# Overwrite the buggy file
with open(custom_trainer_path, 'w') as f:
    f.write(corrected_trainer_code)

print(f"✅ Corrected trainer saved: {custom_trainer_path}")

# Verify import works
print("\n🔍 Verifying corrected trainer...")
result = subprocess.run(
    ["python", "-c", 
     """
from nnunetv2.training.nnUNetTrainer.variants.training_length.nnUNetTrainer_500epochs import nnUNetTrainer_500epochs
import inspect

# Check inheritance
print(f'Class: {nnUNetTrainer_500epochs.__name__}')
print(f'Bases: {[b.__name__ for b in nnUNetTrainer_500epochs.__bases__]}')

# Show parent __init__ signature for verification  
from nnunetv2.training.nnUNetTrainer.nnUNetTrainer import nnUNetTrainer
sig = inspect.signature(nnUNetTrainer.__init__)
print(f'Parent __init__ params: {list(sig.parameters.keys())}')
     """],
    capture_output=True, text=True, timeout=30
)

print(f"stdout: {result.stdout}")
if result.stderr:
    print(f"stderr: {result.stderr[:300]}")

if result.returncode == 0:
    print("\n✅ Trainer verified — ready to retry training")
else:
    print("\n⚠️  Still has issues — need to debug")

In [ ]:
"""
=========================================================================
FINAL FIX — Use EXACT nnU-Net v2.8+ signature
=========================================================================
"""

import os
import shutil
import subprocess
import time
from datetime import datetime
import nnunetv2

# Cleanup old files and cache
nnunet_dir = os.path.dirname(nnunetv2.__file__)
variants_dir = f"{nnunet_dir}/training/nnUNetTrainer/variants/training_length"
trainer_file = f"{variants_dir}/nnUNetTrainer_500epochs.py"
pycache_dir = f"{variants_dir}/__pycache__"

print("=" * 70)
print("🧹 STEP 1: Clean old files and cache")
print("=" * 70)

if os.path.exists(trainer_file):
    os.remove(trainer_file)
    print(f"🗑️  Removed: {trainer_file}")

if os.path.exists(pycache_dir):
    shutil.rmtree(pycache_dir)
    print(f"🗑️  Removed cache: {pycache_dir}")

# ============================================================
# Write CORRECT trainer with EXACT current API signature
# ============================================================
print("\n" + "=" * 70)
print("✍️  STEP 2: Write correct trainer with exact signature")
print("=" * 70)

# nnU-Net v2.8+ signature: plans, configuration, fold, dataset_json, device
correct_code = '''import torch
from nnunetv2.training.nnUNetTrainer.nnUNetTrainer import nnUNetTrainer


class nnUNetTrainer_500epochs(nnUNetTrainer):
    """500 epoch trainer for LiTS baseline (nnU-Net v2.8+ compatible)."""
    
    def __init__(self, plans: dict, configuration: str, fold: int, 
                 dataset_json: dict, 
                 device: torch.device = torch.device("cuda")):
        super().__init__(plans, configuration, fold, dataset_json, device)
        self.num_epochs = 500
'''

os.makedirs(variants_dir, exist_ok=True)
with open(trainer_file, 'w') as f:
    f.write(correct_code)

print(f"✅ Written: {trainer_file}")

# Print content for verification
print("\n📄 File content:")
print("-" * 70)
with open(trainer_file, 'r') as f:
    print(f.read())
print("-" * 70)

# ============================================================
# Test in fresh subprocess (no cache)
# ============================================================
print("\n" + "=" * 70)
print("🧪 STEP 3: Test import + instantiation signature")
print("=" * 70)

test_script = """
import inspect
from nnunetv2.training.nnUNetTrainer.nnUNetTrainer import nnUNetTrainer
from nnunetv2.training.nnUNetTrainer.variants.training_length.nnUNetTrainer_500epochs import nnUNetTrainer_500epochs

parent_sig = inspect.signature(nnUNetTrainer.__init__)
child_sig = inspect.signature(nnUNetTrainer_500epochs.__init__)

print('Parent nnUNetTrainer signature:')
print(f'  {parent_sig}')
print()
print('Child nnUNetTrainer_500epochs signature:')
print(f'  {child_sig}')
print()

# Check they match
parent_params = list(parent_sig.parameters.keys())
child_params = list(child_sig.parameters.keys())

if parent_params == child_params:
    print(f'MATCH: Both have params {parent_params}')
else:
    print(f'MISMATCH!')
    print(f'  Parent: {parent_params}')
    print(f'  Child:  {child_params}')
"""

result = subprocess.run(
    ["python", "-B", "-c", test_script],
    capture_output=True, text=True, timeout=30
)

print(f"stdout:\n{result.stdout}")
if result.stderr:
    print(f"stderr:\n{result.stderr[:500]}")

if "MATCH" not in result.stdout:
    print("\n❌ Signature mismatch! Cannot proceed.")
    raise SystemExit("Fix failed")
else:
    print("\n✅ Signature matches parent — safe to launch")

# ============================================================
# Launch training
# ============================================================
NNUNET_BASE = "/workspace/liver_tumor_project/models/nnunet_v2"
os.environ["nnUNet_raw"] = f"{NNUNET_BASE}/nnUNet_raw"
os.environ["nnUNet_preprocessed"] = f"{NNUNET_BASE}/nnUNet_preprocessed"
os.environ["nnUNet_results"] = f"{NNUNET_BASE}/nnUNet_results"

LOG_DIR = "/workspace/liver_tumor_project/logs/nnunet"
TRAINING_LOG = f"{LOG_DIR}/training_fold0_500ep_{datetime.now().strftime('%Y%m%d_%H%M%S')}.log"

print("\n" + "=" * 70)
print("🚀 STEP 4: LAUNCH TRAINING")
print("=" * 70)
print(f"Started:  {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Expected: ~12-14 hours (500 epochs)")
print(f"Log:      {TRAINING_LOG}")
print("=" * 70)

env = os.environ.copy()

cmd = [
    "nnUNetv2_train",
    "1", "3d_fullres", "0",
    "-tr", "nnUNetTrainer_500epochs",
    "--npz",
]

print(f"\n📝 {' '.join(cmd)}\n")

t0 = time.time()

with open(TRAINING_LOG, 'w') as log_f:
    log_f.write(f"Training started at {datetime.now()}\n\n")
    log_f.flush()
    
    process = subprocess.Popen(
        cmd, env=env,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1,
    )
    
    try:
        for line in process.stdout:
            print(line.rstrip())
            log_f.write(line)
            log_f.flush()
        
        process.wait()
        elapsed = time.time() - t0
        
        print("\n" + "=" * 70)
        if process.returncode == 0:
            print(f"✅ TRAINING COMPLETE in {elapsed/3600:.2f} hours")
        else:
            print(f"❌ FAILED (code {process.returncode}) after {elapsed/3600:.2f} hours")
        print("=" * 70)
        
    except KeyboardInterrupt:
        process.terminate()
        print(f"\n⚠️  Interrupted after {(time.time()-t0)/60:.1f} min")

In [ ]:
"""
=========================================================================
FINAL RESULT ANALYSIS — nnU-Net v2 First Baseline
Extract per-class Dice, loss curves, best checkpoints
=========================================================================
"""

import os
import json
import numpy as np
from datetime import datetime

# Result directory
NNUNET_BASE = "/workspace/liver_tumor_project/models/nnunet_v2"
RESULT_DIR = f"{NNUNET_BASE}/nnUNet_results/Dataset001_LiTS/nnUNetTrainer_500epochs__nnUNetPlans__3d_fullres/fold_0"

print("=" * 70)
print("📊 nnU-Net v2 — FIRST BASELINE FINAL RESULTS")
print("=" * 70)

# Check what's saved
print(f"\n📁 Result directory contents:")
if os.path.exists(RESULT_DIR):
    for f in sorted(os.listdir(RESULT_DIR)):
        full = os.path.join(RESULT_DIR, f)
        if os.path.isfile(full):
            size_mb = os.path.getsize(full) / (1024**2)
            print(f"   {f}  ({size_mb:.1f} MB)")
        else:
            print(f"   {f}/  (folder)")
else:
    print(f"   ❌ Directory not found: {RESULT_DIR}")

# Load progress.json (nnU-Net saves training history)
progress_files = [
    f"{RESULT_DIR}/progress.json",
    f"{RESULT_DIR}/training_log.json",
]

progress = None
for pf in progress_files:
    if os.path.exists(pf):
        with open(pf) as f:
            progress = json.load(f)
        print(f"\n✅ Loaded: {pf}")
        break

# Also check text log
log_files = [f for f in os.listdir(RESULT_DIR) if 'training_log' in f and f.endswith('.txt')]
if log_files:
    log_path = f"{RESULT_DIR}/{log_files[-1]}"
    print(f"\n📄 Latest training log: {log_files[-1]}")
    
    # Extract final epoch stats from log
    print(f"\n📊 Last 20 lines from training log:")
    with open(log_path, 'r') as f:
        lines = f.readlines()
        for line in lines[-20:]:
            print(f"   {line.rstrip()}")

# Load checkpoint metadata
checkpoint_paths = [
    f"{RESULT_DIR}/checkpoint_best.pth",
    f"{RESULT_DIR}/checkpoint_final.pth",
]

print(f"\n💾 Checkpoints saved:")
for cp in checkpoint_paths:
    if os.path.exists(cp):
        size_mb = os.path.getsize(cp) / (1024**2)
        print(f"   ✅ {os.path.basename(cp)} ({size_mb:.1f} MB)")
    else:
        print(f"   ❌ {os.path.basename(cp)} not found")

# Extract per-class metrics from progress.json if available
if progress:
    print(f"\n📈 TRAINING PROGRESSION:")
    
    if isinstance(progress, dict):
        # Common nnU-Net progress format
        if 'epoch' in progress:
            n_epochs = len(progress.get('epoch', []))
            print(f"   Total epochs: {n_epochs}")
        
        if 'val_dice_per_class' in progress or 'mean_fg_dice' in progress:
            # Try common keys
            for key in ['val_dice_per_class', 'mean_fg_dice', 'dice_per_class_or_region']:
                if key in progress:
                    values = progress[key]
                    if isinstance(values, list) and len(values) > 0:
                        last_val = values[-1]
                        best_val = max(values) if isinstance(values[0], (int, float)) else None
                        print(f"   {key}:")
                        print(f"     Final:  {last_val}")
                        if best_val:
                            print(f"     Best:   {best_val}")

# Check summary.json (often contains final metrics)
summary_path = f"{RESULT_DIR}/summary.json"
if os.path.exists(summary_path):
    with open(summary_path) as f:
        summary = json.load(f)
    print(f"\n📊 SUMMARY.JSON:")
    print(json.dumps(summary, indent=2)[:2000])

print("\n" + "=" * 70)
print("✅ ANALYSIS COMPLETE")
print("=" * 70)

In [ ]:
"""
=========================================================================
DEEP RESULT EXTRACTION — Per-class metrics + progress visualization
=========================================================================
"""

import os
import json
import numpy as np
from IPython.display import Image, display

RESULT_DIR = "/workspace/liver_tumor_project/models/nnunet_v2/nnUNet_results/Dataset001_LiTS/nnUNetTrainer_500epochs__nnUNetPlans__3d_fullres/fold_0"

print("=" * 70)
print("📊 DEEP RESULT EXTRACTION")
print("=" * 70)

# ============================================================
# STEP 1: Check validation folder
# ============================================================
print("\n📁 Validation folder contents:")
val_dir = f"{RESULT_DIR}/validation"
if os.path.exists(val_dir):
    files = sorted(os.listdir(val_dir))
    print(f"   Total files: {len(files)}")
    print(f"   Sample files:")
    for f in files[:5]:
        size_mb = os.path.getsize(f"{val_dir}/{f}") / (1024**2)
        print(f"     {f} ({size_mb:.2f} MB)")
    if len(files) > 5:
        print(f"     ... and {len(files)-5} more")

# ============================================================
# STEP 2: Show progress.png
# ============================================================
progress_png = f"{RESULT_DIR}/progress.png"
if os.path.exists(progress_png):
    print(f"\n📈 Training progress visualization:")
    display(Image(filename=progress_png))

# ============================================================
# STEP 3: Extract per-class Dice from validation
# ============================================================
print(f"\n" + "=" * 70)
print("🔬 EXTRACTING PER-CLASS DICE FROM VALIDATION")
print("=" * 70)

# Look for summary.json in validation
val_summary = f"{val_dir}/summary.json"
if os.path.exists(val_summary):
    with open(val_summary) as f:
        summary = json.load(f)
    
    print(f"\n📊 Validation summary structure:")
    print(f"   Keys: {list(summary.keys())}")
    
    # Extract mean metrics
    if 'mean' in summary:
        print(f"\n📊 MEAN METRICS ACROSS VALIDATION SET:")
        for class_id, metrics in summary['mean'].items():
            class_name = {'0': 'Background', '1': 'Liver', '2': 'Tumor'}.get(str(class_id), f'Class {class_id}')
            print(f"\n   {class_name} (class {class_id}):")
            if isinstance(metrics, dict):
                for metric, value in metrics.items():
                    if isinstance(value, (int, float)):
                        print(f"     {metric:20s}: {value:.4f}")
    
    # Extract foreground_mean (this is important!)
    if 'foreground_mean' in summary:
        print(f"\n📊 FOREGROUND MEAN (Liver + Tumor):")
        fg = summary['foreground_mean']
        for metric, value in fg.items():
            if isinstance(value, (int, float)):
                print(f"     {metric:20s}: {value:.4f}")
    
    # Per-case metrics
    if 'metric_per_case' in summary:
        cases = summary['metric_per_case']
        print(f"\n📊 PER-CASE STATISTICS ({len(cases)} cases):")
        
        # Aggregate per class
        for class_key in ['1', '2']:
            class_name = {'1': 'Liver', '2': 'Tumor'}[class_key]
            dice_values = []
            for case in cases:
                if 'metrics' in case and class_key in case['metrics']:
                    dice = case['metrics'][class_key].get('Dice', None)
                    if dice is not None:
                        dice_values.append(dice)
            
            if dice_values:
                print(f"\n   {class_name} (n={len(dice_values)}):")
                print(f"     Mean Dice:   {np.mean(dice_values):.4f}")
                print(f"     Std Dice:    {np.std(dice_values):.4f}")
                print(f"     Median:      {np.median(dice_values):.4f}")
                print(f"     Min:         {np.min(dice_values):.4f}")
                print(f"     Max:         {np.max(dice_values):.4f}")
                print(f"     25th %ile:   {np.percentile(dice_values, 25):.4f}")
                print(f"     75th %ile:   {np.percentile(dice_values, 75):.4f}")
else:
    print(f"   ❌ summary.json not found in {val_dir}")
    print(f"   Files present:")
    if os.path.exists(val_dir):
        for f in sorted(os.listdir(val_dir))[:20]:
            print(f"     {f}")

print("\n" + "=" * 70)
print("✅ EXTRACTION COMPLETE")
print("=" * 70)

In [ ]:
"""
=========================================================================
STEP 1: DIAGNOSTIC — Understand nnU-Net's saved artifacts
No changes made, just exploration
=========================================================================
"""

import os
import json
import glob
import numpy as np
import nibabel as nib

print("=" * 70)
print("🔬 DIAGNOSTIC — Exploring nnU-Net saved data")
print("=" * 70)

# ============================================================
# 1. Find validation predictions folder
# ============================================================
RESULT_DIR = "/workspace/liver_tumor_project/models/nnunet_v2/nnUNet_results/Dataset001_LiTS/nnUNetTrainer_500epochs__nnUNetPlans__3d_fullres/fold_0"
VAL_DIR = f"{RESULT_DIR}/validation"

print(f"\n📁 Validation folder: {VAL_DIR}")

if not os.path.exists(VAL_DIR):
    print("❌ Not found!")
else:
    all_files = sorted(os.listdir(VAL_DIR))
    
    # Categorize files
    nii_files = [f for f in all_files if f.endswith('.nii.gz')]
    npz_files = [f for f in all_files if f.endswith('.npz')]
    json_files = [f for f in all_files if f.endswith('.json')]
    pkl_files = [f for f in all_files if f.endswith('.pkl')]
    other = [f for f in all_files if not any(f.endswith(x) for x in ['.nii.gz', '.npz', '.json', '.pkl'])]
    
    print(f"   .nii.gz files: {len(nii_files)}")
    print(f"   .npz files:    {len(npz_files)}")
    print(f"   .json files:   {len(json_files)}")
    print(f"   .pkl files:    {len(pkl_files)}")
    print(f"   Other:         {len(other)}")
    
    print(f"\n   Sample .nii.gz (first 5):")
    for f in nii_files[:5]:
        print(f"     {f}")
    
    print(f"\n   JSON files:")
    for f in json_files:
        print(f"     {f}")

# ============================================================
# 2. Load summary.json — check its structure
# ============================================================
summary_path = f"{VAL_DIR}/summary.json"
if os.path.exists(summary_path):
    with open(summary_path) as f:
        summary = json.load(f)
    
    print(f"\n📊 summary.json structure:")
    print(f"   Top-level keys: {list(summary.keys())}")
    
    # Check metric_per_case structure
    if 'metric_per_case' in summary:
        cases = summary['metric_per_case']
        print(f"\n   metric_per_case: {len(cases)} entries")
        
        if len(cases) > 0:
            first_case = cases[0]
            print(f"\n   First case structure:")
            print(f"     Top keys: {list(first_case.keys())}")
            
            if 'metrics' in first_case:
                print(f"     metrics keys: {list(first_case['metrics'].keys())}")
                for k in first_case['metrics']:
                    print(f"       Class {k}: {list(first_case['metrics'][k].keys())}")
                    for mk, mv in first_case['metrics'][k].items():
                        print(f"         {mk} = {mv}")
                    break  # Just show first class

# ============================================================
# 3. Investigate the 39% outlier case + nan cases
# ============================================================
print("\n" + "=" * 70)
print("🔍 INVESTIGATING OUTLIERS")
print("=" * 70)

if 'metric_per_case' in summary:
    cases = summary['metric_per_case']
    
    # Analyze each case
    outlier_cases = []
    empty_tumor_cases = []
    
    for case in cases:
        # Get case identifier
        case_id = "unknown"
        if 'reference_file' in case:
            case_id = os.path.basename(case['reference_file']).replace('.nii.gz', '')
        elif 'prediction_file' in case:
            case_id = os.path.basename(case['prediction_file']).replace('.nii.gz', '')
        
        if 'metrics' in case:
            m = case['metrics']
            
            # Liver Dice
            liver_dice = m.get('1', {}).get('Dice', None)
            # Tumor Dice
            tumor_dice = m.get('2', {}).get('Dice', None)
            # Tumor ground truth voxels
            tumor_n_ref = m.get('2', {}).get('n_ref', 0)
            
            # Low liver outlier
            if liver_dice is not None and liver_dice < 0.7:
                outlier_cases.append({
                    'case': case_id,
                    'liver_dice': liver_dice,
                    'tumor_dice': tumor_dice,
                    'tumor_n_ref': tumor_n_ref,
                })
            
            # Empty tumor
            if tumor_n_ref == 0:
                empty_tumor_cases.append({
                    'case': case_id,
                    'liver_dice': liver_dice,
                    'tumor_dice': tumor_dice,
                    'tumor_n_ref': tumor_n_ref,
                })
    
    print(f"\n🚨 LIVER OUTLIER CASES (Dice < 0.7):")
    if outlier_cases:
        for c in outlier_cases:
            print(f"   {c['case']}: liver={c['liver_dice']:.4f}, tumor={c['tumor_dice']}, tumor_n_ref={c['tumor_n_ref']:.0f}")
    else:
        print(f"   None found")
    
    print(f"\n⚠️  EMPTY TUMOR CASES (no tumor in GT):")
    if empty_tumor_cases:
        for c in empty_tumor_cases:
            print(f"   {c['case']}: liver={c['liver_dice']:.4f}, tumor_dice={c['tumor_dice']}, tumor_n_ref={c['tumor_n_ref']:.0f}")
    else:
        print(f"   None found")
    
    # Full case-by-case dice
    print(f"\n📋 ALL CASES DICE (sorted by tumor dice):")
    case_metrics = []
    for case in cases:
        case_id = "unknown"
        if 'reference_file' in case:
            case_id = os.path.basename(case['reference_file']).replace('.nii.gz', '')
        m = case.get('metrics', {})
        liver = m.get('1', {}).get('Dice', np.nan)
        tumor = m.get('2', {}).get('Dice', np.nan)
        tumor_n = m.get('2', {}).get('n_ref', 0)
        case_metrics.append((case_id, liver, tumor, tumor_n))
    
    # Sort by tumor Dice (nan handling)
    case_metrics.sort(key=lambda x: (np.isnan(x[2]) if isinstance(x[2], float) else False, x[2] if not (isinstance(x[2], float) and np.isnan(x[2])) else 0))
    
    print(f"\n   {'Case':<15} {'Liver Dice':>12} {'Tumor Dice':>12} {'Tumor voxels':>15}")
    print(f"   {'-'*15} {'-'*12} {'-'*12} {'-'*15}")
    for case_id, liver, tumor, tumor_n in case_metrics:
        tumor_str = f"{tumor:.4f}" if not (isinstance(tumor, float) and np.isnan(tumor)) else "  NAN"
        print(f"   {case_id:<15} {liver:>12.4f} {tumor_str:>12} {tumor_n:>15.0f}")

print("\n" + "=" * 70)
print("✅ DIAGNOSTIC COMPLETE")
print("=" * 70)

In [ ]:
"""
=========================================================================
DEBUG VOLUME-48 — Investigate the 39% liver Dice outlier
=========================================================================
"""

import os
import numpy as np
import nibabel as nib
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

print("=" * 70)
print("🔬 VOLUME-48 DEEP INVESTIGATION")
print("=" * 70)

# Paths
NNUNET_BASE = "/workspace/liver_tumor_project/models/nnunet_v2"
V2_PREPROCESSED = "/workspace/liver_tumor_project/data/preprocessed"
RAW = "/workspace/liver_tumor_project/data/raw/LiTS/LiTS(train_test)"

# Files
paths = {
    'original_ct':       f"{RAW}/train_CT/volume-48.nii",
    'original_mask':     f"{RAW}/train_mask/segmentation-48.nii",
    'v2_ct':             f"{V2_PREPROCESSED}/imagesTr/volume-48.nii.gz",
    'v2_mask':           f"{V2_PREPROCESSED}/labelsTr/segmentation-48.nii.gz",
    'nnunet_prediction': f"{NNUNET_BASE}/nnUNet_results/Dataset001_LiTS/nnUNetTrainer_500epochs__nnUNetPlans__3d_fullres/fold_0/validation/volume-48.nii.gz",
}

# Load all
data = {}
for key, path in paths.items():
    if os.path.exists(path):
        img = nib.load(path)
        data[key] = {
            'shape': img.shape,
            'spacing': tuple(round(s, 3) for s in img.header.get_zooms()),
            'data': img.get_fdata(),
        }
        print(f"\n✅ {key}:")
        print(f"   Shape:   {data[key]['shape']}")
        print(f"   Spacing: {data[key]['spacing']}")
        
        if 'mask' in key or 'prediction' in key:
            unique, counts = np.unique(data[key]['data'].astype(int), return_counts=True)
            for u, c in zip(unique, counts):
                class_name = {0: 'BG', 1: 'Liver', 2: 'Tumor'}.get(u, str(u))
                print(f"   {class_name}: {c:,} voxels")
    else:
        print(f"\n❌ Missing: {path}")

# ============================================================
# Analyze shape reduction
# ============================================================
if 'original_ct' in data and 'v2_ct' in data:
    orig_shape = data['original_ct']['shape']
    v2_shape = data['v2_ct']['shape']
    
    orig_vox = np.prod(orig_shape)
    v2_vox = np.prod(v2_shape)
    reduction = (1 - v2_vox / orig_vox) * 100
    
    print(f"\n📉 V2 PREPROCESSING REDUCTION:")
    print(f"   Original: {orig_shape} = {orig_vox/1e6:.1f}M voxels")
    print(f"   V2:       {v2_shape}  = {v2_vox/1e6:.1f}M voxels")
    print(f"   Reduced:  {reduction:.1f}%")
    
    # If reduction > 95%, likely too aggressive
    if reduction > 95:
        print(f"   ⚠️  Very aggressive reduction — may have cropped liver!")

# ============================================================
# Analyze prediction vs ground truth
# ============================================================
if 'v2_mask' in data and 'nnunet_prediction' in data:
    gt = data['v2_mask']['data'].astype(np.uint8)
    pred = data['nnunet_prediction']['data'].astype(np.uint8)
    
    # Note: shapes may differ if nnU-Net resampled
    print(f"\n🔬 PREDICTION vs GROUND TRUTH:")
    print(f"   GT shape:   {gt.shape}")
    print(f"   Pred shape: {pred.shape}")
    
    if gt.shape == pred.shape:
        # Per class comparison
        for cls in [1, 2]:
            gt_mask = (gt == cls)
            pred_mask = (pred == cls)
            
            intersection = np.sum(gt_mask & pred_mask)
            union = np.sum(gt_mask) + np.sum(pred_mask)
            dice = 2 * intersection / union if union > 0 else 0
            
            gt_count = np.sum(gt_mask)
            pred_count = np.sum(pred_mask)
            
            class_name = {1: 'Liver', 2: 'Tumor'}[cls]
            print(f"\n   {class_name} (class {cls}):")
            print(f"     GT voxels:   {gt_count:,}")
            print(f"     Pred voxels: {pred_count:,}")
            print(f"     Overlap:     {intersection:,}")
            print(f"     Dice:        {dice:.4f}")
    else:
        print("   Shape mismatch, cannot compare directly")

# ============================================================
# Save visualization
# ============================================================
if 'v2_ct' in data and 'v2_mask' in data and 'nnunet_prediction' in data:
    ct = data['v2_ct']['data']
    gt = data['v2_mask']['data'].astype(np.uint8)
    pred = data['nnunet_prediction']['data'].astype(np.uint8)
    
    if ct.shape == gt.shape == pred.shape:
        # Find slice with liver
        liver_slices = np.where(np.any(gt >= 1, axis=(0, 1)))[0]
        if len(liver_slices) > 0:
            middle_slice = liver_slices[len(liver_slices)//2]
            
            fig, axes = plt.subplots(1, 4, figsize=(20, 5))
            
            axes[0].imshow(ct[:, :, middle_slice], cmap='gray')
            axes[0].set_title('CT Image')
            axes[0].axis('off')
            
            axes[1].imshow(gt[:, :, middle_slice], cmap='viridis', vmin=0, vmax=2)
            axes[1].set_title('Ground Truth')
            axes[1].axis('off')
            
            axes[2].imshow(pred[:, :, middle_slice], cmap='viridis', vmin=0, vmax=2)
            axes[2].set_title('Prediction')
            axes[2].axis('off')
            
            # Diff map
            diff = np.zeros_like(gt[:, :, middle_slice], dtype=np.uint8)
            diff[gt[:, :, middle_slice] == pred[:, :, middle_slice]] = 0  # Match
            diff[(gt[:, :, middle_slice] > 0) & (pred[:, :, middle_slice] == 0)] = 1  # FN
            diff[(gt[:, :, middle_slice] == 0) & (pred[:, :, middle_slice] > 0)] = 2  # FP
            
            axes[3].imshow(ct[:, :, middle_slice], cmap='gray', alpha=0.5)
            axes[3].imshow(diff, cmap='RdYlGn_r', alpha=0.5, vmin=0, vmax=2)
            axes[3].set_title('Errors (FN=orange, FP=red)')
            axes[3].axis('off')
            
            plt.suptitle(f'volume-48 (Slice {middle_slice}) — Liver Dice: 39.26%', fontsize=14)
            plt.tight_layout()
            
            viz_path = "/workspace/liver_tumor_project/results/volume_48_debug.png"
            plt.savefig(viz_path, dpi=100, bbox_inches='tight')
            plt.close()
            print(f"\n📸 Debug visualization saved: {viz_path}")

print("\n" + "=" * 70)
print("✅ INVESTIGATION COMPLETE")
print("=" * 70)

In [ ]:
"""
=========================================================================
CRITICAL BUG DIAGNOSTIC — Check spacing mismatch in ALL 131 volumes
=========================================================================
"""

import os
import nibabel as nib
import numpy as np
from tqdm import tqdm

RAW = "/workspace/liver_tumor_project/data/raw/LiTS/LiTS(train_test)"

print("=" * 70)
print("🔬 SPACING MISMATCH DETECTION — All 131 volumes")
print("=" * 70)

folders = [
    ("train_CT", "train_mask", "train"),
    ("test_CT", "test_mask", "test"),
]

mismatches = []
all_details = []

for ct_folder, mask_folder, subset in folders:
    ct_files = sorted(os.listdir(f"{RAW}/{ct_folder}"))
    
    for ct_file in tqdm(ct_files, desc=f"Checking {subset}"):
        idx = ct_file.replace("volume-", "").replace(".nii", "")
        ct_path = f"{RAW}/{ct_folder}/volume-{idx}.nii"
        mask_path = f"{RAW}/{mask_folder}/segmentation-{idx}.nii"
        
        if not (os.path.exists(ct_path) and os.path.exists(mask_path)):
            continue
        
        ct_nii = nib.load(ct_path)
        mask_nii = nib.load(mask_path)
        
        ct_spacing = tuple(round(s, 3) for s in ct_nii.header.get_zooms())
        mask_spacing = tuple(round(s, 3) for s in mask_nii.header.get_zooms())
        ct_shape = ct_nii.shape
        mask_shape = mask_nii.shape
        
        # Check spacing mismatch
        spacing_match = ct_spacing == mask_spacing
        shape_match = ct_shape == mask_shape
        
        detail = {
            'volume': f"volume-{idx}",
            'subset': subset,
            'ct_shape': ct_shape,
            'mask_shape': mask_shape,
            'ct_spacing': ct_spacing,
            'mask_spacing': mask_spacing,
            'spacing_match': spacing_match,
            'shape_match': shape_match,
        }
        all_details.append(detail)
        
        if not spacing_match:
            mismatches.append(detail)

# ============================================================
# SUMMARY
# ============================================================
print("\n" + "=" * 70)
print("📊 SUMMARY")
print("=" * 70)
print(f"\nTotal volumes checked:  {len(all_details)}")
print(f"Spacing mismatches:     {len(mismatches)}")
print(f"Spacing matches:        {len(all_details) - len(mismatches)}")

if mismatches:
    print(f"\n🚨 SPACING MISMATCH DETAILS:")
    print(f"\n{'Volume':<15} {'Subset':<8} {'CT spacing':<25} {'Mask spacing':<25}")
    print("-" * 75)
    for m in mismatches[:30]:  # Show first 30
        print(f"{m['volume']:<15} {m['subset']:<8} {str(m['ct_spacing']):<25} {str(m['mask_spacing']):<25}")
    
    if len(mismatches) > 30:
        print(f"... and {len(mismatches) - 30} more mismatches")
    
    # Common mismatch patterns
    print(f"\n📊 MISMATCH PATTERNS:")
    unique_ct_spacings = set(m['ct_spacing'] for m in mismatches)
    unique_mask_spacings = set(m['mask_spacing'] for m in mismatches)
    print(f"   Unique CT spacings in mismatches:   {len(unique_ct_spacings)}")
    for s in list(unique_ct_spacings)[:5]:
        print(f"     {s}")
    print(f"   Unique mask spacings in mismatches: {len(unique_mask_spacings)}")
    for s in list(unique_mask_spacings)[:5]:
        print(f"     {s}")

print("\n" + "=" * 70)
if len(mismatches) > 0:
    print(f"⚠️  {len(mismatches)}/{len(all_details)} volumes have header bug!")
    print(f"    This affects entire training + evaluation!")
    print(f"    FIX: Copy CT header to mask before preprocessing")
else:
    print(f"✅ All headers consistent, volume-48 may be edge case")
print("=" * 70)

In [ ]:
"""
=========================================================================
RESUME PIPELINE — Fix backup conflict + Continue from Stage 5
=========================================================================
Stages 1-4 already done. Only training + evaluation remaining.
=========================================================================
"""

import os
import shutil
import json
import time
import subprocess
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import nibabel as nib
from datetime import datetime
from tqdm import tqdm

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# ==================================================================================
# CONFIG (same as before)
# ==================================================================================
class Config:
    RAW = "/workspace/liver_tumor_project/data/raw/LiTS/LiTS(train_test)"
    V2_ROOT = "/workspace/liver_tumor_project/data/preprocessed"
    NNUNET_BASE = "/workspace/liver_tumor_project/models/nnunet_v2"
    RESULTS = "/workspace/liver_tumor_project/results"
    LOG_DIR = "/workspace/liver_tumor_project/logs/master_pipeline"
    
    DATASET_ID = 1
    DATASET_NAME = "Dataset001_LiTS"
    EPOCHS = 500
    MODEL_NAME = "nnU-Net v2"
    BASELINE_KEY = "nnunet_v2"
    BUGGY_VOLUMES = [48, 49, 50, 51, 52]

cfg = Config()

os.environ["nnUNet_raw"] = f"{cfg.NNUNET_BASE}/nnUNet_raw"
os.environ["nnUNet_preprocessed"] = f"{cfg.NNUNET_BASE}/nnUNet_preprocessed"
os.environ["nnUNet_results"] = f"{cfg.NNUNET_BASE}/nnUNet_results"

os.makedirs(cfg.LOG_DIR, exist_ok=True)
LOG_FILE = f"{cfg.LOG_DIR}/resume_{datetime.now().strftime('%Y%m%d_%H%M%S')}.log"

def log(msg, print_console=True):
    ts = datetime.now().strftime('%H:%M:%S')
    line = f"[{ts}] {msg}"
    if print_console:
        print(line)
    with open(LOG_FILE, 'a') as f:
        f.write(line + "\n")

BASELINE_OUT = f"{cfg.RESULTS}/baselines/{cfg.BASELINE_KEY}"
for subdir in ["training", "predictions_val", "predictions_test", 
               "metrics", "visualizations"]:
    os.makedirs(f"{BASELINE_OUT}/{subdir}", exist_ok=True)

resume_start = time.time()

log("=" * 80)
log(f"🔄 RESUMING PIPELINE — {cfg.MODEL_NAME}")
log("=" * 80)
log(f"Skipping Stages 1-4 (already done)")
log(f"Starting from Stage 5 (training)")

# ==================================================================================
# STAGE 5a: SMART BACKUP HANDLING
# ==================================================================================
log("\n" + "=" * 80)
log("🗑️  STAGE 5a: Smart cleanup + backup")
log("=" * 80)

old_train_dir = f"{os.environ['nnUNet_results']}/{cfg.DATASET_NAME}/nnUNetTrainer_500epochs__nnUNetPlans__3d_fullres/fold_0"
backup_dir = f"{old_train_dir}_buggy_backup"

# If backup already exists (from previous attempt), handle gracefully
if os.path.exists(old_train_dir):
    if os.path.exists(backup_dir):
        # Backup already exists, so this old_train_dir is from previous partial run
        # Just delete the new attempt's leftover
        log(f"   ⚠️  Backup already exists at {backup_dir}")
        log(f"   Removing partial retrain attempt: {old_train_dir}")
        shutil.rmtree(old_train_dir)
    else:
        # Normal case - move to backup
        shutil.move(old_train_dir, backup_dir)
        log(f"   📦 Old training moved to: {backup_dir}")
else:
    log(f"   ✅ No old training folder present, clean start")

# Verify clean state
if os.path.exists(old_train_dir):
    log(f"   ❌ Still exists! Force removing...")
    shutil.rmtree(old_train_dir)

log(f"   ✅ Ready for fresh training")

# ==================================================================================
# STAGE 5b: RETRAIN
# ==================================================================================
log("\n" + "=" * 80)
log(f"🚀 STAGE 5b: Retrain nnU-Net {cfg.EPOCHS} epochs (~4 hours)")
log("=" * 80)
log(f"Started at: {datetime.now().strftime('%H:%M:%S')}")

train_start = time.time()

process = subprocess.Popen(
    ["nnUNetv2_train", str(cfg.DATASET_ID), "3d_fullres", "0",
     "-tr", "nnUNetTrainer_500epochs", "--npz"],
    env=os.environ.copy(),
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1,
)

for line in process.stdout:
    line = line.rstrip()
    if line:
        if any(k in line for k in ['Epoch', 'train_loss', 'val_loss', 'Pseudo dice', 
                                     'best', 'Yayy', 'Validation', 'Mean Val']):
            print(line)
        with open(LOG_FILE, 'a') as f:
            f.write(line + "\n")

process.wait()
train_time = time.time() - train_start

if process.returncode != 0:
    log(f"❌ Training failed after {train_time/3600:.2f}h")
    raise SystemExit("Training failed")

log(f"✅ Training done in {train_time/3600:.2f}h")

RESULT_DIR = f"{os.environ['nnUNet_results']}/{cfg.DATASET_NAME}/nnUNetTrainer_500epochs__nnUNetPlans__3d_fullres/fold_0"

# Copy training artifacts
for f in ["progress.png", "checkpoint_best.pth", "checkpoint_final.pth"]:
    src = f"{RESULT_DIR}/{f}"
    if os.path.exists(src):
        shutil.copy2(src, f"{BASELINE_OUT}/training/{f}")

for f in os.listdir(RESULT_DIR):
    if f.startswith("training_log_") and f.endswith(".txt"):
        shutil.copy2(f"{RESULT_DIR}/{f}", f"{BASELINE_OUT}/training/{f}")

# ==================================================================================
# STAGE 7: Test set predictions
# ==================================================================================
log("\n" + "=" * 80)
log("🎯 STAGE 7: Predict on test set (20 volumes)")
log("=" * 80)

DATASET_DIR = f"{os.environ['nnUNet_raw']}/{cfg.DATASET_NAME}"
TEST_PRED_DIR = f"{BASELINE_OUT}/predictions_test"

r = subprocess.run(
    ["nnUNetv2_predict", "-i", f"{DATASET_DIR}/imagesTs",
     "-o", TEST_PRED_DIR, "-d", str(cfg.DATASET_ID), "-c", "3d_fullres",
     "-f", "0", "-tr", "nnUNetTrainer_500epochs",
     "-npp", "4", "-nps", "4"],
    capture_output=True, text=True, env=os.environ.copy(), timeout=3600
)
if r.returncode == 0:
    n_preds = len([f for f in os.listdir(TEST_PRED_DIR) if f.endswith('.nii.gz')])
    log(f"✅ {n_preds} test predictions saved")
else:
    log(f"⚠️  Test prediction warning: {r.stderr[:300]}")

# Copy validation predictions
val_src = f"{RESULT_DIR}/validation"
val_dst = f"{BASELINE_OUT}/predictions_val"
if os.path.exists(val_src):
    for f in os.listdir(val_src):
        if f.endswith('.nii.gz'):
            shutil.copy2(f"{val_src}/{f}", f"{val_dst}/{f}")
    if os.path.exists(f"{val_src}/summary.json"):
        shutil.copy2(f"{val_src}/summary.json", f"{val_dst}/summary.json")
log(f"✅ Validation predictions copied")

# ==================================================================================
# STAGE 8-9: Compute ALL metrics + statistics
# ==================================================================================
log("\n" + "=" * 80)
log("📊 STAGE 8-9: Compute all metrics + statistics")
log("=" * 80)

# Install medpy if needed
try:
    from medpy.metric.binary import hd95, assd
except ImportError:
    subprocess.run(["pip", "install", "-q", "medpy"], check=True)
    from medpy.metric.binary import hd95, assd

def compute_all_metrics(pred, gt, spacing, cls):
    pred_bin = (pred == cls)
    gt_bin = (gt == cls)
    result = {'class': cls, 'gt_voxels': int(np.sum(gt_bin)), 'pred_voxels': int(np.sum(pred_bin))}
    
    if np.sum(gt_bin) == 0 and np.sum(pred_bin) == 0:
        result.update({'Dice': None, 'IoU': None, 'HD95': None, 'ASSD': None, 'VOE': None, 'RVD': None})
        return result
    if np.sum(gt_bin) == 0:
        result.update({'Dice': 0.0, 'IoU': 0.0, 'HD95': np.inf, 'ASSD': np.inf, 'VOE': 1.0, 'RVD': np.inf})
        return result
    if np.sum(pred_bin) == 0:
        result.update({'Dice': 0.0, 'IoU': 0.0, 'HD95': np.inf, 'ASSD': np.inf, 'VOE': 1.0, 'RVD': -1.0})
        return result
    
    intersection = np.sum(pred_bin & gt_bin)
    union = np.sum(pred_bin | gt_bin)
    dice = 2 * intersection / (np.sum(pred_bin) + np.sum(gt_bin))
    iou = intersection / union
    voe = 1 - iou
    rvd = (np.sum(pred_bin) - np.sum(gt_bin)) / np.sum(gt_bin)
    
    try: hd95_v = hd95(pred_bin, gt_bin, voxelspacing=spacing)
    except: hd95_v = np.inf
    try: assd_v = assd(pred_bin, gt_bin, voxelspacing=spacing)
    except: assd_v = np.inf
    
    result.update({'Dice': float(dice), 'IoU': float(iou), 'HD95': float(hd95_v),
                   'ASSD': float(assd_v), 'VOE': float(voe), 'RVD': float(rvd)})
    return result

def evaluate_folder(pred_dir, gt_dir):
    results = []
    pred_files = sorted([f for f in os.listdir(pred_dir) if f.endswith('.nii.gz') and 'volume' in f])
    for pred_file in tqdm(pred_files, desc=f"Eval {os.path.basename(pred_dir)}"):
        idx = pred_file.replace('.nii.gz', '').replace('volume-', '')
        gt_candidates = [f"{gt_dir}/segmentation-{idx}.nii.gz", f"{gt_dir}/volume-{idx}.nii.gz"]
        gt_path = next((p for p in gt_candidates if os.path.exists(p)), None)
        if gt_path is None: continue
        
        pred_nii = nib.load(f"{pred_dir}/{pred_file}")
        gt_nii = nib.load(gt_path)
        pred = pred_nii.get_fdata().astype(np.uint8)
        gt = gt_nii.get_fdata().astype(np.uint8)
        if pred.shape != gt.shape: continue
        
        spacing = tuple(pred_nii.header.get_zooms())
        case_result = {'volume': f"volume-{idx}", 'shape': str(pred.shape)}
        for cls in [1, 2]:
            m = compute_all_metrics(pred, gt, spacing, cls)
            class_name = {1: 'liver', 2: 'tumor'}[cls]
            for k, v in m.items():
                if k != 'class':
                    case_result[f"{class_name}_{k}"] = v
        results.append(case_result)
    return results

val_results = evaluate_folder(f"{BASELINE_OUT}/predictions_val", f"{cfg.V2_ROOT}/labelsTr")
test_results = evaluate_folder(f"{BASELINE_OUT}/predictions_test", f"{cfg.V2_ROOT}/labelsTs")

log(f"   Val cases:  {len(val_results)}")
log(f"   Test cases: {len(test_results)}")

def compute_stats(results, dataset_name):
    stats = {'dataset': dataset_name, 'n_total': len(results)}
    for organ in ['liver', 'tumor']:
        valid = [r for r in results if r.get(f'{organ}_gt_voxels', 0) > 0]
        for metric in ['Dice', 'IoU', 'HD95', 'ASSD', 'VOE', 'RVD']:
            values = [r[f'{organ}_{metric}'] for r in valid
                      if r.get(f'{organ}_{metric}') is not None 
                      and not np.isnan(r[f'{organ}_{metric}']) 
                      and not np.isinf(r[f'{organ}_{metric}'])]
            key = f'{organ}_{metric}'
            if values:
                stats[f'{key}_n'] = len(values)
                stats[f'{key}_mean'] = float(np.mean(values))
                stats[f'{key}_std'] = float(np.std(values))
                stats[f'{key}_median'] = float(np.median(values))
                stats[f'{key}_min'] = float(np.min(values))
                stats[f'{key}_max'] = float(np.max(values))
    return stats

val_stats = compute_stats(val_results, 'validation')
test_stats = compute_stats(test_results, 'test')

# Tumor size breakdown
size_groups = {'small': (0, 3000), 'medium': (3000, 30000), 'large': (30000, np.inf)}
size_stats = {}
for gname, (low, high) in size_groups.items():
    cases = [r for r in test_results 
             if low <= r.get('tumor_gt_voxels', 0) < high 
             and r.get('tumor_gt_voxels', 0) > 0]
    dices = [r['tumor_Dice'] for r in cases 
             if r.get('tumor_Dice') is not None and not np.isnan(r['tumor_Dice'])]
    if dices:
        size_stats[gname] = {'n': len(dices), 'mean_dice': float(np.mean(dices)),
                             'std_dice': float(np.std(dices))}
        log(f"   {gname} tumors (n={len(dices)}): Dice={np.mean(dices):.4f}±{np.std(dices):.4f}")

# ==================================================================================
# STAGE 10-11: Visualizations
# ==================================================================================
log("\n" + "=" * 80)
log("🎨 STAGE 10-11: Visualizations")
log("=" * 80)

VIZ_DIR = f"{BASELINE_OUT}/visualizations"

def save_slice_viz(volume_id, pred_dir, gt_dir, ct_dir, out_path, title_suffix=""):
    try:
        idx = volume_id.replace('volume-', '')
        pred = nib.load(f"{pred_dir}/volume-{idx}.nii.gz").get_fdata().astype(np.uint8)
        gt_candidates = [f"{gt_dir}/segmentation-{idx}.nii.gz", f"{gt_dir}/volume-{idx}.nii.gz"]
        gt_path = next((p for p in gt_candidates if os.path.exists(p)), None)
        if gt_path is None: return
        gt = nib.load(gt_path).get_fdata().astype(np.uint8)
        ct = nib.load(f"{ct_dir}/volume-{idx}.nii.gz").get_fdata()
        
        tumor_slices = np.where(np.any(gt == 2, axis=(0, 1)))[0]
        if len(tumor_slices) > 0:
            z = tumor_slices[len(tumor_slices)//2]
        else:
            liver_slices = np.where(np.any(gt == 1, axis=(0, 1)))[0]
            z = liver_slices[len(liver_slices)//2] if len(liver_slices) > 0 else ct.shape[2]//2
        
        fig, axes = plt.subplots(1, 4, figsize=(20, 5))
        axes[0].imshow(ct[:,:,z], cmap='gray'); axes[0].set_title('CT'); axes[0].axis('off')
        axes[1].imshow(gt[:,:,z], cmap='viridis', vmin=0, vmax=2); axes[1].set_title('Ground Truth'); axes[1].axis('off')
        axes[2].imshow(pred[:,:,z], cmap='viridis', vmin=0, vmax=2); axes[2].set_title('Prediction'); axes[2].axis('off')
        diff = np.zeros_like(gt[:,:,z])
        diff[(gt[:,:,z] > 0) & (pred[:,:,z] == 0)] = 1
        diff[(gt[:,:,z] == 0) & (pred[:,:,z] > 0)] = 2
        axes[3].imshow(ct[:,:,z], cmap='gray', alpha=0.6)
        axes[3].imshow(np.ma.masked_where(diff==0, diff), cmap='autumn', alpha=0.6, vmin=1, vmax=2)
        axes[3].set_title('Errors (FN=yellow, FP=red)'); axes[3].axis('off')
        plt.suptitle(f'{volume_id} {title_suffix}', fontsize=12)
        plt.tight_layout()
        plt.savefig(out_path, dpi=80, bbox_inches='tight')
        plt.close()
    except Exception as e:
        log(f"   Viz failed for {volume_id}: {e}")

test_with_tumor = [r for r in test_results 
                   if r.get('tumor_gt_voxels', 0) > 0 
                   and r.get('tumor_Dice') is not None]

if test_with_tumor:
    sorted_by = sorted(test_with_tumor, key=lambda x: x['tumor_Dice'])
    for i, r in enumerate(sorted_by[:3]):
        save_slice_viz(r['volume'], f"{BASELINE_OUT}/predictions_test",
                      f"{cfg.V2_ROOT}/labelsTs", f"{cfg.V2_ROOT}/imagesTs",
                      f"{VIZ_DIR}/test_worst_{i+1}_{r['volume']}.png",
                      f"(WORST #{i+1}, Tumor Dice={r['tumor_Dice']:.3f})")
    for i, r in enumerate(sorted_by[-3:]):
        save_slice_viz(r['volume'], f"{BASELINE_OUT}/predictions_test",
                      f"{cfg.V2_ROOT}/labelsTs", f"{cfg.V2_ROOT}/imagesTs",
                      f"{VIZ_DIR}/test_best_{i+1}_{r['volume']}.png",
                      f"(BEST #{i+1}, Tumor Dice={r['tumor_Dice']:.3f})")

# Tumor size vs Dice scatter
fig, ax = plt.subplots(figsize=(8, 6))
for r in test_with_tumor:
    ax.scatter(r['tumor_gt_voxels'], r['tumor_Dice'], alpha=0.6, s=50)
ax.set_xscale('log')
ax.set_xlabel('Tumor size (voxels, log scale)')
ax.set_ylabel('Tumor Dice')
ax.set_title(f'{cfg.MODEL_NAME}: Tumor Size vs Segmentation Quality')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f"{VIZ_DIR}/tumor_size_vs_dice.png", dpi=100, bbox_inches='tight')
plt.close()

if os.path.exists(f"{BASELINE_OUT}/training/progress.png"):
    shutil.copy2(f"{BASELINE_OUT}/training/progress.png", f"{VIZ_DIR}/training_curves.png")

log(f"✅ Visualizations saved")

# ==================================================================================
# STAGE 12-14: CSV + LaTeX + JSON
# ==================================================================================
log("\n" + "=" * 80)
log("💾 STAGE 12-14: Exports")
log("=" * 80)

pd.DataFrame(val_results).to_csv(f"{BASELINE_OUT}/metrics/per_volume_val.csv", index=False)
pd.DataFrame(test_results).to_csv(f"{BASELINE_OUT}/metrics/per_volume_test.csv", index=False)

def latex_row(stats, model_name):
    l_d = stats.get('liver_Dice_mean', 0) * 100
    l_ds = stats.get('liver_Dice_std', 0) * 100
    t_d = stats.get('tumor_Dice_mean', 0) * 100
    t_ds = stats.get('tumor_Dice_std', 0) * 100
    mean_d = (l_d + t_d) / 2
    l_hd = stats.get('liver_HD95_mean', 0)
    t_hd = stats.get('tumor_HD95_mean', 0)
    l_assd = stats.get('liver_ASSD_mean', 0)
    t_assd = stats.get('tumor_ASSD_mean', 0)
    return (f"{model_name} & {l_d:.2f}$\\pm${l_ds:.2f} & {t_d:.2f}$\\pm${t_ds:.2f} & "
            f"{mean_d:.2f} & {l_hd:.2f}/{t_hd:.2f} & {l_assd:.2f}/{t_assd:.2f} \\\\")

latex_test = latex_row(test_stats, cfg.MODEL_NAME)
latex_val = latex_row(val_stats, cfg.MODEL_NAME + " (val)")

with open(f"{BASELINE_OUT}/metrics/paper_table_row.tex", 'w') as f:
    f.write(f"% {cfg.MODEL_NAME} — Test set\n{latex_test}\n\n")
    f.write(f"% {cfg.MODEL_NAME} — Val set\n{latex_val}\n")

paper_results = {
    'model': cfg.MODEL_NAME,
    'config': {'epochs': cfg.EPOCHS, 'fold': 0, 'seed': 42,
               'target_spacing': (1.0, 1.0, 1.5), 'patch_size': [128, 128, 128]},
    'training': {'time_hours': train_time / 3600, 'fixed_volumes': cfg.BUGGY_VOLUMES},
    'validation_stats': val_stats,
    'test_stats': test_stats,
    'tumor_size_breakdown': size_stats,
    'per_volume_val': val_results,
    'per_volume_test': test_results,
    'latex_row_test': latex_test,
    'timestamp': datetime.now().isoformat(),
}

json_path = f"{BASELINE_OUT}/paper_results.json"
with open(json_path, 'w') as f:
    json.dump(paper_results, f, indent=2, default=str)

log(f"✅ All exports done")

# ==================================================================================
# STAGE 15: EXECUTIVE SUMMARY
# ==================================================================================
total_time = time.time() - resume_start

log("\n" + "=" * 80)
log(f"🎉 PIPELINE COMPLETE in {total_time/3600:.2f} hours")
log("=" * 80)

log(f"\n📊 FINAL RESULTS — {cfg.MODEL_NAME}")
log(f"\n{'Metric':<20} {'Val (Liver)':<15} {'Val (Tumor)':<15} {'Test (Liver)':<15} {'Test (Tumor)':<15}")
log("-" * 80)
for metric in ['Dice', 'IoU', 'HD95', 'ASSD', 'VOE', 'RVD']:
    vl = val_stats.get(f'liver_{metric}_mean', 0)
    vt = val_stats.get(f'tumor_{metric}_mean', 0)
    tl = test_stats.get(f'liver_{metric}_mean', 0)
    tt = test_stats.get(f'tumor_{metric}_mean', 0)
    log(f"{metric:<20} {vl:<15.4f} {vt:<15.4f} {tl:<15.4f} {tt:<15.4f}")

log(f"\n📄 Log:          {LOG_FILE}")
log(f"📁 All outputs:  {BASELINE_OUT}/")

print("\n" + "=" * 80)
print(f"✅ nnU-Net DONE! Check {BASELINE_OUT}/")
print("=" * 80)

In [ ]:
"""
SwinUNETR Pre-flight Check — Verify prerequisites
"""
import os
import json
import torch

print("=" * 70)
print("🔍 SWINUNETR PRE-FLIGHT CHECK")
print("=" * 70)

checks = []

# 1. nnU-Net baseline complete?
nn_json = "/workspace/liver_tumor_project/results/baselines/nnunet_v2/paper_results.json"
if os.path.exists(nn_json):
    with open(nn_json) as f:
        nn = json.load(f)
    liver = nn.get('test_stats', {}).get('liver_Dice_mean', 0)
    tumor = nn.get('test_stats', {}).get('tumor_Dice_mean', 0)
    print(f"  ✅ nnU-Net baseline: Liver {liver*100:.2f}%, Tumor {tumor*100:.2f}%")
    checks.append(True)
else:
    print(f"  ❌ nnU-Net JSON not found!")
    checks.append(False)

# 2. V2 data available?
v2_train = "/workspace/liver_tumor_project/data/preprocessed/imagesTr"
v2_test = "/workspace/liver_tumor_project/data/preprocessed/imagesTs"
train_n = len([f for f in os.listdir(v2_train) if f.endswith('.nii.gz')])
test_n = len([f for f in os.listdir(v2_test) if f.endswith('.nii.gz')])
print(f"  ✅ V2 data: {train_n} train + {test_n} test volumes")
checks.append(train_n == 111 and test_n == 20)

# 3. GPU free?
free_gb = torch.cuda.mem_get_info()[0] / 1024**3
total_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f"  GPU: {torch.cuda.get_device_name(0)}")
print(f"  Memory free: {free_gb:.1f} / {total_gb:.1f} GB")
if free_gb > 60:
    print(f"  ✅ GPU ready for SwinUNETR training")
    checks.append(True)
else:
    print(f"  ⚠️  GPU still busy - wait for previous process to finish!")
    checks.append(False)

# 4. Splits file
splits = "/workspace/liver_tumor_project/data/splits/fold_splits.json"
if os.path.exists(splits):
    with open(splits) as f:
        s = json.load(f)
    print(f"  ✅ Splits: {len(s['train']['volumes'])} train, {len(s['val']['volumes'])} val, {len(s['test']['volumes'])} test")
    checks.append(True)
else:
    print(f"  ❌ Splits file missing")
    checks.append(False)

# 5. MONAI + SwinUNETR available
try:
    from monai.networks.nets import SwinUNETR
    print(f"  ✅ MONAI SwinUNETR importable")
    checks.append(True)
except ImportError as e:
    print(f"  ❌ MONAI SwinUNETR import failed: {e}")
    checks.append(False)

# 6. medpy for advanced metrics
try:
    from medpy.metric.binary import hd95, assd
    print(f"  ✅ medpy metrics available")
    checks.append(True)
except ImportError:
    print(f"  ⚠️  medpy not installed - will auto-install")
    checks.append(True)  # will install in main cell

print("\n" + "=" * 70)
if all(checks):
    print("✅ ALL CHECKS PASSED — Ready to launch SwinUNETR master cell!")
else:
    print("⚠️  Some checks failed — resolve issues before proceeding")
print("=" * 70)

In [ ]:
"""
=========================================================================
ROOT-LEVEL DEEP VERIFICATION
Use root privileges to bypass all permission restrictions
=========================================================================
"""
import os
import subprocess
import time

print("=" * 80)
print("🔍 ROOT-LEVEL VERIFICATION")
print("=" * 80)

GPU_PIDS = [1091441, 3080041, 2910699, 3297636]

for pid in GPU_PIDS:
    print(f"\n{'=' * 80}")
    print(f"📌 PID {pid}")
    print("=" * 80)
    
    # Multiple methods to find this process
    found_via = []
    
    # Method 1: /proc filesystem (root can see all)
    if os.path.exists(f"/proc/{pid}"):
        found_via.append("/proc")
        try:
            with open(f"/proc/{pid}/cmdline", 'rb') as f:
                cmdline = f.read().decode('utf-8', errors='replace').replace('\x00', ' ')
            print(f"   ✅ /proc/{pid}/cmdline: {cmdline[:150]}")
        except Exception as e:
            print(f"   /proc read error: {e}")
        
        try:
            stat_result = subprocess.run(['stat', '-c', 'Owner=%U(%u) Group=%G(%g)', f"/proc/{pid}"],
                                        capture_output=True, text=True)
            print(f"   {stat_result.stdout.strip()}")
        except:
            pass
        
        try:
            with open(f"/proc/{pid}/comm") as f:
                print(f"   comm: {f.read().strip()}")
        except:
            pass
        
        # Check cwd (working directory)
        try:
            cwd = os.readlink(f"/proc/{pid}/cwd")
            print(f"   cwd:  {cwd}")
        except Exception as e:
            print(f"   cwd:  (cannot read: {e})")
    
    # Method 2: nsenter to check different namespaces (root only)
    try:
        r = subprocess.run(['nsenter', '-t', str(pid), '-p', 'echo', 'exists'],
                          capture_output=True, text=True, timeout=5)
        if r.returncode == 0:
            found_via.append("nsenter")
            print(f"   ✅ Process exists (via nsenter)")
    except FileNotFoundError:
        pass
    except Exception:
        pass
    
    # Method 3: kill -0 (test if process is signalable)
    try:
        os.kill(pid, 0)  # Signal 0 = check existence
        found_via.append("kill -0")
        print(f"   ✅ Process signalable (kill -0)")
    except ProcessLookupError:
        print(f"   ❌ Process not signalable (does not exist)")
    except PermissionError:
        print(f"   ⚠️  Process exists but permission denied even for root??")
    
    # Method 4: fuser on GPU device
    try:
        r = subprocess.run(['fuser', '/dev/nvidia0'],
                          capture_output=True, text=True, timeout=5)
        if str(pid) in r.stdout or str(pid) in r.stderr:
            found_via.append("fuser")
            print(f"   ✅ Holds GPU device")
    except FileNotFoundError:
        pass
    except:
        pass
    
    # Summary
    if found_via:
        print(f"\n   🟢 STATUS: EXISTS (found via: {', '.join(found_via)})")
    else:
        print(f"\n   💀 STATUS: GHOST (no process found - GPU memory only)")

# ============================================================
# nvidia-smi comprehensive
# ============================================================
print("\n" + "=" * 80)
print("📊 CURRENT nvidia-smi:")
print("=" * 80)
try:
    r = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
    print(r.stdout)
except Exception as e:
    print(f"Error: {e}")

# ============================================================
# Recommendation matrix
# ============================================================
print("\n" + "=" * 80)
print("🎯 RECOMMENDATION")
print("=" * 80)
print(f"""
Based on above output:

1. If PID 3297636 shows STATUS: EXISTS with cmdline pointing to /workspace/sir/
   or your notebook path → SAFE TO KILL (its yours)

2. If PID 3297636 shows STATUS: GHOST → CANNOT KILL (already dead)
   Only fix: reboot server or wait for driver to release memory

3. If PID 3297636 shows STATUS: EXISTS with cmdline pointing to /home/otheruser/
   → DO NOT KILL (someone else)

Share output. I will give KILL COMMAND only for what is safe.
""")

In [ ]:
"""
=========================================================================
ADVANCED GHOST MEMORY CLEANUP - Root privileges required
=========================================================================
Try aggressive techniques to reclaim ghost GPU memory
=========================================================================
"""
import os
import subprocess
import time

print("=" * 80)
print("🧹 ADVANCED GHOST MEMORY CLEANUP")
print("=" * 80)

# ============================================================
# Method 1: Find ANYONE holding /dev/nvidia*
# ============================================================
print("\n🔍 Method 1: Find processes holding NVIDIA device files")
print("-" * 80)

for dev in ['/dev/nvidia0', '/dev/nvidiactl', '/dev/nvidia-uvm']:
    if os.path.exists(dev):
        try:
            r = subprocess.run(['lsof', dev], capture_output=True, text=True, timeout=10)
            if r.stdout.strip():
                print(f"\n📄 {dev}:")
                for line in r.stdout.split('\n')[:20]:
                    print(f"   {line}")
            else:
                print(f"   {dev}: no processes")
        except FileNotFoundError:
            print(f"   lsof not installed, trying fuser...")
            try:
                r = subprocess.run(['fuser', '-v', dev], capture_output=True, text=True, timeout=10)
                print(f"\n📄 {dev}: {r.stdout} {r.stderr}")
            except:
                pass
        except Exception as e:
            print(f"   {dev} error: {e}")

# ============================================================
# Method 2: NVIDIA Fabric Manager check
# ============================================================
print("\n🔍 Method 2: Check NVIDIA services")
print("-" * 80)

try:
    r = subprocess.run(['systemctl', 'status', 'nvidia-fabricmanager'],
                      capture_output=True, text=True, timeout=10)
    print(f"Fabric manager status:")
    for line in r.stdout.split('\n')[:8]:
        print(f"   {line}")
except:
    pass

# ============================================================
# Method 3: Try suspend/resume of GPU driver
# ============================================================
print("\n🔍 Method 3: Try nvidia-smi drain mode (advanced)")
print("-" * 80)

try:
    # Get GPU bus ID
    r = subprocess.run(['nvidia-smi', '--query-gpu=pci.bus_id', '--format=csv,noheader'],
                      capture_output=True, text=True)
    bus_id = r.stdout.strip().split('\n')[0]
    print(f"GPU Bus ID: {bus_id}")
    
    # Try drain mode
    print(f"\nAttempting drain mode:")
    r = subprocess.run(['nvidia-smi', '-i', '0', '-dm', '1'],
                      capture_output=True, text=True, timeout=15)
    print(f"stdout: {r.stdout}")
    print(f"stderr: {r.stderr}")
    print(f"return: {r.returncode}")
    
    if r.returncode == 0:
        time.sleep(5)
        # Undrain
        r2 = subprocess.run(['nvidia-smi', '-i', '0', '-dm', '0'],
                           capture_output=True, text=True, timeout=15)
        print(f"\nUndrain: {r2.stdout} (code: {r2.returncode})")
except Exception as e:
    print(f"Drain mode error: {e}")

# ============================================================
# Method 4: Check current memory
# ============================================================
print("\n🔍 Method 4: Current memory status")
print("-" * 80)

import torch
torch.cuda.empty_cache()
torch.cuda.ipc_collect()

free, total = torch.cuda.mem_get_info(0)
print(f"   Free: {free/1024**3:.2f} GB / {total/1024**3:.2f} GB")

# ============================================================
# Method 5: Last resort - dmesg for GPU errors
# ============================================================
print("\n🔍 Method 5: Recent GPU errors in kernel log")
print("-" * 80)
try:
    r = subprocess.run(['dmesg', '-T', '--level=err,crit'],
                      capture_output=True, text=True, timeout=10)
    gpu_errors = [l for l in r.stdout.split('\n')[-100:] 
                  if any(k in l.lower() for k in ['nvidia', 'gpu', 'cuda', 'memory'])]
    if gpu_errors:
        print(f"Recent GPU-related errors:")
        for e in gpu_errors[-10:]:
            print(f"   {e[:200]}")
    else:
        print("   No recent GPU errors in dmesg")
except Exception as e:
    print(f"dmesg error: {e}")

print("\n" + "=" * 80)
print("🎯 FINAL RECOMMENDATION")
print("=" * 80)
free_gb = free / 1024**3
if free_gb > 30:
    print(f"✅ Ghost memory cleared! Free: {free_gb:.1f} GB - launch full SwinUNETR!")
elif free_gb > 10:
    print(f"⚠️  Partial recovery: {free_gb:.1f} GB free")
    print(f"   Launch reduced SwinUNETR config")
else:
    print(f"❌ Still stuck: {free_gb:.1f} GB free")
    print(f"   Two options:")
    print(f"   1. Request admin to reboot server (definitive fix)")
    print(f"   2. Launch reduced SwinUNETR (works in 6 GB)")
print("=" * 80)

In [10]:
"""
=========================================================================
LiTS DATASET VERIFICATION - Complete check
=========================================================================
"""
import os
import json

print("=" * 80)
print("🔍 LiTS DATASET VERIFICATION")
print("=" * 80)

paths_to_check = {
    "Project root": "/workspace/liver_tumor_project",
    "Raw data": "/workspace/liver_tumor_project/data/raw/LiTS/LiTS(train_test)",
    "Raw train_ct": "/workspace/liver_tumor_project/data/raw/LiTS/LiTS(train_test)/train_ct",
    "Raw train_mask_fixed": "/workspace/liver_tumor_project/data/raw/LiTS/LiTS(train_test)/train_mask_fixed",
    "V2 preprocessed": "/workspace/liver_tumor_project/data/preprocessed",
    "V2 imagesTr": "/workspace/liver_tumor_project/data/preprocessed/imagesTr",
    "V2 labelsTr": "/workspace/liver_tumor_project/data/preprocessed/labelsTr",
    "V2 imagesTs": "/workspace/liver_tumor_project/data/preprocessed/imagesTs",
    "V2 labelsTs": "/workspace/liver_tumor_project/data/preprocessed/labelsTs",
    "Splits file": "/workspace/liver_tumor_project/data/splits/fold_splits.json",
    "nnUNet raw": "/workspace/liver_tumor_project/models/nnunet_v2/nnUNet_raw",
    "nnUNet preprocessed": "/workspace/liver_tumor_project/models/nnunet_v2/nnUNet_preprocessed",
    "nnUNet results": "/workspace/liver_tumor_project/models/nnunet_v2/nnUNet_results",
    "nnUNet baseline results": "/workspace/liver_tumor_project/results/baselines/nnunet_v2",
    "SwinUNETR results": "/workspace/liver_tumor_project/results/baselines/swin_unetr",
}

print("\n📁 PATH EXISTENCE CHECK:")
print("-" * 80)
all_ok = True
critical_missing = []

for name, path in paths_to_check.items():
    exists = os.path.exists(path)
    marker = "✅" if exists else "❌"
    
    if exists and os.path.isdir(path):
        try:
            n_items = len(os.listdir(path))
            info = f" ({n_items} items)"
        except:
            info = ""
    elif exists and os.path.isfile(path):
        size_mb = os.path.getsize(path) / 1024**2
        info = f" ({size_mb:.2f} MB)"
    else:
        info = ""
    
    print(f"   {marker} {name:<30} {path}{info}")
    
    if not exists:
        all_ok = False
        if "V2" in name or "Splits" in name:
            critical_missing.append(name)

print("\n📊 DETAILED COUNTS:")
print("-" * 80)

v2_dirs = {
    "imagesTr": "/workspace/liver_tumor_project/data/preprocessed/imagesTr",
    "labelsTr": "/workspace/liver_tumor_project/data/preprocessed/labelsTr",
    "imagesTs": "/workspace/liver_tumor_project/data/preprocessed/imagesTs",
    "labelsTs": "/workspace/liver_tumor_project/data/preprocessed/labelsTs",
}

for name, path in v2_dirs.items():
    if os.path.exists(path):
        files = [f for f in os.listdir(path) if f.endswith('.nii.gz')]
        total_size = sum(os.path.getsize(os.path.join(path, f)) for f in files) / 1024**2
        print(f"   {name:<15}: {len(files):3d} .nii.gz files, {total_size:.1f} MB total")
    else:
        print(f"   {name:<15}: ❌ MISSING")

splits_path = "/workspace/liver_tumor_project/data/splits/fold_splits.json"
if os.path.exists(splits_path):
    print(f"\n📋 SPLITS FILE:")
    with open(splits_path) as f:
        splits = json.load(f)
    for split_name in ["train", "val", "test"]:
        n = len(splits[split_name]["volumes"])
        print(f"   {split_name:<10}: {n} volumes")
        print(f"   First 3: {splits[split_name]['volumes'][:3]}")
else:
    print(f"\n❌ SPLITS FILE MISSING!")

nn_json = "/workspace/liver_tumor_project/results/baselines/nnunet_v2/paper_results.json"
if os.path.exists(nn_json):
    print(f"\n📊 nnU-Net baseline results:")
    with open(nn_json) as f:
        nn = json.load(f)
    liver = nn.get('test_stats', {}).get('liver_Dice_mean', 0) * 100
    tumor = nn.get('test_stats', {}).get('tumor_Dice_mean', 0) * 100
    print(f"   Test Liver Dice: {liver:.2f}%")
    print(f"   Test Tumor Dice: {tumor:.2f}%")
else:
    print(f"\n⚠️  nnU-Net JSON not found - may need re-run")

print("\n" + "=" * 80)
if all_ok:
    print("✅ ALL DATA INTACT - Ready to launch SwinUNETR")
elif critical_missing:
    print(f"❌ CRITICAL MISSING: {', '.join(critical_missing)}")
    print("   Need to re-run preprocessing")
else:
    print("⚠️  Some paths missing but data seems intact")
print("=" * 80)

🔍 LiTS DATASET VERIFICATION

📁 PATH EXISTENCE CHECK:
--------------------------------------------------------------------------------
   ✅ Project root                   /workspace/liver_tumor_project (8 items)
   ❌ Raw data                       /workspace/liver_tumor_project/data/raw/LiTS/LiTS(train_test)
   ❌ Raw train_ct                   /workspace/liver_tumor_project/data/raw/LiTS/LiTS(train_test)/train_ct
   ❌ Raw train_mask_fixed           /workspace/liver_tumor_project/data/raw/LiTS/LiTS(train_test)/train_mask_fixed
   ✅ V2 preprocessed                /workspace/liver_tumor_project/data/preprocessed (5 items)
   ✅ V2 imagesTr                    /workspace/liver_tumor_project/data/preprocessed/imagesTr (111 items)
   ✅ V2 labelsTr                    /workspace/liver_tumor_project/data/preprocessed/labelsTr (111 items)
   ✅ V2 imagesTs                    /workspace/liver_tumor_project/data/preprocessed/imagesTs (20 items)
   ✅ V2 labelsTs                    /workspace/liver_tumo

In [11]:
"""
Check if SwinUNETR was already trained yesterday
"""
import os
import json

BASELINE_OUT = "/workspace/liver_tumor_project/results/baselines/swin_unetr"

print("=" * 70)
print("🔍 SwinUNETR PREVIOUS RUN CHECK")
print("=" * 70)

# Check checkpoints
ckpt_best = f"{BASELINE_OUT}/training/checkpoint_best.pth"
ckpt_final = f"{BASELINE_OUT}/training/checkpoint_final.pth"
history_json = f"{BASELINE_OUT}/training/history.json"
paper_json = f"{BASELINE_OUT}/paper_results.json"

files_to_check = {
    "checkpoint_best.pth": ckpt_best,
    "checkpoint_final.pth": ckpt_final,
    "history.json": history_json,
    "paper_results.json": paper_json,
}

for name, path in files_to_check.items():
    if os.path.exists(path):
        size_mb = os.path.getsize(path) / 1024**2
        mtime = os.path.getmtime(path)
        from datetime import datetime
        mtime_str = datetime.fromtimestamp(mtime).strftime('%Y-%m-%d %H:%M:%S')
        print(f"  ✅ {name}: {size_mb:.2f} MB, modified {mtime_str}")
    else:
        print(f"  ❌ {name}: MISSING")

# Check history if exists
if os.path.exists(history_json):
    with open(history_json) as f:
        history = json.load(f)
    epochs_done = len(history.get('epoch', []))
    if epochs_done > 0:
        best_liver = max([d for d in history.get('val_dice_liver', []) if d > 0], default=0)
        best_tumor = max([d for d in history.get('val_dice_tumor', []) if d > 0], default=0)
        best_mean = max([d for d in history.get('val_dice_mean', []) if d > 0], default=0)
        last_loss = history.get('train_loss', [0])[-1]
        print(f"\n📊 Training history:")
        print(f"   Epochs completed: {epochs_done}/500")
        print(f"   Best val liver Dice: {best_liver:.4f}")
        print(f"   Best val tumor Dice: {best_tumor:.4f}")
        print(f"   Best val mean Dice: {best_mean:.4f}")
        print(f"   Last train loss: {last_loss:.4f}")

# Check paper results
if os.path.exists(paper_json):
    with open(paper_json) as f:
        paper = json.load(f)
    print(f"\n📄 Paper results found:")
    print(f"   Model: {paper.get('model', 'unknown')}")
    print(f"   Config used: {paper.get('config_used', 'unknown')}")
    if 'test_stats' in paper:
        liver = paper['test_stats'].get('liver_Dice_mean', 0) * 100
        tumor = paper['test_stats'].get('tumor_Dice_mean', 0) * 100
        print(f"   Test Liver Dice: {liver:.2f}%")
        print(f"   Test Tumor Dice: {tumor:.2f}%")
    print(f"   ⚠️  Complete results already exist!")

# Check log files
log_dir = "/workspace/liver_tumor_project/logs/swin_unetr"
if os.path.exists(log_dir):
    logs = sorted([f for f in os.listdir(log_dir) if f.endswith('.log')])
    if logs:
        print(f"\n📝 Log files ({len(logs)}):")
        for l in logs[-5:]:
            path = f"{log_dir}/{l}"
            size = os.path.getsize(path) / 1024
            print(f"   {l}: {size:.1f} KB")

print("=" * 70)

🔍 SwinUNETR PREVIOUS RUN CHECK
  ✅ checkpoint_best.pth: 67.15 MB, modified 2026-08-13 11:04:39
  ✅ checkpoint_final.pth: 67.15 MB, modified 2026-08-13 10:53:06
  ✅ history.json: 0.06 MB, modified 2026-08-13 10:53:06
  ✅ paper_results.json: 0.03 MB, modified 2026-08-13 10:57:48

📊 Training history:
   Epochs completed: 500/500
   Best val liver Dice: 0.9409
   Best val tumor Dice: 0.6612
   Best val mean Dice: 0.7988
   Last train loss: 0.2769

📄 Paper results found:
   Model: SwinUNETR (reduced)
   Config used: reduced
   Test Liver Dice: 94.67%
   Test Tumor Dice: 47.62%
   ⚠️  Complete results already exist!

📝 Log files (14):
   hybrid_v4_20260813_142121.log: 1.9 KB
   master_20260812_161031.log: 1.3 KB
   master_20260812_161407.log: 5.9 KB
   master_20260812_162703.log: 1.2 KB
   master_20260812_163730.log: 1.4 KB


In [12]:
"""
=========================================================================
SegMamba-V2 SETUP - Step 1 & 2
Environment verification + Repository clone
=========================================================================
"""
import os
import subprocess
import sys

print("=" * 80)
print("🚀 SegMamba-V2 SETUP - PHASE 1")
print("=" * 80)

# ============================================================
# 1. Environment check
# ============================================================
print("\n📦 STEP 1: Environment Verification")
print("-" * 80)

# Python version
print(f"Python version: {sys.version.split()[0]}")

# PyTorch info
try:
    import torch
    print(f"PyTorch version: {torch.__version__}")
    print(f"CUDA available: {torch.cuda.is_available()}")
    print(f"CUDA version (PyTorch): {torch.version.cuda}")
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU count: {torch.cuda.device_count()}")
    free, total = torch.cuda.mem_get_info(0)
    print(f"GPU free memory: {free/1024**3:.1f} / {total/1024**3:.1f} GB")
except Exception as e:
    print(f"❌ PyTorch check failed: {e}")

# System CUDA version
try:
    result = subprocess.run(['nvcc', '--version'], capture_output=True, text=True)
    if result.returncode == 0:
        for line in result.stdout.split('\n'):
            if 'release' in line:
                print(f"System CUDA (nvcc): {line.strip()}")
    else:
        print("nvcc not found - checking nvidia-smi driver version")
        r2 = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
        for line in r2.stdout.split('\n'):
            if 'Driver Version' in line or 'CUDA Version' in line:
                print(f"nvidia-smi: {line.strip()}")
                break
except Exception as e:
    print(f"CUDA check error: {e}")

# Check if mamba_ssm already installed
print("\n🔍 Check existing installations:")
for lib in ['mamba_ssm', 'causal_conv1d', 'monai', 'nibabel', 'medpy']:
    try:
        mod = __import__(lib)
        version = getattr(mod, '__version__', 'unknown')
        print(f"   ✅ {lib}: {version}")
    except ImportError:
        print(f"   ❌ {lib}: NOT installed")

# ============================================================
# 2. Setup project directory
# ============================================================
print("\n" + "=" * 80)
print("📁 STEP 2: Project Directory Setup")
print("-" * 80)

PROJECT_ROOT = "/workspace/liver_tumor_project"
SEGMAMBA_DIR = f"{PROJECT_ROOT}/baselines_code/segmamba_v2"
RESULTS_DIR = f"{PROJECT_ROOT}/results/baselines/segmamba_v2"

# Create directories
os.makedirs(f"{PROJECT_ROOT}/baselines_code", exist_ok=True)
for subdir in ["training", "predictions_val", "predictions_test", "metrics", "visualizations"]:
    os.makedirs(f"{RESULTS_DIR}/{subdir}", exist_ok=True)

print(f"✅ Baselines code dir: {PROJECT_ROOT}/baselines_code/")
print(f"✅ SegMamba-V2 dir:    {SEGMAMBA_DIR}")
print(f"✅ Results dir:        {RESULTS_DIR}")

# ============================================================
# 3. Clone SegMamba-V2 repo
# ============================================================
print("\n" + "=" * 80)
print("📥 STEP 3: Clone SegMamba-V2 Repository")
print("-" * 80)

if os.path.exists(SEGMAMBA_DIR):
    print(f"⚠️  Directory already exists: {SEGMAMBA_DIR}")
    # Check if it has files
    files = os.listdir(SEGMAMBA_DIR)
    if len(files) > 0:
        print(f"   Contains {len(files)} items - skipping clone")
        print(f"   To re-clone: rm -rf {SEGMAMBA_DIR}")
    else:
        print(f"   Empty - proceeding with clone")
        os.rmdir(SEGMAMBA_DIR)

if not os.path.exists(SEGMAMBA_DIR):
    print(f"Cloning from GitHub...")
    result = subprocess.run(
        ['git', 'clone', 'https://github.com/ge-xing/SegMamba-V2.git', SEGMAMBA_DIR],
        capture_output=True, text=True, timeout=120
    )
    if result.returncode == 0:
        print(f"✅ Clone successful!")
    else:
        print(f"❌ Clone failed:")
        print(f"   stdout: {result.stdout}")
        print(f"   stderr: {result.stderr}")

# ============================================================
# 4. Show repository structure
# ============================================================
print("\n" + "=" * 80)
print("📂 REPOSITORY STRUCTURE:")
print("-" * 80)

if os.path.exists(SEGMAMBA_DIR):
    for root, dirs, files in os.walk(SEGMAMBA_DIR):
        # Only go 2 levels deep
        level = root.replace(SEGMAMBA_DIR, '').count(os.sep)
        if level > 2:
            continue
        indent = '  ' * level
        rel_root = os.path.basename(root) if root != SEGMAMBA_DIR else "SegMamba-V2/"
        print(f'{indent}📁 {rel_root}/')
        subindent = '  ' * (level + 1)
        for file in sorted(files)[:15]:
            size = os.path.getsize(os.path.join(root, file)) / 1024
            print(f'{subindent}📄 {file} ({size:.1f} KB)')
        if len(files) > 15:
            print(f'{subindent}... ({len(files) - 15} more files)')

print("\n" + "=" * 80)
print("✅ PHASE 1 COMPLETE")
print("=" * 80)
print(f"""
NEXT STEPS:
1. Share this output - I will confirm CUDA/PyTorch compatibility
2. Then we install mamba_ssm + causal_conv1d (Step 3)
3. Then verify installation + adapt for LiTS (Steps 4-6)
""")

🚀 SegMamba-V2 SETUP - PHASE 1

📦 STEP 1: Environment Verification
--------------------------------------------------------------------------------
Python version: 3.10.12
PyTorch version: 2.4.0+cu121
CUDA available: True
CUDA version (PyTorch): 12.1
GPU: NVIDIA A100-SXM4-80GB
GPU count: 1
GPU free memory: 38.4 / 79.2 GB
System CUDA (nvcc): Cuda compilation tools, release 12.5, V12.5.40

🔍 Check existing installations:
   ❌ mamba_ssm: NOT installed
   ❌ causal_conv1d: NOT installed
   ✅ monai: 1.4.0
   ✅ nibabel: 5.4.2
   ✅ medpy: 0.5.2

📁 STEP 2: Project Directory Setup
--------------------------------------------------------------------------------
✅ Baselines code dir: /workspace/liver_tumor_project/baselines_code/
✅ SegMamba-V2 dir:    /workspace/liver_tumor_project/baselines_code/segmamba_v2
✅ Results dir:        /workspace/liver_tumor_project/results/baselines/segmamba_v2

📥 STEP 3: Clone SegMamba-V2 Repository
----------------------------------------------------------------------

In [13]:
"""
==================================================================================
SegMamba-V2 ALL-IN-ONE MASTER CELL
==================================================================================
Baseline #3 for LiTS paper
Auto-install + Model setup + Training + Evaluation
Expected: 5-7 hours total (25 min install + 5-6 hours training + 15 min eval)
==================================================================================
"""

import os
import sys
import subprocess
import time
from datetime import datetime

pipeline_start = time.time()

print("=" * 80)
print("🚀 SegMamba-V2 MASTER PIPELINE - Baseline #3")
print("=" * 80)

# ==================================================================================
# STAGE 0: AUTO-INSTALL DEPENDENCIES
# ==================================================================================
print("\n📦 STAGE 0: Install SegMamba-V2 dependencies")
print("-" * 80)

def install_pkg(cmd_list, name, timeout=1800):
    print(f"\n🔧 Installing {name}...")
    print(f"   Command: {' '.join(cmd_list)}")
    result = subprocess.run(cmd_list, capture_output=True, text=True, timeout=timeout)
    if result.returncode == 0:
        print(f"   ✅ {name} installed successfully")
        return True
    else:
        print(f"   ❌ {name} install failed")
        print(f"   stderr (last 500 chars): {result.stderr[-500:]}")
        return False

# Check existing installations first
existing = {}
for lib in ['causal_conv1d', 'mamba_ssm', 'monai', 'medpy']:
    try:
        mod = __import__(lib)
        existing[lib] = getattr(mod, '__version__', 'unknown')
        print(f"   ✅ {lib}: {existing[lib]} already installed")
    except ImportError:
        existing[lib] = None
        print(f"   ⏳ {lib}: needs install")

# Install causal-conv1d (mamba dependency)
if existing.get('causal_conv1d') is None:
    success = install_pkg(
        ['pip', 'install', 'causal-conv1d>=1.2.0', '--no-build-isolation'],
        'causal-conv1d', timeout=1200
    )
    if not success:
        print("\n⚠️  Trying alternative install method...")
        install_pkg(
            ['pip', 'install', 'causal-conv1d==1.4.0'],
            'causal-conv1d (v1.4.0)'
        )

# Install mamba_ssm
if existing.get('mamba_ssm') is None:
    success = install_pkg(
        ['pip', 'install', 'mamba-ssm==2.2.2', '--no-build-isolation'],
        'mamba-ssm', timeout=1800
    )
    if not success:
        print("\n❌ mamba_ssm install failed - cannot proceed with SegMamba-V2")
        print("   Alternative: Use MedNeXt-v2 instead (no compilation needed)")
        raise SystemExit("mamba_ssm installation failed")

# Install medpy if needed
if existing.get('medpy') is None:
    install_pkg(['pip', 'install', '-q', 'medpy'], 'medpy')

# Verify installations
print("\n🧪 Verify installations:")
try:
    import mamba_ssm
    from mamba_ssm import Mamba
    print(f"   ✅ mamba_ssm: {mamba_ssm.__version__}")
except ImportError as e:
    print(f"   ❌ mamba_ssm import failed: {e}")
    raise SystemExit("Cannot proceed")

try:
    import causal_conv1d
    print(f"   ✅ causal_conv1d: OK")
except ImportError as e:
    print(f"   ⚠️  causal_conv1d import: {e}")

# ==================================================================================
# STAGE 1: SETUP + CLONE + EXTRACT MODEL CODE
# ==================================================================================
print("\n" + "=" * 80)
print("📥 STAGE 1: Clone SegMamba-V2 + Extract model")
print("-" * 80)

PROJECT_ROOT = "/workspace/liver_tumor_project"
SEGMAMBA_DIR = f"{PROJECT_ROOT}/baselines_code/segmamba_v2"

os.makedirs(f"{PROJECT_ROOT}/baselines_code", exist_ok=True)

# Clone if not exists
if not os.path.exists(SEGMAMBA_DIR):
    print(f"Cloning SegMamba-V2 repository...")
    result = subprocess.run(
        ['git', 'clone', 'https://github.com/ge-xing/SegMamba-V2.git', SEGMAMBA_DIR],
        capture_output=True, text=True, timeout=120
    )
    if result.returncode == 0:
        print(f"✅ Clone successful")
    else:
        print(f"❌ Clone failed: {result.stderr}")
        raise SystemExit("Clone failed")
else:
    print(f"✅ Repo already exists at {SEGMAMBA_DIR}")

# Add to Python path for imports
if SEGMAMBA_DIR not in sys.path:
    sys.path.insert(0, SEGMAMBA_DIR)

# Try to import SegMamba model
print("\n🔧 Importing SegMamba model...")
try:
    from models_segmamba.segmambav2 import SegMamba
    print(f"   ✅ SegMamba imported successfully")
except ImportError as e:
    print(f"   ❌ Import failed: {e}")
    print("   Checking alternative import paths...")
    
    # Try other possible paths
    for possible_path in ['segmamba', 'model_segmamba', 'models.segmamba', 'src.models.segmamba']:
        try:
            exec(f"from {possible_path} import SegMamba")
            print(f"   ✅ Found via {possible_path}")
            break
        except ImportError:
            continue
    else:
        # List what's in the repo
        print(f"\n   Repo contents:")
        for root, dirs, files in os.walk(SEGMAMBA_DIR):
            level = root.replace(SEGMAMBA_DIR, '').count(os.sep)
            if level > 2: continue
            for f in files:
                if 'segmamba' in f.lower() or 'model' in f.lower():
                    print(f"      {os.path.join(root, f)}")
        raise SystemExit("Cannot import SegMamba model")

# ==================================================================================
# STAGE 2: STANDARD IMPORTS + CONFIG
# ==================================================================================
print("\n" + "=" * 80)
print("🏗️  STAGE 2: Setup training pipeline")
print("-" * 80)

os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'max_split_size_mb:512'

import json
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import nibabel as nib
from tqdm import tqdm

import torch
from torch.utils.data import DataLoader
from torch.cuda.amp import autocast, GradScaler

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from monai.losses import DiceCELoss
from monai.metrics import DiceMetric
from monai.inferers import sliding_window_inference
from monai.data import CacheDataset, Dataset, list_data_collate
from monai.transforms import (
    Compose, LoadImaged, EnsureChannelFirstd, SpatialPadd,
    RandCropByPosNegLabeld, RandFlipd, RandRotate90d,
    RandShiftIntensityd, RandScaleIntensityd,
    RandGaussianNoised, RandAdjustContrastd,
    EnsureTyped, AsDiscrete,
)
from monai.utils import set_determinism
from medpy.metric.binary import hd95, assd

torch.cuda.empty_cache()

# ==================================================================================
# CONFIGS
# ==================================================================================
class Config:
    MODEL_NAME = "SegMamba_V2"
    BASELINE_KEY = "segmamba_v2"
    V2_ROOT = "/workspace/liver_tumor_project/data/preprocessed"
    SPLIT_FILE = "/workspace/liver_tumor_project/data/splits/fold_splits.json"
    RESULTS = "/workspace/liver_tumor_project/results"
    LOG_DIR = "/workspace/liver_tumor_project/logs/segmamba_v2"
    
    EPOCHS = 500
    PATCH_SIZE = (128, 128, 128)  # SegMamba-V2 official config
    BATCH_SIZE = 2
    GRAD_ACCUM = 1
    NUM_SAMPLES = 4
    LEARNING_RATE = 1e-4
    WEIGHT_DECAY = 1e-5
    POS_NEG_RATIO = 2.0
    
    IN_CHANNELS = 1  # CT single channel
    OUT_CHANNELS = 3  # bg + liver + tumor
    
    NUM_WORKERS = 4
    CACHE_RATE = 1.0
    SW_BATCH = 2
    SW_OVERLAP = 0.5
    SEED = 42
    VAL_INTERVAL = 5

cfg = Config()

set_determinism(seed=cfg.SEED)
torch.backends.cudnn.benchmark = True

os.makedirs(cfg.LOG_DIR, exist_ok=True)
LOG_FILE = f"{cfg.LOG_DIR}/master_{datetime.now().strftime('%Y%m%d_%H%M%S')}.log"

def log(msg, echo=True):
    ts = datetime.now().strftime('%H:%M:%S')
    line = f"[{ts}] {msg}"
    if echo: print(line)
    with open(LOG_FILE, 'a') as f: f.write(line + "\n")

def gpu_mem():
    free, total = torch.cuda.mem_get_info(0)
    alloc = torch.cuda.memory_allocated(0)
    return f"free={free/1024**3:.1f}GB alloc={alloc/1024**3:.1f}GB"

BASELINE_OUT = f"{cfg.RESULTS}/baselines/{cfg.BASELINE_KEY}"
for subdir in ["training", "predictions_val", "predictions_test", "metrics", "visualizations"]:
    os.makedirs(f"{BASELINE_OUT}/{subdir}", exist_ok=True)

log(f"\n🎮 Memory: {gpu_mem()}")

# ==================================================================================
# STAGE 3: DATA LOADERS (same as SwinUNETR - proven working)
# ==================================================================================
log("\n" + "=" * 80)
log("📦 STAGE 3: Data loaders")
log("=" * 80)

with open(cfg.SPLIT_FILE, 'r') as f:
    splits = json.load(f)

def build_dicts(volume_names, subset):
    dicts = []
    for vol_name in volume_names:
        idx = vol_name.replace("volume-", "").replace(".nii", "")
        if subset == "test":
            img = f"{cfg.V2_ROOT}/imagesTs/volume-{idx}.nii.gz"
            lbl = f"{cfg.V2_ROOT}/labelsTs/segmentation-{idx}.nii.gz"
        else:
            img = f"{cfg.V2_ROOT}/imagesTr/volume-{idx}.nii.gz"
            lbl = f"{cfg.V2_ROOT}/labelsTr/segmentation-{idx}.nii.gz"
        if os.path.exists(img) and os.path.exists(lbl):
            dicts.append({"image": img, "label": lbl, "name": f"volume-{idx}"})
    return dicts

train_dicts = build_dicts(splits["train"]["volumes"], "train")
val_dicts = build_dicts(splits["val"]["volumes"], "train")
test_dicts = build_dicts(splits["test"]["volumes"], "test")

log(f"Train: {len(train_dicts)}, Val: {len(val_dicts)}, Test: {len(test_dicts)}")

train_transforms = Compose([
    LoadImaged(keys=["image", "label"]),
    EnsureChannelFirstd(keys=["image", "label"]),
    SpatialPadd(keys=["image", "label"], spatial_size=cfg.PATCH_SIZE, mode=("constant", "constant")),
    RandCropByPosNegLabeld(
        keys=["image", "label"], label_key="label",
        spatial_size=cfg.PATCH_SIZE,
        pos=cfg.POS_NEG_RATIO, neg=1.0,
        num_samples=cfg.NUM_SAMPLES,
        image_key="image", image_threshold=0,
        allow_smaller=True,
    ),
    RandFlipd(keys=["image", "label"], spatial_axis=[0], prob=0.5),
    RandFlipd(keys=["image", "label"], spatial_axis=[1], prob=0.5),
    RandFlipd(keys=["image", "label"], spatial_axis=[2], prob=0.5),
    RandRotate90d(keys=["image", "label"], prob=0.5, max_k=3),
    RandShiftIntensityd(keys=["image"], offsets=0.10, prob=0.5),
    RandScaleIntensityd(keys=["image"], factors=0.10, prob=0.5),
    RandAdjustContrastd(keys=["image"], gamma=(0.7, 1.5), prob=0.3),
    RandGaussianNoised(keys=["image"], mean=0.0, std=0.01, prob=0.15),
    EnsureTyped(keys=["image", "label"]),
])

val_transforms = Compose([
    LoadImaged(keys=["image", "label"]),
    EnsureChannelFirstd(keys=["image", "label"]),
    SpatialPadd(keys=["image", "label"], spatial_size=cfg.PATCH_SIZE, mode=("constant", "constant")),
    EnsureTyped(keys=["image", "label"]),
])

train_ds = CacheDataset(data=train_dicts, transform=train_transforms,
                       cache_rate=cfg.CACHE_RATE, num_workers=cfg.NUM_WORKERS)
val_ds = CacheDataset(data=val_dicts, transform=val_transforms,
                     cache_rate=cfg.CACHE_RATE, num_workers=cfg.NUM_WORKERS)
test_ds = Dataset(data=test_dicts, transform=val_transforms)

train_loader = DataLoader(train_ds, batch_size=cfg.BATCH_SIZE, shuffle=True,
                          num_workers=cfg.NUM_WORKERS, pin_memory=True,
                          collate_fn=list_data_collate)
val_loader = DataLoader(val_ds, batch_size=1, shuffle=False,
                        num_workers=cfg.NUM_WORKERS, pin_memory=True)
test_loader = DataLoader(test_ds, batch_size=1, shuffle=False,
                         num_workers=cfg.NUM_WORKERS, pin_memory=True)

log("✅ Data loaders ready")

# ==================================================================================
# STAGE 4: MODEL
# ==================================================================================
log("\n" + "=" * 80)
log("🏗️  STAGE 4: SegMamba-V2 model")
log("=" * 80)

device = torch.device("cuda")

# SegMamba-V2 uses in_channels=4, out_channels=4 in original (BraTS 4-modal)
# We adapt to in_channels=1 (CT), out_channels=3 (bg+liver+tumor)
try:
    model = SegMamba(in_channels=cfg.IN_CHANNELS, out_channels=cfg.OUT_CHANNELS).to(device)
    log(f"✅ Model created with in={cfg.IN_CHANNELS}, out={cfg.OUT_CHANNELS}")
except Exception as e:
    log(f"❌ Model creation failed: {e}")
    log(f"   Trying default constructor...")
    try:
        model = SegMamba().to(device)
        # Modify first/last layers for LiTS
        log(f"   Default model created, may need layer adjustments")
    except Exception as e2:
        log(f"   ❌ Default constructor also failed: {e2}")
        raise

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
log(f"Parameters: {n_params/1e6:.2f}M")
log(f"GPU after model: {gpu_mem()}")

loss_fn = DiceCELoss(to_onehot_y=True, softmax=True, squared_pred=True, 
                    smooth_nr=0, smooth_dr=1e-6)
optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.LEARNING_RATE, 
                              weight_decay=cfg.WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=cfg.EPOCHS)
scaler = GradScaler()

dice_metric = DiceMetric(include_background=False, reduction="mean_batch", get_not_nans=False)
post_pred = AsDiscrete(argmax=True, to_onehot=cfg.OUT_CHANNELS)
post_label = AsDiscrete(to_onehot=cfg.OUT_CHANNELS)

# ==================================================================================
# STAGE 5: TRAINING
# ==================================================================================
log("\n" + "=" * 80)
log(f"🚀 STAGE 5: Training {cfg.EPOCHS} epochs")
log("=" * 80)

train_start = time.time()
best_val_dice = 0.0
history = {'epoch': [], 'train_loss': [], 'val_dice_liver': [], 
           'val_dice_tumor': [], 'val_dice_mean': [], 'lr': [], 'time': []}

try:
    for epoch in range(cfg.EPOCHS):
        epoch_start = time.time()
        model.train()
        epoch_loss = 0
        step = 0
        optimizer.zero_grad()
        
        for batch_idx, batch in enumerate(train_loader):
            step += 1
            images = batch["image"].to(device)
            labels = batch["label"].to(device)
            
            with autocast():
                outputs = model(images)
                loss = loss_fn(outputs, labels) / cfg.GRAD_ACCUM
            
            scaler.scale(loss).backward()
            
            if (batch_idx + 1) % cfg.GRAD_ACCUM == 0:
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad()
            
            epoch_loss += loss.item() * cfg.GRAD_ACCUM
        
        if step % cfg.GRAD_ACCUM != 0:
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
        
        epoch_loss /= step
        scheduler.step()
        epoch_time = time.time() - epoch_start
        
        if epoch == 0:
            log(f"✅ First epoch passed! {gpu_mem()}")
        
        val_dice_liver, val_dice_tumor, val_dice_mean = 0.0, 0.0, 0.0
        
        if (epoch + 1) % cfg.VAL_INTERVAL == 0 or epoch == cfg.EPOCHS - 1:
            model.eval()
            dice_metric.reset()
            with torch.no_grad():
                for val_batch in val_loader:
                    val_images = val_batch["image"].to(device)
                    val_labels = val_batch["label"].to(device)
                    with autocast():
                        val_outputs = sliding_window_inference(
                            val_images, cfg.PATCH_SIZE, cfg.SW_BATCH,
                            model, overlap=cfg.SW_OVERLAP,
                        )
                    val_outputs_list = [post_pred(x) for x in val_outputs]
                    val_labels_list = [post_label(x) for x in val_labels]
                    dice_metric(y_pred=val_outputs_list, y=val_labels_list)
            
            dice_scores = dice_metric.aggregate()
            val_dice_liver = dice_scores[0].item()
            val_dice_tumor = dice_scores[1].item()
            val_dice_mean = (val_dice_liver + val_dice_tumor) / 2
            
            if val_dice_mean > best_val_dice:
                best_val_dice = val_dice_mean
                torch.save({
                    'epoch': epoch,
                    'model_state_dict': model.state_dict(),
                    'val_dice_mean': val_dice_mean,
                    'val_dice_liver': val_dice_liver,
                    'val_dice_tumor': val_dice_tumor,
                }, f"{BASELINE_OUT}/training/checkpoint_best.pth")
        
        history['epoch'].append(epoch + 1)
        history['train_loss'].append(epoch_loss)
        history['val_dice_liver'].append(val_dice_liver)
        history['val_dice_tumor'].append(val_dice_tumor)
        history['val_dice_mean'].append(val_dice_mean)
        history['lr'].append(scheduler.get_last_lr()[0])
        history['time'].append(epoch_time)
        
        mem_str = f" | {gpu_mem()}" if (epoch + 1) % 20 == 0 else ""
        if val_dice_mean > 0:
            log(f"Epoch {epoch+1}/{cfg.EPOCHS}: loss={epoch_loss:.4f} | "
                f"liver={val_dice_liver:.4f} tumor={val_dice_tumor:.4f} | "
                f"time={epoch_time:.1f}s{mem_str}")
        else:
            log(f"Epoch {epoch+1}/{cfg.EPOCHS}: loss={epoch_loss:.4f} | time={epoch_time:.1f}s{mem_str}")
    
    torch.save({
        'epoch': cfg.EPOCHS - 1,
        'model_state_dict': model.state_dict(),
    }, f"{BASELINE_OUT}/training/checkpoint_final.pth")
    
    with open(f"{BASELINE_OUT}/training/history.json", 'w') as f:
        json.dump(history, f, indent=2)
    
    train_time = time.time() - train_start
    log(f"\n✅ Training done in {train_time/3600:.2f}h, best Dice: {best_val_dice:.4f}")

except Exception as e:
    log(f"\n❌ Training failed: {str(e)[:300]}")
    raise

# ==================================================================================
# STAGE 6: PREDICTIONS + METRICS
# ==================================================================================
log("\n🎯 STAGE 6: Predictions")

ckpt = torch.load(f"{BASELINE_OUT}/training/checkpoint_best.pth")
model.load_state_dict(ckpt['model_state_dict'])
model.eval()

def predict_and_save(loader, out_dir, dataset_name):
    os.makedirs(out_dir, exist_ok=True)
    with torch.no_grad():
        for batch in tqdm(loader, desc=f"Predicting {dataset_name}"):
            images = batch["image"].to(device)
            name = batch["name"][0]
            with autocast():
                outputs = sliding_window_inference(
                    images, cfg.PATCH_SIZE, cfg.SW_BATCH,
                    model, overlap=cfg.SW_OVERLAP,
                )
            pred = torch.argmax(outputs, dim=1)[0].cpu().numpy().astype(np.uint8)
            affine = np.diag([1.0, 1.0, 1.5, 1.0])
            idx = name.replace('volume-', '')
            nib.save(nib.Nifti1Image(pred, affine), f"{out_dir}/volume-{idx}.nii.gz")

predict_and_save(val_loader, f"{BASELINE_OUT}/predictions_val", "val")
predict_and_save(test_loader, f"{BASELINE_OUT}/predictions_test", "test")
log("✅ Predictions saved")

def compute_all_metrics(pred, gt, spacing, cls):
    pred_bin = (pred == cls); gt_bin = (gt == cls)
    result = {'class': cls, 'gt_voxels': int(np.sum(gt_bin)), 'pred_voxels': int(np.sum(pred_bin))}
    if np.sum(gt_bin) == 0 and np.sum(pred_bin) == 0:
        result.update({'Dice': None, 'IoU': None, 'HD95': None, 'ASSD': None, 'VOE': None, 'RVD': None}); return result
    if np.sum(gt_bin) == 0:
        result.update({'Dice': 0.0, 'IoU': 0.0, 'HD95': np.inf, 'ASSD': np.inf, 'VOE': 1.0, 'RVD': np.inf}); return result
    if np.sum(pred_bin) == 0:
        result.update({'Dice': 0.0, 'IoU': 0.0, 'HD95': np.inf, 'ASSD': np.inf, 'VOE': 1.0, 'RVD': -1.0}); return result
    intersection = np.sum(pred_bin & gt_bin); union = np.sum(pred_bin | gt_bin)
    dice = 2 * intersection / (np.sum(pred_bin) + np.sum(gt_bin))
    iou = intersection / union; voe = 1 - iou
    rvd = (np.sum(pred_bin) - np.sum(gt_bin)) / np.sum(gt_bin)
    try: hd95_v = hd95(pred_bin, gt_bin, voxelspacing=spacing)
    except: hd95_v = np.inf
    try: assd_v = assd(pred_bin, gt_bin, voxelspacing=spacing)
    except: assd_v = np.inf
    result.update({'Dice': float(dice), 'IoU': float(iou), 'HD95': float(hd95_v),
                   'ASSD': float(assd_v), 'VOE': float(voe), 'RVD': float(rvd)})
    return result

def evaluate_folder(pred_dir, gt_dir):
    results = []
    pred_files = sorted([f for f in os.listdir(pred_dir) if f.endswith('.nii.gz')])
    for pred_file in tqdm(pred_files, desc=f"Eval"):
        idx = pred_file.replace('.nii.gz', '').replace('volume-', '')
        gt_candidates = [f"{gt_dir}/segmentation-{idx}.nii.gz", f"{gt_dir}/volume-{idx}.nii.gz"]
        gt_path = next((p for p in gt_candidates if os.path.exists(p)), None)
        if gt_path is None: continue
        pred_nii = nib.load(f"{pred_dir}/{pred_file}"); gt_nii = nib.load(gt_path)
        pred = pred_nii.get_fdata().astype(np.uint8); gt = gt_nii.get_fdata().astype(np.uint8)
        if pred.shape != gt.shape: continue
        spacing = tuple(pred_nii.header.get_zooms())
        case_result = {'volume': f"volume-{idx}", 'shape': str(pred.shape)}
        for cls in [1, 2]:
            m = compute_all_metrics(pred, gt, spacing, cls)
            class_name = {1: 'liver', 2: 'tumor'}[cls]
            for k, v in m.items():
                if k != 'class': case_result[f"{class_name}_{k}"] = v
        results.append(case_result)
    return results

val_results = evaluate_folder(f"{BASELINE_OUT}/predictions_val", f"{cfg.V2_ROOT}/labelsTr")
test_results = evaluate_folder(f"{BASELINE_OUT}/predictions_test", f"{cfg.V2_ROOT}/labelsTs")

def compute_stats(results, name):
    stats = {'dataset': name, 'n_total': len(results)}
    for organ in ['liver', 'tumor']:
        valid = [r for r in results if r.get(f'{organ}_gt_voxels', 0) > 0]
        for metric in ['Dice', 'IoU', 'HD95', 'ASSD', 'VOE', 'RVD']:
            values = [r[f'{organ}_{metric}'] for r in valid
                      if r.get(f'{organ}_{metric}') is not None 
                      and not np.isnan(r[f'{organ}_{metric}']) 
                      and not np.isinf(r[f'{organ}_{metric}'])]
            key = f'{organ}_{metric}'
            if values:
                stats[f'{key}_n'] = len(values)
                stats[f'{key}_mean'] = float(np.mean(values))
                stats[f'{key}_std'] = float(np.std(values))
                stats[f'{key}_median'] = float(np.median(values))
                stats[f'{key}_min'] = float(np.min(values))
                stats[f'{key}_max'] = float(np.max(values))
    return stats

val_stats = compute_stats(val_results, 'validation')
test_stats = compute_stats(test_results, 'test')

size_groups = {'small': (0, 3000), 'medium': (3000, 30000), 'large': (30000, np.inf)}
size_stats = {}
for gname, (low, high) in size_groups.items():
    cases = [r for r in test_results 
             if low <= r.get('tumor_gt_voxels', 0) < high 
             and r.get('tumor_gt_voxels', 0) > 0]
    dices = [r['tumor_Dice'] for r in cases 
             if r.get('tumor_Dice') is not None and not np.isnan(r['tumor_Dice'])]
    if dices:
        size_stats[gname] = {'n': len(dices), 'mean_dice': float(np.mean(dices)),
                             'std_dice': float(np.std(dices))}

VIZ_DIR = f"{BASELINE_OUT}/visualizations"

def save_slice_viz(volume_id, pred_dir, gt_dir, ct_dir, out_path, suffix=""):
    try:
        idx = volume_id.replace('volume-', '')
        pred = nib.load(f"{pred_dir}/volume-{idx}.nii.gz").get_fdata().astype(np.uint8)
        gt_candidates = [f"{gt_dir}/segmentation-{idx}.nii.gz", f"{gt_dir}/volume-{idx}.nii.gz"]
        gt_path = next((p for p in gt_candidates if os.path.exists(p)), None)
        if gt_path is None: return
        gt = nib.load(gt_path).get_fdata().astype(np.uint8)
        ct = nib.load(f"{ct_dir}/volume-{idx}.nii.gz").get_fdata()
        tumor_slices = np.where(np.any(gt == 2, axis=(0, 1)))[0]
        if len(tumor_slices) > 0: z = tumor_slices[len(tumor_slices)//2]
        else:
            liver_slices = np.where(np.any(gt == 1, axis=(0, 1)))[0]
            z = liver_slices[len(liver_slices)//2] if len(liver_slices) > 0 else ct.shape[2]//2
        fig, axes = plt.subplots(1, 4, figsize=(20, 5))
        axes[0].imshow(ct[:,:,z], cmap='gray'); axes[0].set_title('CT'); axes[0].axis('off')
        axes[1].imshow(gt[:,:,z], cmap='viridis', vmin=0, vmax=2); axes[1].set_title('GT'); axes[1].axis('off')
        axes[2].imshow(pred[:,:,z], cmap='viridis', vmin=0, vmax=2); axes[2].set_title('Pred'); axes[2].axis('off')
        diff = np.zeros_like(gt[:,:,z])
        diff[(gt[:,:,z] > 0) & (pred[:,:,z] == 0)] = 1
        diff[(gt[:,:,z] == 0) & (pred[:,:,z] > 0)] = 2
        axes[3].imshow(ct[:,:,z], cmap='gray', alpha=0.6)
        axes[3].imshow(np.ma.masked_where(diff==0, diff), cmap='autumn', alpha=0.6, vmin=1, vmax=2)
        axes[3].set_title('Errors'); axes[3].axis('off')
        plt.suptitle(f'{volume_id} {suffix}', fontsize=12)
        plt.tight_layout()
        plt.savefig(out_path, dpi=80, bbox_inches='tight')
        plt.close()
    except Exception as e:
        log(f"   Viz failed {volume_id}: {e}")

test_with_tumor = [r for r in test_results if r.get('tumor_gt_voxels', 0) > 0 
                   and r.get('tumor_Dice') is not None]
if test_with_tumor:
    sorted_by = sorted(test_with_tumor, key=lambda x: x['tumor_Dice'])
    for i, r in enumerate(sorted_by[:3]):
        save_slice_viz(r['volume'], f"{BASELINE_OUT}/predictions_test",
                      f"{cfg.V2_ROOT}/labelsTs", f"{cfg.V2_ROOT}/imagesTs",
                      f"{VIZ_DIR}/test_worst_{i+1}_{r['volume']}.png",
                      f"(WORST #{i+1}, Dice={r['tumor_Dice']:.3f})")
    for i, r in enumerate(sorted_by[-3:]):
        save_slice_viz(r['volume'], f"{BASELINE_OUT}/predictions_test",
                      f"{cfg.V2_ROOT}/labelsTs", f"{cfg.V2_ROOT}/imagesTs",
                      f"{VIZ_DIR}/test_best_{i+1}_{r['volume']}.png",
                      f"(BEST #{i+1}, Dice={r['tumor_Dice']:.3f})")

fig, ax = plt.subplots(figsize=(8, 6))
for r in test_with_tumor:
    ax.scatter(r['tumor_gt_voxels'], r['tumor_Dice'], alpha=0.6, s=50)
ax.set_xscale('log'); ax.set_xlabel('Tumor size'); ax.set_ylabel('Tumor Dice')
ax.set_title(f'{cfg.MODEL_NAME}: Tumor Size vs Dice'); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.savefig(f"{VIZ_DIR}/tumor_size_vs_dice.png", dpi=100, bbox_inches='tight')
plt.close()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(history['epoch'], history['train_loss'], label='Train Loss', color='blue')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss'); axes[0].legend(); axes[0].grid(True, alpha=0.3)
val_epochs = [e for e, d in zip(history['epoch'], history['val_dice_mean']) if d > 0]
val_liver = [d for d in history['val_dice_liver'] if d > 0]
val_tumor = [d for d in history['val_dice_tumor'] if d > 0]
axes[1].plot(val_epochs, val_liver, label='Liver', color='green')
axes[1].plot(val_epochs, val_tumor, label='Tumor', color='red')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Dice'); axes[1].legend(); axes[1].grid(True, alpha=0.3)
plt.tight_layout(); plt.savefig(f"{VIZ_DIR}/training_curves.png", dpi=100, bbox_inches='tight')
plt.close()

pd.DataFrame(val_results).to_csv(f"{BASELINE_OUT}/metrics/per_volume_val.csv", index=False)
pd.DataFrame(test_results).to_csv(f"{BASELINE_OUT}/metrics/per_volume_test.csv", index=False)

def latex_row(stats, model_name):
    l_d = stats.get('liver_Dice_mean', 0) * 100
    l_ds = stats.get('liver_Dice_std', 0) * 100
    t_d = stats.get('tumor_Dice_mean', 0) * 100
    t_ds = stats.get('tumor_Dice_std', 0) * 100
    mean_d = (l_d + t_d) / 2
    l_hd = stats.get('liver_HD95_mean', 0)
    t_hd = stats.get('tumor_HD95_mean', 0)
    l_assd = stats.get('liver_ASSD_mean', 0)
    t_assd = stats.get('tumor_ASSD_mean', 0)
    return (f"{model_name} & {l_d:.2f}$\\pm${l_ds:.2f} & {t_d:.2f}$\\pm${t_ds:.2f} & "
            f"{mean_d:.2f} & {l_hd:.2f}/{t_hd:.2f} & {l_assd:.2f}/{t_assd:.2f} \\\\")

latex_test = latex_row(test_stats, cfg.MODEL_NAME)
latex_val = latex_row(val_stats, cfg.MODEL_NAME + " (val)")

with open(f"{BASELINE_OUT}/metrics/paper_table_row.tex", 'w') as f:
    f.write(f"% {cfg.MODEL_NAME} — Test\n{latex_test}\n\n% Val\n{latex_val}\n")

paper_results = {
    'model': cfg.MODEL_NAME,
    'paper_reference': 'SegMamba-V2 (IEEE TMI Jan 2026)',
    'config': {
        'epochs': cfg.EPOCHS,
        'batch_size': cfg.BATCH_SIZE,
        'patch_size': list(cfg.PATCH_SIZE),
        'learning_rate': cfg.LEARNING_RATE,
        'parameters_M': n_params / 1e6,
        'seed': cfg.SEED,
    },
    'training': {'time_hours': train_time / 3600, 'best_val_dice': best_val_dice},
    'validation_stats': val_stats, 'test_stats': test_stats,
    'tumor_size_breakdown': size_stats,
    'per_volume_val': val_results, 'per_volume_test': test_results,
    'latex_row_test': latex_test,
    'timestamp': datetime.now().isoformat(),
}

with open(f"{BASELINE_OUT}/paper_results.json", 'w') as f:
    json.dump(paper_results, f, indent=2, default=str)

log("✅ All exports done")

total_time = time.time() - pipeline_start
log("\n" + "=" * 80)
log(f"🎉 {cfg.MODEL_NAME} COMPLETE in {total_time/3600:.2f}h")
log("=" * 80)

log(f"\n📊 FINAL RESULTS")
log(f"\n{'Metric':<15} {'Val (Liver)':<15} {'Val (Tumor)':<15} {'Test (Liver)':<15} {'Test (Tumor)':<15}")
log("-" * 80)
for metric in ['Dice', 'IoU', 'HD95', 'ASSD', 'VOE', 'RVD']:
    vl = val_stats.get(f'liver_{metric}_mean', 0)
    vt = val_stats.get(f'tumor_{metric}_mean', 0)
    tl = test_stats.get(f'liver_{metric}_mean', 0)
    tt = test_stats.get(f'tumor_{metric}_mean', 0)
    log(f"{metric:<15} {vl:<15.4f} {vt:<15.4f} {tl:<15.4f} {tt:<15.4f}")

print(f"\n✅ SegMamba-V2 DONE!")

🚀 SegMamba-V2 MASTER PIPELINE - Baseline #3

📦 STAGE 0: Install SegMamba-V2 dependencies
--------------------------------------------------------------------------------
   ⏳ causal_conv1d: needs install
   ⏳ mamba_ssm: needs install
   ✅ monai: 1.4.0 already installed
   ✅ medpy: 0.5.2 already installed

🔧 Installing causal-conv1d...
   Command: pip install causal-conv1d>=1.2.0 --no-build-isolation
   ✅ causal-conv1d installed successfully

🔧 Installing mamba-ssm...
   Command: pip install mamba-ssm==2.2.2 --no-build-isolation
   ✅ mamba-ssm installed successfully

🧪 Verify installations:
   ✅ mamba_ssm: 2.2.2
   ✅ causal_conv1d: OK

📥 STAGE 1: Clone SegMamba-V2 + Extract model
--------------------------------------------------------------------------------
✅ Repo already exists at /workspace/liver_tumor_project/baselines_code/segmamba_v2

🔧 Importing SegMamba model...
   ❌ Import failed: No module named 'models_segmamba'
   Checking alternative import paths...

   Repo contents:
    

AttributeError: 'tuple' object has no attribute 'tb_frame'

In [15]:
"""
Discover SegMamba-V2 actual API
"""
import sys
import inspect

SEGMAMBA_REPO = "/workspace/liver_tumor_project/baselines_code/segmamba_v2"

# Add paths
for p in [f"{SEGMAMBA_REPO}/colorectal", f"{SEGMAMBA_REPO}/brats23", SEGMAMBA_REPO]:
    if p not in sys.path:
        sys.path.insert(0, p)

from models_segmamba.segmambav2 import SegMamba

print("=" * 80)
print("🔍 SegMamba-V2 ACTUAL API")
print("=" * 80)

# Full signature
sig = inspect.signature(SegMamba.__init__)
print(f"\n📋 Constructor signature:")
print(f"   SegMamba{sig}")

print(f"\n📋 Parameters:")
for param_name, param in sig.parameters.items():
    if param_name != 'self':
        default = param.default if param.default != inspect.Parameter.empty else '<REQUIRED>'
        print(f"   {param_name}: {default}")

# Show source code (first 60 lines) - to understand structure
print(f"\n📄 Source code (first 80 lines):")
print("-" * 80)
source_file = inspect.getfile(SegMamba)
print(f"File: {source_file}\n")

with open(source_file, 'r') as f:
    lines = f.readlines()

for i, line in enumerate(lines[:80], 1):
    print(f"{i:4d}: {line}", end='')

# Try default constructor
print(f"\n\n🧪 Testing default constructor SegMamba()...")
try:
    import torch
    model = SegMamba().cuda()
    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"   ✅ Default works! {n_params/1e6:.2f}M parameters")
    
    # Try forward pass with default
    # First, figure out expected input shape
    print(f"\n   Trying forward pass with (1, 4, 128, 128, 128)...")
    try:
        with torch.no_grad():
            x = torch.randn(1, 4, 128, 128, 128).cuda()
            y = model(x)
            print(f"   ✅ Forward pass OK: {tuple(x.shape)} -> {tuple(y.shape)}")
    except Exception as e:
        print(f"   Trying (1, 1, 128, 128, 128)...")
        try:
            del x
            torch.cuda.empty_cache()
            x = torch.randn(1, 1, 128, 128, 128).cuda()
            y = model(x)
            print(f"   ✅ Works with 1 channel: {tuple(x.shape)} -> {tuple(y.shape)}")
        except Exception as e2:
            print(f"   Both failed. Error: {e2}")
    
    del model
    torch.cuda.empty_cache()
except Exception as e:
    print(f"   ❌ Default failed: {e}")

🔍 SegMamba-V2 ACTUAL API

📋 Constructor signature:
   SegMamba(self, in_chans=1, out_chans=13, depths=[2, 2, 2, 2], feat_size=[48, 96, 192, 384], drop_path_rate=0, layer_scale_init_value=1e-06, hidden_size: 'int' = 768, norm_name='instance', conv_block: 'bool' = True, res_block: 'bool' = True, spatial_dims=3) -> 'None'

📋 Parameters:
   in_chans: 1
   out_chans: 13
   depths: [2, 2, 2, 2]
   feat_size: [48, 96, 192, 384]
   drop_path_rate: 0
   layer_scale_init_value: 1e-06
   hidden_size: 768
   norm_name: instance
   conv_block: True
   res_block: True
   spatial_dims: 3

📄 Source code (first 80 lines):
--------------------------------------------------------------------------------
File: /workspace/liver_tumor_project/baselines_code/segmamba_v2/brats23/models_segmamba/segmambav2.py

   1: # Copyright (c) MONAI Consortium
   2: # Licensed under the Apache License, Version 2.0 (the "License");
   3: # you may not use this file except in compliance with the License.
   4: # You may obt

In [19]:
"""
=========================================================================
CLEAN DEBUG - Wheel install + verbose error capture
=========================================================================
"""
import subprocess
import sys
import os
import traceback

print("=" * 80)
print("🔍 CLEAN DEBUG - Step by step")
print("=" * 80)

# Check what got installed
print("\n📊 Currently installed:")
for pkg in ['mamba-ssm', 'causal-conv1d']:
    r = subprocess.run(['pip', 'show', pkg], capture_output=True, text=True)
    if r.returncode == 0:
        for line in r.stdout.split('\n'):
            if 'Name:' in line or 'Version:' in line or 'Location:' in line:
                print(f"   {line.strip()}")
        print()
    else:
        print(f"   {pkg}: NOT installed\n")

# ============================================================
# Try latest wheels from correct URLs
# ============================================================
import torch
py_ver = f"cp{sys.version_info.major}{sys.version_info.minor}"
print(f"\n📋 Env: Python {py_ver}, PyTorch {torch.__version__}, CUDA {torch.version.cuda}")

# First check if latest already works (from Combo A that installed)
print("\n" + "=" * 80)
print("🧪 STEP 1: Test current install (whatever version)")
print("=" * 80)

# Clean import cache
for mod in list(sys.modules.keys()):
    if any(k in mod.lower() for k in ['mamba', 'causal_conv', 'segmamba', 'triton']):
        try:
            del sys.modules[mod]
        except:
            pass

# Test imports
try:
    import causal_conv1d
    print(f"   ✅ causal_conv1d imports")
except Exception as e:
    print(f"   ❌ causal_conv1d: {type(e).__name__}: {str(e)[:200]}")

try:
    import mamba_ssm
    print(f"   ✅ mamba_ssm imports, version: {mamba_ssm.__version__}")
except Exception as e:
    print(f"   ❌ mamba_ssm: {type(e).__name__}: {str(e)[:200]}")

# Try basic Mamba block with detailed error capture
print("\n🧪 Test basic Mamba block:")
try:
    from mamba_ssm import Mamba
    m = Mamba(d_model=64).cuda()
    x = torch.randn(1, 32, 64).cuda()
    with torch.no_grad():
        y = m(x)
    print(f"   ✅ Basic Mamba works: {tuple(x.shape)} -> {tuple(y.shape)}")
    mamba_works = True
    del m, x, y
    torch.cuda.empty_cache()
except Exception as e:
    print(f"   ❌ Mamba block failed")
    print(f"      Error type: {type(e).__name__}")
    print(f"      Error msg:  {str(e)[:500]}")
    # Get full traceback WITHOUT using IPython
    tb_str = traceback.format_exc()
    print(f"\n      Full traceback:")
    for line in tb_str.split('\n')[-15:]:
        print(f"      {line}")
    mamba_works = False

# ============================================================
# If basic Mamba works, try SegMamba
# ============================================================
if mamba_works:
    print("\n" + "=" * 80)
    print("🧪 STEP 2: Test SegMamba model")
    print("=" * 80)
    
    SEGMAMBA_REPO = "/workspace/liver_tumor_project/baselines_code/segmamba_v2"
    for p in [f"{SEGMAMBA_REPO}/colorectal", f"{SEGMAMBA_REPO}/brats23", SEGMAMBA_REPO]:
        if p not in sys.path:
            sys.path.insert(0, p)
    
    for mod in list(sys.modules.keys()):
        if 'segmamba' in mod.lower() or 'models_segmamba' in mod.lower():
            try: del sys.modules[mod]
            except: pass
    
    try:
        from models_segmamba.segmambav2 import SegMamba
        print(f"   ✅ SegMamba imported")
        
        import inspect
        sig = inspect.signature(SegMamba.__init__)
        print(f"   Signature: {sig}")
        
        # Try default construction
        model = SegMamba().cuda()
        n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
        print(f"   ✅ Default model: {n_params/1e6:.2f}M params")
        
        # Try forward
        for n_ch in [4, 1]:
            try:
                torch.cuda.empty_cache()
                x = torch.randn(1, n_ch, 64, 64, 64).cuda()
                with torch.no_grad():
                    y = model(x)
                print(f"   ✅ {n_ch}-ch forward: {tuple(x.shape)} -> {tuple(y.shape)}")
                del x, y
                torch.cuda.empty_cache()
                
                print(f"\n{'=' * 80}")
                print(f"🎉 EVERYTHING WORKS! SegMamba-V2 ready with {n_ch}-channel input")
                print(f"{'=' * 80}")
                break
            except Exception as e:
                print(f"   ❌ {n_ch}-ch failed: {str(e)[:200]}")
                try: del x
                except: pass
                torch.cuda.empty_cache()
        
        del model
        torch.cuda.empty_cache()
        
    except Exception as e:
        print(f"   ❌ SegMamba failed")
        print(f"      Error type: {type(e).__name__}")
        print(f"      Error: {str(e)[:500]}")
        tb_str = traceback.format_exc()
        for line in tb_str.split('\n')[-15:]:
            print(f"      {line}")
else:
    print("\n" + "=" * 80)
    print("⚠️  Basic Mamba block not working - cannot proceed with SegMamba-V2")
    print("=" * 80)

print("\n" + "=" * 80)
print("✅ DEBUG CELL COMPLETE - No SystemExit, share full output")
print("=" * 80)

🔍 CLEAN DEBUG - Step by step

📊 Currently installed:
   Name: mamba-ssm
   Version: 2.2.4
   Location: /usr/local/lib/python3.10/dist-packages

   Name: causal-conv1d
   Version: 1.5.0.post8
   Location: /usr/local/lib/python3.10/dist-packages


📋 Env: Python cp310, PyTorch 2.4.0+cu121, CUDA 12.1

🧪 STEP 1: Test current install (whatever version)
   ✅ causal_conv1d imports
   ❌ mamba_ssm: ImportError: cannot import name 'getenv' from 'triton._C.libtriton' (/usr/local/lib/python3.10/dist-packages/triton/_C/libtriton.so)

🧪 Test basic Mamba block:
   ❌ Mamba block failed
      Error type: ImportError
      Error msg:  cannot import name 'getenv' from 'triton._C.libtriton' (/usr/local/lib/python3.10/dist-packages/triton/_C/libtriton.so)

      Full traceback:
          from mamba_ssm.ops.selective_scan_interface import selective_scan_fn, mamba_inner_fn
        File "/usr/local/lib/python3.10/dist-packages/mamba_ssm/ops/selective_scan_interface.py", line 16, in <module>
          from mamb

In [20]:
"""
=========================================================================
CLEANUP + CLONE TriALS Repository
=========================================================================
Step 1: Uninstall broken mamba/causal-conv1d
Step 2: Clone TriALS repo for MedNeXt trainer
Step 3: Discover available trainers
=========================================================================
"""
import subprocess
import os
import sys

print("=" * 80)
print("🧹 STEP 1: CLEANUP broken mamba installations")
print("=" * 80)

for pkg in ['mamba-ssm', 'causal-conv1d']:
    r = subprocess.run(['pip', 'uninstall', '-y', pkg],
                      capture_output=True, text=True, timeout=60)
    status = "✅ uninstalled" if r.returncode == 0 else "⚠️  skip"
    print(f"   {pkg}: {status}")

print("\n✅ Cleanup done\n")

# ============================================================
print("=" * 80)
print("📥 STEP 2: CLONE TriALS repository")
print("=" * 80)

TRIALS_DIR = "/workspace/liver_tumor_project/baselines_code/trials"

os.makedirs("/workspace/liver_tumor_project/baselines_code", exist_ok=True)

if os.path.exists(TRIALS_DIR):
    print(f"⚠️  Already exists: {TRIALS_DIR}")
    files = os.listdir(TRIALS_DIR)
    print(f"   Contains {len(files)} items - skipping clone")
else:
    print(f"Cloning from GitHub...")
    result = subprocess.run(
        ['git', 'clone', 'https://github.com/xmed-lab/TriALS.git', TRIALS_DIR],
        capture_output=True, text=True, timeout=180
    )
    if result.returncode == 0:
        print(f"✅ Clone successful!")
    else:
        print(f"❌ Clone failed:")
        print(f"   stderr: {result.stderr[-500:]}")

# ============================================================
print("\n" + "=" * 80)
print("🔍 STEP 3: DISCOVER MedNeXt trainer")
print("=" * 80)

if os.path.exists(TRIALS_DIR):
    # Find all files with MedNeXt, mednext in name
    print("\n📂 Searching for MedNeXt trainers...")
    mednext_files = []
    for root, dirs, files in os.walk(TRIALS_DIR):
        for f in files:
            if 'mednext' in f.lower() or 'MedNeXt' in f:
                full_path = os.path.join(root, f)
                size_kb = os.path.getsize(full_path) / 1024
                rel_path = full_path.replace(TRIALS_DIR + "/", "")
                mednext_files.append((rel_path, size_kb))
                print(f"   📄 {rel_path} ({size_kb:.1f} KB)")
    
    if not mednext_files:
        print("   ❌ No MedNeXt files found")
        print("\n   Checking all trainer variants:")
        for root, dirs, files in os.walk(TRIALS_DIR):
            if 'variants' in root or 'trainers' in root.lower():
                for f in files:
                    if f.endswith('.py'):
                        print(f"      {os.path.join(root, f).replace(TRIALS_DIR + '/', '')}")
    
    # Show top-level structure
    print("\n📁 Top-level structure:")
    for item in sorted(os.listdir(TRIALS_DIR))[:30]:
        full = os.path.join(TRIALS_DIR, item)
        marker = "📁" if os.path.isdir(full) else "📄"
        print(f"   {marker} {item}")
    
    # Look for README
    for readme in ['README.md', 'readme.md', 'README.rst']:
        readme_path = os.path.join(TRIALS_DIR, readme)
        if os.path.exists(readme_path):
            print(f"\n📖 README.md preview (first 60 lines):")
            print("-" * 80)
            with open(readme_path) as f:
                for i, line in enumerate(f.readlines()[:60], 1):
                    print(f"{i:3d}: {line}", end='')
            break

print("\n" + "=" * 80)
print("✅ Discovery complete - share output for next step")
print("=" * 80)

🧹 STEP 1: CLEANUP broken mamba installations
   mamba-ssm: ✅ uninstalled
   causal-conv1d: ✅ uninstalled

✅ Cleanup done

📥 STEP 2: CLONE TriALS repository
Cloning from GitHub...
✅ Clone successful!

🔍 STEP 3: DISCOVER MedNeXt trainer

📂 Searching for MedNeXt trainers...
   📄 docker/template/src/nnunetv2/nets/mednextv1/MedNextV1.py (14.5 KB)
   📄 docker/template/src/nnunetv2/nets/mednextv1/create_mednext_v1.py (2.5 KB)
   📄 docker/template/src/nnunetv2/training/nnUNetTrainer/nnUNetTrainerMedNext.py (10.3 KB)
   📄 nnunetv2/nets/mednextv1/MedNextV1.py (14.5 KB)
   📄 nnunetv2/nets/mednextv1/create_mednext_v1.py (2.5 KB)
   📄 nnunetv2/training/nnUNetTrainer/nnUNetTrainerMedNext.py (10.4 KB)

📁 Top-level structure:
   📁 .git
   📄 LICENSE
   📄 README.md
   📁 TriALS_2025_Interactive_Task_2
   📁 assets
   📁 checkpoints
   📁 data
   📁 docker
   📁 eval_scripts
   📄 main.py
   📁 nnunetv2
   📄 setup.py

📖 README.md preview (first 60 lines):
---------------------------------------------------------

In [21]:
"""
=========================================================================
Find MedNeXt trainer files + read README installation section
=========================================================================
"""
import os

TRIALS_DIR = "/workspace/liver_tumor_project/baselines_code/trials"

print("=" * 80)
print("🔍 MEDNEXT TRAINER FILE HUNT")
print("=" * 80)

# 1. Find ALL trainer files
print("\n📄 ALL trainer variant files:")
print("-" * 80)
trainer_files = []
for root, dirs, files in os.walk(TRIALS_DIR):
    for f in files:
        if f.endswith('.py') and ('Trainer' in f or 'trainer' in f):
            full_path = os.path.join(root, f)
            size_kb = os.path.getsize(full_path) / 1024
            rel = full_path.replace(TRIALS_DIR + "/", "")
            trainer_files.append((rel, size_kb))

# Filter for interesting trainers
mednext_related = [(p, s) for p, s in trainer_files if 'mednext' in p.lower()]
mamba_related = [(p, s) for p, s in trainer_files if 'mamba' in p.lower()]
samed_related = [(p, s) for p, s in trainer_files if 'samed' in p.lower()]
swin_related = [(p, s) for p, s in trainer_files if 'swin' in p.lower()]

print(f"\n🎯 MedNeXt trainers ({len(mednext_related)}):")
for p, s in mednext_related:
    print(f"   {p} ({s:.1f} KB)")

print(f"\n🐍 Mamba trainers ({len(mamba_related)}):")
for p, s in mamba_related:
    print(f"   {p} ({s:.1f} KB)")

print(f"\n🎪 SAMed trainers ({len(samed_related)}):")
for p, s in samed_related:
    print(f"   {p} ({s:.1f} KB)")

print(f"\n🌀 SwinUNETR trainers ({len(swin_related)}):")
for p, s in swin_related:
    print(f"   {p} ({s:.1f} KB)")

# 2. Show variants folder structure
print("\n" + "=" * 80)
print("📁 nnUNetTrainer variants folder structure:")
print("=" * 80)
for root, dirs, files in os.walk(TRIALS_DIR):
    if 'variants' in root and 'nnUNetTrainer' in root:
        rel = root.replace(TRIALS_DIR + "/", "")
        py_files = [f for f in files if f.endswith('.py')]
        if py_files:
            print(f"\n📁 {rel}/")
            for f in sorted(py_files):
                full_path = os.path.join(root, f)
                size_kb = os.path.getsize(full_path) / 1024
                print(f"   📄 {f} ({size_kb:.1f} KB)")

# 3. README - top installation section
print("\n" + "=" * 80)
print("📖 README - Installation & Usage section:")
print("=" * 80)
readme_path = os.path.join(TRIALS_DIR, "README.md")
if os.path.exists(readme_path):
    with open(readme_path) as f:
        content = f.read()
    
    # Find installation section
    lines = content.split('\n')
    
    # Show lines 1-30 (intro)
    print("\n📖 First 30 lines (intro):")
    print("-" * 80)
    for i, line in enumerate(lines[:30], 1):
        print(f"{i:3d}: {line}")
    
    # Show lines 55-120 (Getting Started section)
    print("\n📖 Lines 55-120 (Getting Started):")
    print("-" * 80)
    for i, line in enumerate(lines[54:120], 55):
        print(f"{i:3d}: {line}")

print("\n" + "=" * 80)
print("✅ Discovery complete")
print("=" * 80)

🔍 MEDNEXT TRAINER FILE HUNT

📄 ALL trainer variant files:
--------------------------------------------------------------------------------

🎯 MedNeXt trainers (2):
   docker/template/src/nnunetv2/training/nnUNetTrainer/nnUNetTrainerMedNext.py (10.3 KB)
   nnunetv2/training/nnUNetTrainer/nnUNetTrainerMedNext.py (10.4 KB)

🐍 Mamba trainers (4):
   docker/template/src/nnunetv2/training/nnUNetTrainer/nnUNetTrainerUMambaBot.py (1.4 KB)
   docker/template/src/nnunetv2/training/nnUNetTrainer/nnUNetTrainerUMambaEnc.py (1.2 KB)
   nnunetv2/training/nnUNetTrainer/nnUNetTrainerUMambaBot.py (1.4 KB)
   nnunetv2/training/nnUNetTrainer/nnUNetTrainerUMambaEnc.py (1.2 KB)

🎪 SAMed trainers (2):
   docker/template/src/nnunetv2/training/nnUNetTrainer/nnUNetTrainerSAMed.py (15.0 KB)
   nnunetv2/training/nnUNetTrainer/nnUNetTrainerSAMed.py (15.1 KB)

🌀 SwinUNETR trainers (2):
   docker/template/src/nnunetv2/training/nnUNetTrainer/nnUNetTrainerSwinUNETR.py (6.8 KB)
   nnunetv2/training/nnUNetTrainer/nnUNet

In [22]:
"""
Deep dive into TriALS structure - find MedNeXt files with better search
"""
import os
import subprocess

TRIALS_DIR = "/workspace/liver_tumor_project/baselines_code/trials"

print("=" * 80)
print("🔍 DEEP TRIALS EXPLORATION")
print("=" * 80)

# 1. Top-level structure
print("\n📁 TOP-LEVEL STRUCTURE:")
print("-" * 80)
for item in sorted(os.listdir(TRIALS_DIR)):
    full = os.path.join(TRIALS_DIR, item)
    if os.path.isdir(full):
        try:
            n = len(os.listdir(full))
            print(f"   📁 {item}/ ({n} items)")
        except:
            print(f"   📁 {item}/ (?)")
    else:
        size_kb = os.path.getsize(full) / 1024
        print(f"   📄 {item} ({size_kb:.1f} KB)")

# 2. Deep search for "MedNeXt" or "mednext" in ANY file (name or content)
print("\n\n🔍 SEARCH: 'mednext' or 'MedNeXt' in filenames anywhere:")
print("-" * 80)
result = subprocess.run(
    ['find', TRIALS_DIR, '-type', 'f', '-iname', '*mednext*'],
    capture_output=True, text=True
)
found = result.stdout.strip().split('\n') if result.stdout.strip() else []
if found and found[0]:
    for f in sorted(found):
        rel = f.replace(TRIALS_DIR + "/", "")
        try:
            size_kb = os.path.getsize(f) / 1024
            print(f"   📄 {rel} ({size_kb:.1f} KB)")
        except:
            print(f"   📄 {rel}")
else:
    print("   ❌ No files with 'mednext' in name!")

# 3. Same for other trainers
for pattern in ['*mamba*', '*samed*', '*trainer*', '*Trainer*']:
    print(f"\n🔍 SEARCH: files matching '{pattern}':")
    print("-" * 80)
    result = subprocess.run(
        ['find', TRIALS_DIR, '-type', 'f', '-name', pattern, '-not', '-path', '*/node_modules/*'],
        capture_output=True, text=True
    )
    found = result.stdout.strip().split('\n') if result.stdout.strip() else []
    if found and found[0]:
        # Limit to first 30
        for f in sorted(found)[:30]:
            rel = f.replace(TRIALS_DIR + "/", "")
            print(f"   📄 {rel}")
        if len(found) > 30:
            print(f"   ... ({len(found) - 30} more)")
    else:
        print("   (none)")

# 4. Search for the nnunetv2 sub-package
print("\n\n📁 Looking for nnunetv2 package structure:")
print("-" * 80)
for root, dirs, files in os.walk(TRIALS_DIR):
    if os.path.basename(root) == 'nnunetv2':
        rel = root.replace(TRIALS_DIR + "/", "")
        print(f"   📁 FOUND: {rel}")
        # Show subdirectories
        for d in sorted(os.listdir(root)):
            full = os.path.join(root, d)
            if os.path.isdir(full):
                try:
                    n = len(os.listdir(full))
                    print(f"      📁 {d}/ ({n})")
                except: pass
        break

# 5. Check network_architecture folder (where models usually are)
print("\n\n📁 Looking for network_architecture folders:")
print("-" * 80)
for root, dirs, files in os.walk(TRIALS_DIR):
    if 'network_architecture' in root:
        rel = root.replace(TRIALS_DIR + "/", "")
        py_files = [f for f in files if f.endswith('.py')]
        if py_files:
            print(f"\n   📁 {rel}/")
            for f in sorted(py_files)[:20]:
                size_kb = os.path.getsize(os.path.join(root, f)) / 1024
                print(f"      📄 {f} ({size_kb:.1f} KB)")

# 6. Check for trainer variants folder
print("\n\n📁 Looking for variants folders:")
print("-" * 80)
for root, dirs, files in os.walk(TRIALS_DIR):
    if os.path.basename(root) == 'variants':
        rel = root.replace(TRIALS_DIR + "/", "")
        print(f"\n   📁 {rel}/")
        # Recurse into it
        for r2, d2, f2 in os.walk(root):
            level = r2.replace(root, '').count(os.sep)
            indent = '   ' * level
            print(f"      {indent}📁 {os.path.basename(r2)}/")
            for f in sorted(f2):
                if f.endswith('.py'):
                    size_kb = os.path.getsize(os.path.join(r2, f)) / 1024
                    print(f"      {indent}   📄 {f} ({size_kb:.1f} KB)")

print("\n" + "=" * 80)
print("✅ Deep search complete")
print("=" * 80)

🔍 DEEP TRIALS EXPLORATION

📁 TOP-LEVEL STRUCTURE:
--------------------------------------------------------------------------------
   📁 .git/ (11 items)
   📄 LICENSE (11.2 KB)
   📄 README.md (11.8 KB)
   📁 TriALS_2025_Interactive_Task_2/ (3 items)
   📁 assets/ (3 items)
   📁 checkpoints/ (1 items)
   📁 data/ (3 items)
   📁 docker/ (3 items)
   📁 eval_scripts/ (8 items)
   📄 main.py (0.5 KB)
   📁 nnunetv2/ (18 items)
   📄 setup.py (4.3 KB)


🔍 SEARCH: 'mednext' or 'MedNeXt' in filenames anywhere:
--------------------------------------------------------------------------------
   📄 docker/template/src/nnunetv2/nets/mednextv1/MedNextV1.py (14.5 KB)
   📄 docker/template/src/nnunetv2/nets/mednextv1/create_mednext_v1.py (2.5 KB)
   📄 docker/template/src/nnunetv2/training/nnUNetTrainer/nnUNetTrainerMedNext.py (10.3 KB)
   📄 nnunetv2/nets/mednextv1/MedNextV1.py (14.5 KB)
   📄 nnunetv2/nets/mednextv1/create_mednext_v1.py (2.5 KB)
   📄 nnunetv2/training/nnUNetTrainer/nnUNetTrainerMedNext.py (10.

In [23]:
"""
Ultimate MedNeXt hunt - use grep to find any reference in code
"""
import subprocess

TRIALS_DIR = "/workspace/liver_tumor_project/baselines_code/trials"

print("=" * 70)
print("🎯 ULTIMATE MEDNEXT HUNT")
print("=" * 70)

# Search 1: any file with "mednext" in NAME (case-insensitive)
print("\n1️⃣  Files with 'mednext' in NAME (case-insensitive):")
r = subprocess.run(
    ['find', TRIALS_DIR, '-iname', '*mednext*', '-type', 'f'],
    capture_output=True, text=True
)
if r.stdout.strip():
    for line in r.stdout.strip().split('\n'):
        print(f"   {line}")
else:
    print("   ❌ NONE")

# Search 2: any file with "MedNext" or "MedNeXt" (both spellings)
print("\n2️⃣  Files with 'MedNext' or 'MedNeXt' in NAME:")
for pattern in ['*MedNext*', '*MedNeXt*']:
    r = subprocess.run(
        ['find', TRIALS_DIR, '-name', pattern, '-type', 'f'],
        capture_output=True, text=True
    )
    if r.stdout.strip():
        for line in r.stdout.strip().split('\n'):
            print(f"   {line}")

# Search 3: grep for 'MedNeXt' in ALL python files (find where it's referenced)
print("\n3️⃣  Files that CONTAIN 'MedNeXt' or 'mednext' in code:")
r = subprocess.run(
    ['grep', '-r', '-l', '-i', 'mednext', TRIALS_DIR, '--include=*.py'],
    capture_output=True, text=True
)
if r.stdout.strip():
    for line in r.stdout.strip().split('\n')[:20]:
        rel = line.replace(TRIALS_DIR + '/', '')
        print(f"   {rel}")
else:
    print("   ❌ NONE - MedNeXt not mentioned in any .py file!")

# Search 4: mamba
print("\n4️⃣  Files with 'mamba' in NAME (case-insensitive):")
r = subprocess.run(
    ['find', TRIALS_DIR, '-iname', '*mamba*', '-type', 'f'],
    capture_output=True, text=True
)
if r.stdout.strip():
    for line in r.stdout.strip().split('\n'):
        print(f"   {line}")
else:
    print("   ❌ NONE")

# Search 5: SAMed
print("\n5️⃣  Files with 'samed' in NAME (case-insensitive):")
r = subprocess.run(
    ['find', TRIALS_DIR, '-iname', '*samed*', '-type', 'f'],
    capture_output=True, text=True
)
if r.stdout.strip():
    for line in r.stdout.strip().split('\n'):
        print(f"   {line}")
else:
    print("   ❌ NONE")

# Search 6: Count total .py files
print("\n6️⃣  Total .py files in repo:")
r = subprocess.run(
    ['find', TRIALS_DIR, '-name', '*.py', '-type', 'f'],
    capture_output=True, text=True
)
py_count = len(r.stdout.strip().split('\n')) if r.stdout.strip() else 0
print(f"   {py_count} .py files")

# Search 7: git log to see if MedNeXt was ever committed
print("\n7️⃣  Recent commits mentioning MedNeXt:")
r = subprocess.run(
    ['git', '-C', TRIALS_DIR, 'log', '--oneline', '--grep=MedNeXt', '--grep=mednext', '-i', '-n', '10'],
    capture_output=True, text=True
)
if r.stdout.strip():
    for line in r.stdout.strip().split('\n'):
        print(f"   {line}")
else:
    print("   No commits with MedNeXt in message")

# Search 8: Show branches
print("\n8️⃣  Available branches:")
r = subprocess.run(
    ['git', '-C', TRIALS_DIR, 'branch', '-a'],
    capture_output=True, text=True
)
print(r.stdout)

print("=" * 70)
print("✅ Search done")
print("=" * 70)

🎯 ULTIMATE MEDNEXT HUNT

1️⃣  Files with 'mednext' in NAME (case-insensitive):
   /workspace/liver_tumor_project/baselines_code/trials/docker/template/src/nnunetv2/nets/mednextv1/MedNextV1.py
   /workspace/liver_tumor_project/baselines_code/trials/docker/template/src/nnunetv2/nets/mednextv1/create_mednext_v1.py
   /workspace/liver_tumor_project/baselines_code/trials/docker/template/src/nnunetv2/training/nnUNetTrainer/nnUNetTrainerMedNext.py
   /workspace/liver_tumor_project/baselines_code/trials/nnunetv2/nets/mednextv1/MedNextV1.py
   /workspace/liver_tumor_project/baselines_code/trials/nnunetv2/nets/mednextv1/create_mednext_v1.py
   /workspace/liver_tumor_project/baselines_code/trials/nnunetv2/training/nnUNetTrainer/nnUNetTrainerMedNext.py

2️⃣  Files with 'MedNext' or 'MedNeXt' in NAME:
   /workspace/liver_tumor_project/baselines_code/trials/docker/template/src/nnunetv2/nets/mednextv1/MedNextV1.py
   /workspace/liver_tumor_project/baselines_code/trials/docker/template/src/nnunetv2/tr

In [24]:
"""
=========================================================================
COMPREHENSIVE HUNT - MedNeXt code kaha hai TriALS mein
=========================================================================
"""
import subprocess
import os

TRIALS_DIR = "/workspace/liver_tumor_project/baselines_code/trials"

print("=" * 70)
print("🔍 COMPREHENSIVE MEDNEXT HUNT")
print("=" * 70)

# 1. Check ALL branches
print("\n1️⃣  Available branches:")
r = subprocess.run(['git', '-C', TRIALS_DIR, 'branch', '-a'],
                  capture_output=True, text=True)
print(r.stdout if r.stdout else "   (git error)")

# 2. Git submodules?
print("\n2️⃣  Git submodules:")
r = subprocess.run(['git', '-C', TRIALS_DIR, 'submodule', 'status'],
                  capture_output=True, text=True)
if r.stdout.strip():
    print(r.stdout)
else:
    print("   (no submodules)")

# Also check .gitmodules file
gitmod = os.path.join(TRIALS_DIR, ".gitmodules")
if os.path.exists(gitmod):
    print(f"\n   Found .gitmodules file:")
    with open(gitmod) as f:
        print(f"   {f.read()}")

# 3. Case-insensitive file name search (multiple patterns)
print("\n3️⃣  All possible MedNeXt file patterns:")
patterns = ['*mednext*', '*MedNext*', '*MedNeXt*', '*med_next*', '*med-next*', '*MedNeXtV*']
found_files = set()
for pattern in patterns:
    r = subprocess.run(['find', TRIALS_DIR, '-iname', pattern, '-type', 'f'],
                      capture_output=True, text=True)
    if r.stdout.strip():
        for line in r.stdout.strip().split('\n'):
            found_files.add(line)

if found_files:
    for f in sorted(found_files):
        print(f"   ✅ {f.replace(TRIALS_DIR + '/', '')}")
else:
    print("   ❌ No MedNeXt file with any variation")

# 4. GREP: Find any .py file that mentions MedNeXt in code
print("\n4️⃣  Python files that MENTION MedNeXt in code:")
r = subprocess.run(
    ['grep', '-r', '-l', '-i', '-E', 'mednext|med_next', TRIALS_DIR, '--include=*.py'],
    capture_output=True, text=True
)
if r.stdout.strip():
    for line in r.stdout.strip().split('\n')[:15]:
        print(f"   ✅ {line.replace(TRIALS_DIR + '/', '')}")
else:
    print("   ❌ MedNeXt not mentioned in any Python file")

# 5. Also check YAML, MD, TXT files (may have config or setup)
print("\n5️⃣  Non-python files that mention MedNeXt:")
r = subprocess.run(
    ['grep', '-r', '-l', '-i', '-E', 'mednext|med_next', TRIALS_DIR,
     '--include=*.yaml', '--include=*.yml', '--include=*.md',
     '--include=*.txt', '--include=*.sh', '--include=*.cfg',
     '--include=setup.py', '--include=setup.cfg', '--include=pyproject.toml'],
    capture_output=True, text=True
)
if r.stdout.strip():
    for line in r.stdout.strip().split('\n')[:15]:
        print(f"   ✅ {line.replace(TRIALS_DIR + '/', '')}")

# 6. Check requirements.txt and setup.py for dependency
print("\n6️⃣  Requirements & setup files:")
for fname in ['requirements.txt', 'setup.py', 'setup.cfg', 'pyproject.toml']:
    fpath = os.path.join(TRIALS_DIR, fname)
    if os.path.exists(fpath):
        print(f"\n   📄 {fname}:")
        with open(fpath) as f:
            content = f.read()
        for line in content.split('\n'):
            if any(k in line.lower() for k in ['mednext', 'mamba', 'samed', 'nnunet', 'monai', 'timm']):
                print(f"      {line}")

# 7. All folders at each depth level, with unique names
print("\n7️⃣  Unique folder names in repo (find hidden folders):")
r = subprocess.run(['find', TRIALS_DIR, '-type', 'd', '-not', '-path', '*/.git/*'],
                  capture_output=True, text=True)
if r.stdout:
    folders = set()
    for line in r.stdout.strip().split('\n'):
        name = os.path.basename(line)
        if name and name != 'trials':
            folders.add(name)
    unique_folders = sorted(folders)
    print(f"   Total unique folder names: {len(unique_folders)}")
    for f in unique_folders:
        print(f"   📁 {f}")

# 8. Also check if there's a MedNeXt or medn folder anywhere
print("\n8️⃣  Folders named like 'med*' or 'mamba' or 'samed':")
r = subprocess.run(['find', TRIALS_DIR, '-type', 'd', '-iname', '*med*'],
                  capture_output=True, text=True)
if r.stdout.strip():
    for line in r.stdout.strip().split('\n'):
        print(f"   {line.replace(TRIALS_DIR + '/', '')}")

r = subprocess.run(['find', TRIALS_DIR, '-type', 'd', '-iname', '*mamba*'],
                  capture_output=True, text=True)
if r.stdout.strip():
    for line in r.stdout.strip().split('\n'):
        print(f"   {line.replace(TRIALS_DIR + '/', '')}")

print("\n" + "=" * 70)
print("✅ Comprehensive hunt complete")
print("=" * 70)

🔍 COMPREHENSIVE MEDNEXT HUNT

1️⃣  Available branches:
* main
  remotes/origin/HEAD -> origin/main
  remotes/origin/main


2️⃣  Git submodules:
   (no submodules)

3️⃣  All possible MedNeXt file patterns:
   ✅ docker/template/src/nnunetv2/nets/mednextv1/MedNextV1.py
   ✅ docker/template/src/nnunetv2/nets/mednextv1/create_mednext_v1.py
   ✅ docker/template/src/nnunetv2/training/nnUNetTrainer/nnUNetTrainerMedNext.py
   ✅ nnunetv2/nets/mednextv1/MedNextV1.py
   ✅ nnunetv2/nets/mednextv1/create_mednext_v1.py
   ✅ nnunetv2/training/nnUNetTrainer/nnUNetTrainerMedNext.py

4️⃣  Python files that MENTION MedNeXt in code:
   ✅ docker/template/src/nnunetv2/nets/mednextv1/MedNextV1.py
   ✅ docker/template/src/nnunetv2/nets/mednextv1/blocks.py
   ✅ docker/template/src/nnunetv2/nets/mednextv1/create_mednext_v1.py
   ✅ docker/template/src/nnunetv2/training/nnUNetTrainer/nnUNetTrainerMedNext.py
   ✅ nnunetv2/nets/mednextv1/MedNextV1.py
   ✅ nnunetv2/nets/mednextv1/blocks.py
   ✅ nnunetv2/nets/mednextv

In [25]:
"""
Preview MedNeXt trainer file - understand what it needs
"""
import os

TRIALS_DIR = "/workspace/liver_tumor_project/baselines_code/trials"
MEDNEXT_TRAINER = f"{TRIALS_DIR}/nnunetv2/training/nnUNetTrainer/nnUNetTrainerMedNext.py"

print("=" * 80)
print("📖 MedNeXt Trainer Preview")
print("=" * 80)

if os.path.exists(MEDNEXT_TRAINER):
    with open(MEDNEXT_TRAINER) as f:
        content = f.read()
    
    lines = content.split('\n')
    print(f"\nTotal lines: {len(lines)}\n")
    print("-" * 80)
    print("FULL CONTENT:")
    print("-" * 80)
    for i, line in enumerate(lines, 1):
        print(f"{i:4d}: {line}")
else:
    print(f"❌ File not found: {MEDNEXT_TRAINER}")

# Also check MedNext-B config to know model variant
print("\n\n" + "=" * 80)
print("📖 create_mednext_v1.py preview (model variants)")
print("=" * 80)
CREATE_FILE = f"{TRIALS_DIR}/nnunetv2/nets/mednextv1/create_mednext_v1.py"
if os.path.exists(CREATE_FILE):
    with open(CREATE_FILE) as f:
        content = f.read()
    lines = content.split('\n')
    print(f"\nTotal lines: {len(lines)}\n")
    # Show first 80 lines
    for i, line in enumerate(lines[:80], 1):
        print(f"{i:4d}: {line}")

# Setup.py to see dependencies
print("\n\n" + "=" * 80)
print("📖 setup.py preview")
print("=" * 80)
SETUP_FILE = f"{TRIALS_DIR}/setup.py"
if os.path.exists(SETUP_FILE):
    with open(SETUP_FILE) as f:
        content = f.read()
    print(content)

print("\n" + "=" * 80)
print("✅ Preview done")
print("=" * 80)

📖 MedNeXt Trainer Preview

Total lines: 260

--------------------------------------------------------------------------------
FULL CONTENT:
--------------------------------------------------------------------------------
   1: from nnunetv2.training.nnUNetTrainer.variants.network_architecture.nnUNetTrainerNoDeepSupervision import \
   2:     nnUNetTrainerNoDeepSupervision
   3: from nnunetv2.utilities.plans_handling.plans_handler import ConfigurationManager, PlansManager
   4: from nnunetv2.training.loss.dice import get_tp_fp_fn_tn
   5: import torch
   6: from torch.optim import AdamW
   7: from torch.optim.lr_scheduler import CosineAnnealingLR
   8: from torch import nn
   9: from nnunetv2.nets.mednextv1.MedNextV1 import MedNeXt
  10: from monai.networks.nets import SwinUNETR
  11: 
  12: class nnUNetTrainerMedNext(nnUNetTrainerNoDeepSupervision):
  13:     def __init__(self, plans: dict, configuration: str, fold: int, dataset_json: dict, unpack_dataset: bool = True,
  14:           

In [26]:
"""
Preview MedNeXt trainer file + create_mednext_v1.py + setup.py
"""
import os

TRIALS_DIR = "/workspace/liver_tumor_project/baselines_code/trials"

print("=" * 80)
print("📖 PREVIEW 1: MedNeXt Trainer")
print("=" * 80)

MEDNEXT_TRAINER = f"{TRIALS_DIR}/nnunetv2/training/nnUNetTrainer/nnUNetTrainerMedNext.py"

if os.path.exists(MEDNEXT_TRAINER):
    with open(MEDNEXT_TRAINER) as f:
        content = f.read()
    lines = content.split('\n')
    print(f"\nFile: {MEDNEXT_TRAINER}")
    print(f"Total lines: {len(lines)}\n")
    print("-" * 80)
    for i, line in enumerate(lines, 1):
        print(f"{i:4d}: {line}")
else:
    print("❌ File not found")

print("\n\n" + "=" * 80)
print("📖 PREVIEW 2: create_mednext_v1.py (variants)")
print("=" * 80)

CREATE_FILE = f"{TRIALS_DIR}/nnunetv2/nets/mednextv1/create_mednext_v1.py"
if os.path.exists(CREATE_FILE):
    with open(CREATE_FILE) as f:
        content = f.read()
    print(content)

print("\n\n" + "=" * 80)
print("📖 PREVIEW 3: setup.py (dependencies)")
print("=" * 80)

SETUP_FILE = f"{TRIALS_DIR}/setup.py"
if os.path.exists(SETUP_FILE):
    with open(SETUP_FILE) as f:
        content = f.read()
    print(content)

print("\n" + "=" * 80)
print("✅ All previews done")
print("=" * 80)

📖 PREVIEW 1: MedNeXt Trainer

File: /workspace/liver_tumor_project/baselines_code/trials/nnunetv2/training/nnUNetTrainer/nnUNetTrainerMedNext.py
Total lines: 260

--------------------------------------------------------------------------------
   1: from nnunetv2.training.nnUNetTrainer.variants.network_architecture.nnUNetTrainerNoDeepSupervision import \
   2:     nnUNetTrainerNoDeepSupervision
   3: from nnunetv2.utilities.plans_handling.plans_handler import ConfigurationManager, PlansManager
   4: from nnunetv2.training.loss.dice import get_tp_fp_fn_tn
   5: import torch
   6: from torch.optim import AdamW
   7: from torch.optim.lr_scheduler import CosineAnnealingLR
   8: from torch import nn
   9: from nnunetv2.nets.mednextv1.MedNextV1 import MedNeXt
  10: from monai.networks.nets import SwinUNETR
  11: 
  12: class nnUNetTrainerMedNext(nnUNetTrainerNoDeepSupervision):
  13:     def __init__(self, plans: dict, configuration: str, fold: int, dataset_json: dict, unpack_dataset: bool =

In [27]:
# Just show trainer file - nothing else
import os

TRAINER = "/workspace/liver_tumor_project/baselines_code/trials/nnunetv2/training/nnUNetTrainer/nnUNetTrainerMedNext.py"

print(f"File exists: {os.path.exists(TRAINER)}")
print(f"Size: {os.path.getsize(TRAINER)} bytes")
print(f"Lines: {sum(1 for _ in open(TRAINER))}")
print()
print("=" * 70)
print("FULL CONTENT OF nnUNetTrainerMedNext.py:")
print("=" * 70)

with open(TRAINER) as f:
    print(f.read())

print("=" * 70)
print("END")
print("=" * 70)

File exists: True
Size: 10630 bytes
Lines: 259

FULL CONTENT OF nnUNetTrainerMedNext.py:
from nnunetv2.training.nnUNetTrainer.variants.network_architecture.nnUNetTrainerNoDeepSupervision import \
    nnUNetTrainerNoDeepSupervision
from nnunetv2.utilities.plans_handling.plans_handler import ConfigurationManager, PlansManager
from nnunetv2.training.loss.dice import get_tp_fp_fn_tn
import torch
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch import nn
from nnunetv2.nets.mednextv1.MedNextV1 import MedNeXt
from monai.networks.nets import SwinUNETR

class nnUNetTrainerMedNext(nnUNetTrainerNoDeepSupervision):
    def __init__(self, plans: dict, configuration: str, fold: int, dataset_json: dict, unpack_dataset: bool = True,
                 device: torch.device = torch.device('cuda')):
        super().__init__(plans, configuration, fold, dataset_json, unpack_dataset, device)
        original_patch_size = self.configuration_manager.patch_size
    

In [29]:
"""
=========================================================================
FIX: Patch MedNeXt imports for standalone use
=========================================================================
"""
import os
import re

MEDNEXT_DIR = "/workspace/liver_tumor_project/baselines_code/mednext_standalone/mednextv1"

print("=" * 80)
print("🔧 PATCHING MedNeXt imports")
print("=" * 80)

# Find all .py files
py_files = []
for root, dirs, files in os.walk(MEDNEXT_DIR):
    for f in files:
        if f.endswith('.py'):
            py_files.append(os.path.join(root, f))

print(f"\nFound {len(py_files)} .py files to check")

# Patterns to replace: "from nnunetv2.nets.mednextv1.X" -> "from mednextv1.X"
patterns = [
    (r'from nnunetv2\.nets\.mednextv1', 'from mednextv1'),
    (r'from nnunetv2\.nets\.mednextv1\.', 'from mednextv1.'),
    (r'import nnunetv2\.nets\.mednextv1', 'import mednextv1'),
]

patched_count = 0
for filepath in py_files:
    rel = filepath.replace(MEDNEXT_DIR + '/', '')
    with open(filepath, 'r') as f:
        content = f.read()
    
    original = content
    for pattern, replacement in patterns:
        content = re.sub(pattern, replacement, content)
    
    if content != original:
        with open(filepath, 'w') as f:
            f.write(content)
        patched_count += 1
        print(f"   ✅ Patched: {rel}")
    else:
        print(f"   ⏭️  No changes needed: {rel}")

print(f"\n📊 Patched {patched_count} files")

# Verify by testing import
print("\n" + "=" * 80)
print("🧪 Testing import after patch")
print("=" * 80)

import sys
STANDALONE_DIR = "/workspace/liver_tumor_project/baselines_code/mednext_standalone"
if STANDALONE_DIR not in sys.path:
    sys.path.insert(0, STANDALONE_DIR)

# Clear any cached mednextv1 modules
for mod in list(sys.modules.keys()):
    if 'mednext' in mod.lower():
        try: del sys.modules[mod]
        except: pass

try:
    from mednextv1.create_mednext_v1 import create_mednext_v1
    print(f"   ✅ create_mednext_v1 imported successfully")
    
    import torch
    torch.cuda.empty_cache()
    
    # Test model creation
    model = create_mednext_v1(
        num_input_channels=1,
        num_classes=3,
        model_id='B',
        kernel_size=3,
        deep_supervision=False
    ).cuda()
    
    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"   ✅ MedNeXt-B: {n_params/1e6:.2f}M parameters")
    
    # Forward pass
    with torch.no_grad():
        x = torch.randn(1, 1, 64, 64, 64).cuda()
        y = model(x)
    print(f"   ✅ Forward pass OK: {tuple(x.shape)} -> {tuple(y.shape)}")
    
    del model, x, y
    torch.cuda.empty_cache()
    
    print(f"\n{'=' * 80}")
    print(f"🎉 ALL WORKING! Ready to launch full training")
    print(f"{'=' * 80}")
    
except ImportError as ie:
    print(f"   ❌ Import still fails: {ie}")
    # Show remaining imports in files
    print(f"\n   Checking remaining nnunetv2 references:")
    for filepath in py_files:
        with open(filepath) as f:
            content = f.read()
        for i, line in enumerate(content.split('\n'), 1):
            if 'nnunetv2' in line:
                rel = filepath.replace(MEDNEXT_DIR + '/', '')
                print(f"      {rel}:{i}: {line.strip()}")

except Exception as e:
    import traceback
    print(f"   ❌ Failed: {e}")
    tb = traceback.format_exc()
    for line in tb.split('\n')[-15:]:
        print(f"      {line}")

🔧 PATCHING MedNeXt imports

Found 4 .py files to check
   ✅ Patched: MedNextV1.py
   ⏭️  No changes needed: __init__.py
   ⏭️  No changes needed: blocks.py
   ✅ Patched: create_mednext_v1.py

📊 Patched 2 files

🧪 Testing import after patch
   ✅ create_mednext_v1 imported successfully
   ✅ MedNeXt-B: 10.53M parameters
   ✅ Forward pass OK: (1, 1, 64, 64, 64) -> (1, 3, 64, 64, 64)

🎉 ALL WORKING! Ready to launch full training


In [32]:
"""
Find all torch.utils.checkpoint usage in MedNeXt code
"""
import os

MEDNEXT_DIR = "/workspace/liver_tumor_project/baselines_code/mednext_standalone/mednextv1"

print("=" * 80)
print("🔍 SEARCHING checkpoint usage in MedNeXt code")
print("=" * 80)

# Find all py files
py_files = []
for root, dirs, files in os.walk(MEDNEXT_DIR):
    for f in files:
        if f.endswith('.py'):
            py_files.append(os.path.join(root, f))

# Search for checkpoint patterns
patterns = ['checkpoint', 'gradient_checkpointing', 'utils.checkpoint']

for filepath in py_files:
    rel = filepath.replace(MEDNEXT_DIR + '/', '')
    with open(filepath) as f:
        content = f.read()
    
    lines = content.split('\n')
    matches = []
    for i, line in enumerate(lines, 1):
        for pattern in patterns:
            if pattern in line.lower():
                matches.append((i, line.rstrip()))
                break
    
    if matches:
        print(f"\n📄 {rel} ({len(matches)} matches):")
        print("-" * 80)
        for line_num, line in matches:
            print(f"   Line {line_num}: {line}")

print("\n" + "=" * 80)
print("✅ Search done")
print("=" * 80)

🔍 SEARCHING checkpoint usage in MedNeXt code

📄 MedNextV1.py (37 matches):
--------------------------------------------------------------------------------
   Line 3: import torch.utils.checkpoint as checkpoint
   Line 20:         checkpoint_style: bool = None,            # Either inside block or outside block
   Line 31:         assert checkpoint_style in [None, 'outside_block']
   Line 32:         self.inside_block_checkpointing = False
   Line 33:         self.outside_block_checkpointing = False
   Line 34:         if checkpoint_style == 'outside_block':
   Line 35:             self.outside_block_checkpointing = True
   Line 266:         # Used to fix PyTorch checkpointing bug
   Line 278:     def iterative_checkpoint(self, sequential_block, x):
   Line 281:         using gradient_checkpointing. This implementation is designed to bypass
   Line 282:         the following issue in PyTorch's gradient checkpointing:
   Line 283:         https://discuss.pytorch.org/t/checkpoint-with-no-

In [41]:
"""
Check both raw data locations for actual HU values
"""
import os
import nibabel as nib

print("=" * 80)
print("🔍 CHECK BOTH RAW LOCATIONS")
print("=" * 80)

# Location 1: data/raw
raw_dir_1 = "/workspace/liver_tumor_project/data/raw"
print(f"\n📁 Location 1: {raw_dir_1}")
if os.path.exists(raw_dir_1):
    contents = os.listdir(raw_dir_1)
    print(f"   Contents: {contents[:10]}")
    
    # Try to find volume-79 or similar
    for root, dirs, files in os.walk(raw_dir_1):
        for f in files:
            if 'volume-79' in f or 'volume_79' in f:
                path = os.path.join(root, f)
                print(f"\n   Found: {path}")
                try:
                    nii = nib.load(path)
                    data = nii.get_fdata()
                    print(f"   Shape: {data.shape}")
                    print(f"   Range: [{data.min():.1f}, {data.max():.1f}]")
                    print(f"   Mean: {data.mean():.1f}, Std: {data.std():.1f}")
                    print(f"   Is HU scale? {'YES ✅' if data.min() < -100 else 'NO ❌ (looks normalized)'}")
                except Exception as e:
                    print(f"   Error: {e}")
                break
        break  # only top-level

# Location 2: nnUNet_raw
raw_dir_2 = "/workspace/liver_tumor_project/models/nnunet_v2/nnUNet_raw/Dataset001_LiTS/imagesTr"
print(f"\n📁 Location 2: {raw_dir_2}")
if os.path.exists(raw_dir_2):
    files = sorted(os.listdir(raw_dir_2))[:5]
    print(f"   Sample files: {files}")
    
    # nnU-Net naming convention: volume-79_0000.nii.gz
    for f in files:
        if '79' in f:
            path = os.path.join(raw_dir_2, f)
            print(f"\n   Found: {path}")
            try:
                nii = nib.load(path)
                data = nii.get_fdata()
                print(f"   Shape: {data.shape}")
                print(f"   Range: [{data.min():.1f}, {data.max():.1f}]")
                print(f"   Mean: {data.mean():.1f}, Std: {data.std():.1f}")
                print(f"   Is HU scale? {'YES ✅' if data.min() < -100 else 'NO ❌ (looks normalized)'}")
            except Exception as e:
                print(f"   Error: {e}")
            break

🔍 CHECK BOTH RAW LOCATIONS

📁 Location 1: /workspace/liver_tumor_project/data/raw
   Contents: ['LiTS']

📁 Location 2: /workspace/liver_tumor_project/models/nnunet_v2/nnUNet_raw/Dataset001_LiTS/imagesTr
   Sample files: ['volume-0_0000.nii.gz', 'volume-100_0000.nii.gz', 'volume-101_0000.nii.gz', 'volume-102_0000.nii.gz', 'volume-103_0000.nii.gz']


In [44]:
"""
=========================================================================
COMPARATIVE STUDY: 36 vessel extraction combinations
=========================================================================
Grid: 4 filters × 3 preprocessing × 3 thresholds = 36
Score each combo objectively, rank, save top 5 visualizations
=========================================================================
"""
import os
import time
import numpy as np
import pandas as pd
import nibabel as nib
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from skimage.filters import frangi, sato, meijering, threshold_otsu, hessian
from skimage.exposure import equalize_adapthist
from scipy.ndimage import gaussian_filter, label as ndi_label
from itertools import product

print("=" * 80)
print("🧪 COMPARATIVE VESSEL EXTRACTION STUDY")
print("=" * 80)

V2_ROOT = "/workspace/liver_tumor_project/data/preprocessed"
OUT_DIR = "/workspace/liver_tumor_project/novel_method/comparative_study"
os.makedirs(OUT_DIR, exist_ok=True)

# ============================================================
# Preprocessing functions
# ============================================================
def preprocess_none(ct, liver_mask):
    """Just apply liver mask"""
    return (ct * liver_mask).astype(np.float32)

def preprocess_percentile(ct, liver_mask):
    """Stretch 2-98 percentile within liver"""
    liver_ct = ct[liver_mask]
    if len(liver_ct) == 0:
        return ct * liver_mask
    p2, p98 = np.percentile(liver_ct, [2, 98])
    result = np.clip((ct - p2) / (p98 - p2 + 1e-6), 0, 1)
    return (result * liver_mask).astype(np.float32)

def preprocess_clahe(ct, liver_mask):
    """CLAHE per slice"""
    result = np.zeros_like(ct)
    for z in range(ct.shape[2]):
        s = ct[:, :, z]
        if s.max() > s.min():
            try:
                result[:, :, z] = equalize_adapthist(s, clip_limit=0.02, kernel_size=32)
            except:
                result[:, :, z] = s
    return (result * liver_mask).astype(np.float32)

# ============================================================
# Filter functions (all with gaussian smoothing 0.5)
# ============================================================
def filter_frangi(volume):
    v = gaussian_filter(volume, sigma=0.5)
    return frangi(v, sigmas=np.arange(1.0, 3.5, 0.5), 
                  alpha=0.5, beta=0.5, gamma=None, black_ridges=False)

def filter_sato(volume):
    v = gaussian_filter(volume, sigma=0.5)
    return sato(v, sigmas=[1.0, 1.5, 2.0, 2.5, 3.0], black_ridges=False)

def filter_meijering(volume):
    v = gaussian_filter(volume, sigma=0.5)
    return meijering(v, sigmas=[1.0, 1.5, 2.0, 2.5, 3.0], black_ridges=False)

def filter_hessian(volume):
    v = gaussian_filter(volume, sigma=0.5)
    return hessian(v, sigmas=[1.0, 1.5, 2.0, 2.5, 3.0], 
                   alpha=0.5, beta=0.5, gamma=15, black_ridges=False)

# ============================================================
# Threshold functions
# ============================================================
def thresh_percentile(response, liver_mask, pct=85):
    r = response[liver_mask]
    r_nz = r[r > 0]
    if len(r_nz) == 0:
        return 0.0
    return float(np.percentile(r_nz, pct))

def thresh_otsu(response, liver_mask):
    r = response[liver_mask]
    r_nz = r[r > 0]
    if len(r_nz) < 100:
        return 0.0
    try:
        return float(threshold_otsu(r_nz))
    except:
        return float(np.percentile(r_nz, 75))

def thresh_adaptive(response, liver_mask):
    """Mean + 2 std within liver"""
    r = response[liver_mask]
    r_nz = r[r > 0]
    if len(r_nz) == 0:
        return 0.0
    return float(r_nz.mean() + 2 * r_nz.std())

# ============================================================
# Scoring function
# ============================================================
def score_extraction(vessel_mask, liver_mask, tumor_mask):
    """Objective quality score"""
    n_vessel = vessel_mask.sum()
    n_liver = liver_mask.sum()
    n_tumor = tumor_mask.sum()
    
    if n_vessel == 0 or n_liver == 0:
        return {'pct_score': 0, 'tumor_score': 0, 'conn_score': 0, 
                'size_score': 0, 'total': 0, 'n_vessel': 0, 'pct': 0, 
                'tumor_overlap': 0, 'n_components': 0, 'mean_size': 0}
    
    # M1: Percentage score (5-15% ideal)
    pct = 100 * n_vessel / n_liver
    if 5 <= pct <= 15:
        pct_score = 1.0
    elif 3 <= pct <= 20:
        pct_score = 0.5
    else:
        pct_score = max(0, 1.0 - abs(pct - 10) / 20)
    
    # M2: Tumor overlap (should be low)
    if n_tumor > 0:
        vessel_in_tumor = (vessel_mask & tumor_mask).sum()
        tumor_overlap = vessel_in_tumor / n_tumor
        tumor_score = 1.0 - min(1.0, tumor_overlap)
    else:
        tumor_overlap = 0
        tumor_score = 1.0
    
    # M3+M4: Connected components
    labeled, n_comp = ndi_label(vessel_mask)
    if n_comp == 0:
        conn_score = 0
        size_score = 0
        mean_size = 0
    else:
        comp_sizes = np.bincount(labeled.ravel())[1:]  # skip background
        mean_size = comp_sizes.mean()
        
        # Fewer larger components is better
        # Expected ~50-200 components for vessel tree
        if 20 <= n_comp <= 300:
            conn_score = 1.0
        elif n_comp > 1000:
            conn_score = 0.1  # too fragmented
        else:
            conn_score = max(0, 1.0 - abs(n_comp - 150) / 500)
        
        # Mean component size (want > 50 voxels typically)
        if mean_size >= 100:
            size_score = 1.0
        elif mean_size >= 30:
            size_score = 0.7
        elif mean_size >= 10:
            size_score = 0.4
        else:
            size_score = 0.1
    
    total = 0.3 * pct_score + 0.3 * tumor_score + 0.2 * conn_score + 0.2 * size_score
    
    return {
        'pct_score': pct_score, 'tumor_score': tumor_score,
        'conn_score': conn_score, 'size_score': size_score,
        'total': total, 'n_vessel': int(n_vessel), 'pct': float(pct),
        'tumor_overlap': float(tumor_overlap), 'n_components': int(n_comp),
        'mean_size': float(mean_size)
    }

# ============================================================
# Registry
# ============================================================
FILTERS = {
    'frangi': filter_frangi,
    'sato': filter_sato,
    'meijering': filter_meijering,
    'hessian': filter_hessian,
}

PREPROCESSORS = {
    'none': preprocess_none,
    'percentile': preprocess_percentile,
    'clahe': preprocess_clahe,
}

THRESHOLDS = {
    'pct85': lambda r, l: thresh_percentile(r, l, 85),
    'otsu': lambda r, l: thresh_otsu(r, l),
    'adaptive': lambda r, l: thresh_adaptive(r, l),
}

# ============================================================
# Test volumes
# ============================================================
test_volumes = [
    ("imagesTr", "volume-79.nii.gz", "labelsTr", "segmentation-79.nii.gz"),
    ("imagesTs", "volume-111.nii.gz", "labelsTs", "segmentation-111.nii.gz"),
]

# Preload volumes
data = {}
for img_dir, img_file, lbl_dir, lbl_file in test_volumes:
    ct = nib.load(f"{V2_ROOT}/{img_dir}/{img_file}").get_fdata().astype(np.float32)
    lbl = nib.load(f"{V2_ROOT}/{lbl_dir}/{lbl_file}").get_fdata().astype(np.uint8)
    vol_key = img_file.replace('.nii.gz', '')
    data[vol_key] = {
        'ct': ct, 'lbl': lbl,
        'liver_mask': (lbl >= 1), 'tumor_mask': (lbl == 2)
    }
    print(f"✅ Loaded {vol_key}: liver={data[vol_key]['liver_mask'].sum():,}, "
          f"tumor={data[vol_key]['tumor_mask'].sum():,}")

# ============================================================
# GRID SEARCH
# ============================================================
print(f"\n{'='*80}")
print(f"🔬 GRID SEARCH: {len(FILTERS)} filters × {len(PREPROCESSORS)} preprocs × "
      f"{len(THRESHOLDS)} thresh = {len(FILTERS)*len(PREPROCESSORS)*len(THRESHOLDS)} combos")
print(f"   × {len(data)} volumes = {len(FILTERS)*len(PREPROCESSORS)*len(THRESHOLDS)*len(data)} runs")
print(f"{'='*80}")

results = []
grid_start = time.time()

# For each preproc + filter combo, compute response once (then vary thresholds)
for vol_key, d in data.items():
    print(f"\n📊 Volume: {vol_key}")
    for prep_name, prep_fn in PREPROCESSORS.items():
        print(f"   Preproc: {prep_name}")
        preprocessed = prep_fn(d['ct'], d['liver_mask'])
        
        for filt_name, filt_fn in FILTERS.items():
            t0 = time.time()
            try:
                response = filt_fn(preprocessed)
                response = response * d['liver_mask']  # zero outside liver
                filt_time = time.time() - t0
                
                for thresh_name, thresh_fn in THRESHOLDS.items():
                    thresh = thresh_fn(response, d['liver_mask'])
                    vessel_mask = ((response > thresh) & d['liver_mask'])
                    
                    scores = score_extraction(vessel_mask, d['liver_mask'], d['tumor_mask'])
                    
                    result = {
                        'volume': vol_key,
                        'preproc': prep_name,
                        'filter': filt_name,
                        'threshold': thresh_name,
                        'combo': f"{prep_name}+{filt_name}+{thresh_name}",
                        'thresh_value': thresh,
                        'filt_time': filt_time,
                        **scores
                    }
                    results.append(result)
                
                print(f"      {filt_name}: response max={response.max():.4f}, "
                      f"filt_time={filt_time:.1f}s")
            except Exception as e:
                print(f"      ❌ {filt_name} failed: {e}")

grid_time = time.time() - grid_start
print(f"\n⏱️ Grid search done in {grid_time/60:.1f} minutes")

# ============================================================
# ANALYSIS
# ============================================================
df = pd.DataFrame(results)

# Aggregate across volumes (mean scores per combo)
combo_scores = df.groupby('combo').agg({
    'total': 'mean',
    'pct_score': 'mean', 
    'tumor_score': 'mean',
    'conn_score': 'mean',
    'size_score': 'mean',
    'pct': 'mean',
    'tumor_overlap': 'mean',
    'n_components': 'mean',
    'mean_size': 'mean',
    'filt_time': 'mean',
}).sort_values('total', ascending=False)

print(f"\n{'='*80}")
print(f"🏆 TOP 10 COMBOS (ranked by total score)")
print(f"{'='*80}")
print(combo_scores.head(10).to_string())

# Save results
df.to_csv(f"{OUT_DIR}/all_combinations_raw.csv", index=False)
combo_scores.to_csv(f"{OUT_DIR}/combo_rankings.csv")
print(f"\n💾 Saved: {OUT_DIR}/combo_rankings.csv")
print(f"💾 Saved: {OUT_DIR}/all_combinations_raw.csv")

# ============================================================
# VISUALIZE TOP 5 COMBOS
# ============================================================
print(f"\n🖼️ Generating visualizations for top 5 combos...")

top_5 = combo_scores.head(5).index.tolist()
print(f"Top 5: {top_5}")

for combo_name in top_5:
    parts = combo_name.split('+')
    prep_name, filt_name, thresh_name = parts[0], parts[1], parts[2]
    
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    
    for row, (vol_key, d) in enumerate(data.items()):
        preprocessed = PREPROCESSORS[prep_name](d['ct'], d['liver_mask'])
        response = FILTERS[filt_name](preprocessed) * d['liver_mask']
        thresh = THRESHOLDS[thresh_name](response, d['liver_mask'])
        vessel_mask = ((response > thresh) & d['liver_mask'])
        
        liver_z = np.where(np.any(d['liver_mask'], axis=(0,1)))[0]
        mid_z = liver_z[len(liver_z)//2]
        
        # CT + labels
        axes[row, 0].imshow(d['ct'][:,:,mid_z], cmap='gray')
        lm = d['liver_mask'][:,:,mid_z].astype(float)
        tm = d['tumor_mask'][:,:,mid_z].astype(float)
        axes[row, 0].imshow(np.ma.masked_where(lm==0, lm), cmap='Greens', alpha=0.3)
        axes[row, 0].imshow(np.ma.masked_where(tm==0, tm), cmap='Reds', alpha=0.8)
        axes[row, 0].set_title(f'{vol_key}: CT + labels')
        axes[row, 0].axis('off')
        
        # Response
        vr = response[:,:,mid_z]
        axes[row, 1].imshow(preprocessed[:,:,mid_z], cmap='gray')
        if vr.max() > 0:
            axes[row, 1].imshow(np.ma.masked_where(vr < vr.max()*0.1, vr), 
                              cmap='hot', alpha=0.7)
        axes[row, 1].set_title(f'{filt_name} response')
        axes[row, 1].axis('off')
        
        # Vessel mask + tumor
        axes[row, 2].imshow(d['ct'][:,:,mid_z], cmap='gray', alpha=0.6)
        vm = vessel_mask[:,:,mid_z].astype(float)
        axes[row, 2].imshow(np.ma.masked_where(vm==0, vm), cmap='cool', alpha=0.7)
        axes[row, 2].imshow(np.ma.masked_where(tm==0, tm), cmap='Reds', alpha=0.7)
        pct = 100 * vessel_mask.sum() / d['liver_mask'].sum()
        axes[row, 2].set_title(f'Vessels (cyan) {pct:.1f}% + tumor (red)')
        axes[row, 2].axis('off')
    
    plt.suptitle(f'Combo: {combo_name} | Total score: {combo_scores.loc[combo_name, "total"]:.3f}', 
                fontsize=14)
    plt.tight_layout()
    safe_name = combo_name.replace('+', '_')
    plt.savefig(f"{OUT_DIR}/rank_{top_5.index(combo_name)+1}_{safe_name}.png", 
               dpi=80, bbox_inches='tight')
    plt.close()

print(f"\n✅ COMPARATIVE STUDY COMPLETE")
print(f"\n📁 Files at {OUT_DIR}/")
print(f"   - combo_rankings.csv (ranked list)")
print(f"   - all_combinations_raw.csv (per-volume raw)")
print(f"   - rank_1_XX.png ... rank_5_XX.png (top 5 visualizations)")

🧪 COMPARATIVE VESSEL EXTRACTION STUDY
✅ Loaded volume-79: liver=843,134, tumor=11,816
✅ Loaded volume-111: liver=1,502,585, tumor=2,394

🔬 GRID SEARCH: 4 filters × 3 preprocs × 3 thresh = 36 combos
   × 2 volumes = 72 runs

📊 Volume: volume-79
   Preproc: none
      frangi: response max=0.5475, filt_time=43.0s
      sato: response max=0.1419, filt_time=41.9s
      meijering: response max=1.0000, filt_time=43.1s
      hessian: response max=1.0000, filt_time=43.4s
   Preproc: percentile
      frangi: response max=0.5535, filt_time=43.1s
      sato: response max=0.1927, filt_time=41.8s
      meijering: response max=1.0000, filt_time=42.3s
      hessian: response max=1.0000, filt_time=43.6s
   Preproc: clahe
      frangi: response max=0.6762, filt_time=43.4s
      sato: response max=0.2076, filt_time=42.5s
      meijering: response max=1.0000, filt_time=42.6s
      hessian: response max=1.0000, filt_time=43.4s

📊 Volume: volume-111
   Preproc: none
      frangi: response max=0.4144, filt_t

KeyboardInterrupt: 

In [47]:
"""
=========================================================================
PRE-DOWNLOAD: Check disk space + search for existing LiTS raw data
=========================================================================
"""
import os
import subprocess

print("=" * 80)
print("💾 DISK SPACE CHECK")
print("=" * 80)

# Check disk space
result = subprocess.run(['df', '-h', '/workspace'], capture_output=True, text=True)
print(result.stdout)

result = subprocess.run(['df', '-h', '/'], capture_output=True, text=True)
print(result.stdout)

print("\n" + "=" * 80)
print("🔍 SEARCH: Existing LiTS raw data anywhere on server")
print("=" * 80)

search_paths = [
    "/workspace",
    "/data",
    "/mnt",
    "/home",
    "/opt",
    "/scratch",
    "/tmp",
]

print("\nSearching for LiTS-related data (may take 1-2 min)...")
for path in search_paths:
    if not os.path.exists(path):
        continue
    print(f"\n📁 Searching {path}:")
    try:
        result = subprocess.run(
            ['find', path, '-maxdepth', '5', '-type', 'f', 
             '(', '-name', '*LiTS*', '-o', '-name', '*LITS*', 
             '-o', '-name', 'volume-*.nii*', ')',
             '-not', '-path', '*/proc/*',
             '-not', '-path', '*/sys/*'],
            capture_output=True, text=True, timeout=60
        )
        if result.stdout:
            all_lines = result.stdout.strip().split('\n')  # split OUTSIDE f-string
            lines_to_show = all_lines[:30]
            for line in lines_to_show:
                print(f"   {line}")
            if len(all_lines) > 30:
                remaining = len(all_lines) - 30
                print(f"   ... and {remaining} more files")
        else:
            print(f"   (nothing found)")
    except subprocess.TimeoutExpired:
        print(f"   (search timeout)")
    except Exception as e:
        print(f"   (error: {e})")

# Check nnUNet_raw once more
print("\n" + "=" * 80)
print("🔬 QUICK HU CHECK: Sample nii.gz files in nnUNet_raw")
print("=" * 80)

nnunet_raw = "/workspace/liver_tumor_project/models/nnunet_v2/nnUNet_raw/Dataset001_LiTS/imagesTr"
if os.path.exists(nnunet_raw):
    import nibabel as nib
    files = sorted(os.listdir(nnunet_raw))
    if len(files) >= 3:
        check_files = [files[0], files[len(files)//2], files[-1]]
    else:
        check_files = files
    print(f"\nSampling {len(check_files)} files:")
    for f in check_files:
        try:
            nii = nib.load(f"{nnunet_raw}/{f}")
            data = nii.get_fdata()
            has_hu = data.min() < -100
            hu_status = '✅ HU' if has_hu else '❌ normalized'
            print(f"   {f}: range [{data.min():.1f}, {data.max():.1f}] {hu_status}")
        except Exception as e:
            print(f"   {f}: error - {e}")

# Network check
print("\n" + "=" * 80)
print("🌐 NETWORK CHECK")
print("=" * 80)
domains_to_check = [
    "www.kaggle.com",
    "competitions.codalab.org",
    "academictorrents.com",
    "download.pytorch.org",
]
for domain in domains_to_check:
    try:
        result = subprocess.run(
            ['curl', '-sI', '-m', '5', '-o', '/dev/null', '-w', '%{http_code}', 
             f'https://{domain}'],
            capture_output=True, text=True, timeout=10
        )
        code = result.stdout.strip()
        if code in ['200', '301', '302', '403']:
            status = "✅ reachable"
        else:
            status = f"⚠️ HTTP {code}"
        print(f"   {domain}: {status}")
    except Exception:
        print(f"   {domain}: ❌ blocked/timeout")

print("\n" + "=" * 80)
print("✅ Diagnostic complete")
print("=" * 80)

💾 DISK SPACE CHECK
Filesystem                    Size  Used Avail Use% Mounted on
172.16.225.216:/gaurav_goyal  501G  277G  224G  56% /workspace

Filesystem      Size  Used Avail Use% Mounted on
overlay         1.8T  934G  733G  57% /


🔍 SEARCH: Existing LiTS raw data anywhere on server

Searching for LiTS-related data (may take 1-2 min)...

📁 Searching /workspace:
   (search timeout)

📁 Searching /mnt:
   (nothing found)

📁 Searching /home:
   (nothing found)

📁 Searching /opt:
   (nothing found)

📁 Searching /tmp:
   (nothing found)

🔬 QUICK HU CHECK: Sample nii.gz files in nnUNet_raw

Sampling 3 files:
   volume-0_0000.nii.gz: range [0.0, 1.0] ❌ normalized
   volume-4_0000.nii.gz: range [0.0, 1.0] ❌ normalized
   volume-9_0000.nii.gz: range [0.0, 1.0] ❌ normalized

🌐 NETWORK CHECK
   www.kaggle.com: ⚠️ HTTP 404
   competitions.codalab.org: ✅ reachable
   academictorrents.com: ✅ reachable
   download.pytorch.org: ✅ reachable

✅ Diagnostic complete


In [50]:
"""
Setup Kaggle credentials on server
=========================================
INSTRUCTIONS:
1. Open kaggle.json file you downloaded (opens in any text editor)
2. Copy the "username" value → paste below in USERNAME
3. Copy the "key" value → paste below in KEY
4. Run this cell
=========================================
"""
import os
import json

# ⬇️⬇️⬇️ PASTE YOUR CREDENTIALS HERE ⬇️⬇️⬇️
USERNAME = "anuragtiwari575"
KEY = "KGAT_5d107dfd5a911d2319feffcfd644f9bd"
# ⬆️⬆️⬆️ REPLACE ABOVE VALUES ⬆️⬆️⬆️

# Sanity check
if "YOUR_KAGGLE" in USERNAME or "YOUR_KAGGLE" in KEY:
    print("❌ Please replace USERNAME and KEY with your actual credentials from kaggle.json")
    raise SystemExit("Credentials not filled in")

# Create ~/.kaggle directory
kaggle_dir = '/root/.kaggle'
os.makedirs(kaggle_dir, exist_ok=True)

# Write credentials
kaggle_json = f'{kaggle_dir}/kaggle.json'
creds = {'username': USERNAME, 'key': KEY}
with open(kaggle_json, 'w') as f:
    json.dump(creds, f)

# Fix permissions (kaggle CLI requires 600)
os.chmod(kaggle_json, 0o600)

print(f"✅ Credentials saved to {kaggle_json}")
print(f"✅ Permissions set to 600")

# Install kaggle CLI
import subprocess
print(f"\n📦 Installing kaggle CLI...")
result = subprocess.run(['pip', 'install', '-q', 'kaggle'], 
                       capture_output=True, text=True)
print(f"   {'✅ Installed' if result.returncode == 0 else '❌ Failed: ' + result.stderr[:200]}")

# Test API
print(f"\n🧪 Testing Kaggle API...")
result = subprocess.run(['kaggle', '--version'], capture_output=True, text=True)
print(f"   Kaggle CLI: {result.stdout.strip()}")

# Try to list something to verify auth works
result = subprocess.run(
    ['kaggle', 'datasets', 'files', '-d', 'javariatahir/litstrain-val'],
    capture_output=True, text=True, timeout=30
)

if result.returncode == 0:
    print(f"   ✅ API authentication works!")
    print(f"\n📋 Dataset file structure (first 30 lines):")
    lines = result.stdout.split('\n')[:30]
    for line in lines:
        print(f"   {line}")
    print(f"\n(Total lines in output: {len(result.stdout.split(chr(10)))})")
else:
    print(f"   ❌ API test failed:")
    print(f"   {result.stderr[:500]}")
    print(f"\n   Possible issues:")
    print(f"   - Wrong username or key")
    print(f"   - Need to accept dataset rules on Kaggle website first")
    print(f"   - Network/firewall issue")

print("\n" + "=" * 80)
print("Next: If ✅ above, run the download cell")
print("=" * 80)

✅ Credentials saved to /root/.kaggle/kaggle.json
✅ Permissions set to 600

📦 Installing kaggle CLI...
   ✅ Installed

🧪 Testing Kaggle API...
   Kaggle CLI: Kaggle API 1.7.4.5
   ✅ API authentication works!

📋 Dataset file structure (first 30 lines):
   Next Page Token = CfDJ8GNxPX3qpaJEpxfoBSRBZPfpedU2IpAh0-GXElFAtR1N7_S906DwaufvtJGAL2gi7dUs4iqb1_dnpQFhY49FWvOBv6FOBZhCvciUAYU9Djy-nYZtx0FL7kQYQ58aXYBRJlI7_TK5imZJ_EfVneiCBShIyx-QuBYnUdkBAxNGxke6aG01sihL
   name                                          size  creationDate                
   ---------------------------------------  ---------  --------------------------  
   LiTS(train_test)/test_CT/volume-111.nii  398983520  2023-01-27 14:00:46.895000  
   LiTS(train_test)/test_CT/volume-112.nii  393740640  2023-01-27 14:00:52         
   LiTS(train_test)/test_CT/volume-113.nii  438305120  2023-01-27 14:00:57.591000  
   LiTS(train_test)/test_CT/volume-114.nii  443548000  2023-01-27 14:01:02.477000  
   LiTS(train_test)/test_CT/volume-115.

In [ ]:
"""
=========================================================================
FULL DATASET DOWNLOAD (bypass parens bug)
=========================================================================
Downloads entire 54 GB litstrain-val dataset with progress
Auto-extracts and verifies HU values
=========================================================================
"""
import os
import subprocess
import time
import shutil
import glob

DATASET = "javariatahir/litstrain-val"
DOWNLOAD_DIR = "/workspace/liver_tumor_project/data/lits17_raw"

# Clean any partial downloads
test_dir = "/workspace/liver_tumor_project/data/kaggle_test"
if os.path.exists(test_dir):
    shutil.rmtree(test_dir)
    print(f"🧹 Cleaned {test_dir}")

os.makedirs(DOWNLOAD_DIR, exist_ok=True)

# Check disk space
result = subprocess.run(['df', '-h', DOWNLOAD_DIR], capture_output=True, text=True)
print("💾 Disk space:")
print(result.stdout)

print("=" * 80)
print(f"📥 FULL DOWNLOAD: {DATASET}")
print(f"   Target: {DOWNLOAD_DIR}")
print(f"   Size: ~54 GB")
print(f"   Est time: 20-90 minutes")
print("=" * 80)
print(f"\nStarting download at {time.strftime('%H:%M:%S')}...")
print(f"You will see live progress from Kaggle CLI\n")

start = time.time()

# Use Popen for real-time output streaming
cmd = ['kaggle', 'datasets', 'download',
       '-d', DATASET,
       '-p', DOWNLOAD_DIR,
       '--unzip']

process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          text=True, bufsize=1, universal_newlines=True)

try:
    for line in process.stdout:
        print(line, end='', flush=True)
    process.wait()
    elapsed = time.time() - start
    
    if process.returncode == 0:
        print(f"\n\n{'='*80}")
        print(f"✅ DOWNLOAD COMPLETE in {elapsed/60:.1f} minutes")
        print(f"{'='*80}")
    else:
        print(f"\n\n❌ Download failed with code {process.returncode}")
        raise SystemExit("Download failed - check error above")
        
except KeyboardInterrupt:
    process.kill()
    print("\n⚠️ Interrupted by user")
    raise

# ============================================================
# VERIFY
# ============================================================
print("\n" + "=" * 80)
print("📁 DOWNLOAD VERIFICATION")
print("=" * 80)

# Show structure
print(f"\nDirectory structure:")
for root, dirs, files in os.walk(DOWNLOAD_DIR):
    level = root.replace(DOWNLOAD_DIR, '').count(os.sep)
    if level <= 2:
        indent = ' ' * 3 * level
        print(f"{indent}📁 {os.path.basename(root)}/")
        subindent = ' ' * 3 * (level + 1)
        # Just show first 3 files at each level
        for f in files[:3]:
            fpath = os.path.join(root, f)
            size_mb = os.path.getsize(fpath) / 1024**2
            print(f"{subindent}📄 {f} ({size_mb:.1f} MB)")
        if len(files) > 3:
            print(f"{subindent}... and {len(files) - 3} more files")

# Count all nii files
all_nii = glob.glob(f"{DOWNLOAD_DIR}/**/*.nii*", recursive=True)
print(f"\n📊 Total .nii/.nii.gz files: {len(all_nii)}")

# Check HU on 3 sample files
if all_nii:
    import nibabel as nib
    samples = [all_nii[0], all_nii[len(all_nii)//2], all_nii[-1]]
    print(f"\n🔬 HU verification on {len(samples)} samples:")
    hu_confirmed = False
    for f in samples:
        try:
            nii = nib.load(f)
            data = nii.get_fdata()
            print(f"\n   {f}")
            print(f"   Shape: {data.shape}")
            print(f"   Range: [{data.min():.1f}, {data.max():.1f}]")
            
            if data.min() < -500:
                print(f"   ✅ RAW HU DATA")
                hu_confirmed = True
            elif 0 <= data.min() <= 3 and data.max() <= 3:
                print(f"   ❌ Labels (0/1/2)")
            else:
                print(f"   ⚠️ Unusual range")
        except Exception as e:
            print(f"   Error: {e}")
    
    print(f"\n{'='*80}")
    if hu_confirmed:
        print(f"🎉 SUCCESS - Ready for vessel extraction with real HU values")
    else:
        print(f"⚠️ HU not confirmed in samples - inspect manually")
    print(f"{'='*80}")

# Total downloaded size
total_size = sum(os.path.getsize(f) for f in all_nii)
print(f"\n💾 Total downloaded: {total_size/1024**3:.2f} GB in {len(all_nii)} files")
print(f"⏱️ Total time: {elapsed/60:.1f} minutes")
print(f"⚡ Speed: {(total_size/1024**2)/elapsed:.1f} MB/s")

🧹 Cleaned /workspace/liver_tumor_project/data/kaggle_test
💾 Disk space:
Filesystem                    Size  Used Avail Use% Mounted on
172.16.225.216:/gaurav_goyal  501G  262G  239G  53% /workspace

📥 FULL DOWNLOAD: javariatahir/litstrain-val
   Target: /workspace/liver_tumor_project/data/lits17_raw
   Size: ~54 GB
   Est time: 20-90 minutes

Starting download at 06:11:30...
You will see live progress from Kaggle CLI

Dataset URL: https://www.kaggle.com/datasets/javariatahir/litstrain-val
License(s): unknown

  0%|                                              | 0.00/15.6G [00:00<?, ?B/s]
  0%|                                      | 28.0M/15.6G [00:00<00:57, 290MB/s]
  0%|▏                                     | 57.0M/15.6G [00:00<00:56, 298MB/s]
  1%|▏                                     | 87.0M/15.6G [00:00<00:55, 301MB/s]
  1%|▎                                      | 118M/15.6G [00:00<00:53, 309MB/s]
  1%|▎                                      | 149M/15.6G [00:00<00:53, 312MB/s]
  1%|

 17%|██████▍                               | 2.65G/15.6G [00:09<00:48, 285MB/s]
 17%|██████▌                               | 2.68G/15.6G [00:09<00:48, 285MB/s]
 17%|██████▌                               | 2.70G/15.6G [00:09<00:48, 284MB/s]
 17%|██████▋                               | 2.73G/15.6G [00:09<00:48, 283MB/s]
 18%|██████▌                              | 2.76G/15.6G [00:11<03:36, 63.8MB/s]
 18%|██████▌                              | 2.79G/15.6G [00:11<02:45, 83.3MB/s]
 18%|██████▊                               | 2.81G/15.6G [00:11<02:08, 107MB/s]
 18%|██████▉                               | 2.84G/15.6G [00:11<01:43, 133MB/s]
 18%|██████▉                               | 2.87G/15.6G [00:11<01:26, 159MB/s]
 19%|███████                               | 2.90G/15.6G [00:11<01:14, 184MB/s]
 19%|███████                               | 2.92G/15.6G [00:11<01:05, 208MB/s]
 19%|███████▏                              | 2.95G/15.6G [00:11<00:59, 229MB/s]
 19%|███████▏                           

 35%|█████████████▏                        | 5.45G/15.6G [00:22<00:37, 294MB/s]
 35%|█████████████▎                        | 5.48G/15.6G [00:22<00:38, 284MB/s]
 35%|█████████████▍                        | 5.51G/15.6G [00:22<00:38, 283MB/s]
 35%|█████████████▍                        | 5.53G/15.6G [00:22<00:37, 286MB/s]
 36%|█████████████▌                        | 5.56G/15.6G [00:22<00:37, 286MB/s]
 36%|█████████████▌                        | 5.59G/15.6G [00:23<00:37, 288MB/s]
 36%|█████████████▋                        | 5.62G/15.6G [00:23<00:37, 287MB/s]
 36%|█████████████▋                        | 5.65G/15.6G [00:23<00:36, 291MB/s]
 36%|█████████████▊                        | 5.67G/15.6G [00:23<00:37, 287MB/s]
 36%|█████████████▊                        | 5.70G/15.6G [00:23<00:37, 285MB/s]
 37%|█████████████▉                        | 5.73G/15.6G [00:23<00:37, 284MB/s]
 37%|█████████████▉                        | 5.75G/15.6G [00:23<00:37, 285MB/s]
 37%|██████████████                     

 53%|████████████████████                  | 8.25G/15.6G [00:34<00:36, 218MB/s]
 53%|████████████████████                  | 8.27G/15.6G [00:34<00:37, 213MB/s]
 53%|████████████████████▏                 | 8.29G/15.6G [00:34<00:34, 231MB/s]
 53%|████████████████████▏                 | 8.32G/15.6G [00:35<00:30, 255MB/s]
 53%|████████████████████▎                 | 8.35G/15.6G [00:35<00:28, 271MB/s]
 54%|████████████████████▎                 | 8.38G/15.6G [00:35<00:27, 279MB/s]
 54%|████████████████████▍                 | 8.41G/15.6G [00:35<00:27, 283MB/s]
 54%|████████████████████▍                 | 8.44G/15.6G [00:35<00:26, 287MB/s]
 54%|████████████████████▌                 | 8.46G/15.6G [00:35<00:26, 290MB/s]
 54%|████████████████████▋                 | 8.49G/15.6G [00:35<00:26, 293MB/s]
 54%|████████████████████▋                 | 8.52G/15.6G [00:35<00:26, 292MB/s]
 55%|████████████████████▊                 | 8.55G/15.6G [00:35<00:25, 299MB/s]
 55%|████████████████████▊              

 71%|███████████████████████████           | 11.2G/15.6G [00:47<00:15, 306MB/s]
 71%|███████████████████████████▏          | 11.2G/15.6G [00:47<00:15, 305MB/s]
 72%|███████████████████████████▏          | 11.2G/15.6G [00:48<00:15, 304MB/s]
 72%|███████████████████████████▎          | 11.2G/15.6G [00:48<00:15, 304MB/s]
 72%|███████████████████████████▍          | 11.3G/15.6G [00:48<00:15, 305MB/s]
 72%|███████████████████████████▍          | 11.3G/15.6G [00:48<00:15, 310MB/s]
 72%|███████████████████████████▌          | 11.3G/15.6G [00:48<00:14, 312MB/s]
 73%|███████████████████████████▌          | 11.4G/15.6G [00:48<00:14, 308MB/s]
 73%|███████████████████████████▋          | 11.4G/15.6G [00:48<00:14, 306MB/s]
 73%|███████████████████████████▋          | 11.4G/15.6G [00:48<00:14, 305MB/s]
 73%|███████████████████████████▊          | 11.4G/15.6G [00:48<00:15, 297MB/s]
 73%|███████████████████████████▉          | 11.5G/15.6G [00:49<00:15, 298MB/s]
 74%|███████████████████████████▉       

 91%|██████████████████████████████████▍   | 14.2G/15.6G [00:59<00:09, 167MB/s]
 91%|██████████████████████████████████▌   | 14.2G/15.6G [00:59<00:08, 193MB/s]
 91%|██████████████████████████████████▌   | 14.2G/15.6G [00:59<00:06, 220MB/s]
 91%|██████████████████████████████████▋   | 14.3G/15.6G [00:59<00:06, 243MB/s]
 91%|██████████████████████████████████▋   | 14.3G/15.6G [00:59<00:05, 265MB/s]
 92%|██████████████████████████████████▊   | 14.3G/15.6G [01:00<00:05, 281MB/s]
 92%|██████████████████████████████████▊   | 14.4G/15.6G [01:00<00:04, 289MB/s]
 92%|██████████████████████████████████▉   | 14.4G/15.6G [01:00<00:04, 296MB/s]
 92%|███████████████████████████████████   | 14.4G/15.6G [01:00<00:04, 301MB/s]
 92%|███████████████████████████████████   | 14.4G/15.6G [01:00<00:04, 302MB/s]
 93%|███████████████████████████████████▏  | 14.5G/15.6G [01:00<00:04, 305MB/s]
 93%|███████████████████████████████████▏  | 14.5G/15.6G [01:00<00:03, 311MB/s]
 93%|███████████████████████████████████

In [54]:
"""
=========================================================================
PROTECT RAW DATA - Make read-only
=========================================================================
"""
import os
import subprocess
import time

RAW_DIR = "/workspace/liver_tumor_project/data/lits17_raw"

print("=" * 80)
print("🔒 PROTECTING RAW DATA")
print("=" * 80)

# Show current size
result = subprocess.run(['du', '-sh', RAW_DIR], capture_output=True, text=True)
print(f"\n📊 Size: {result.stdout.strip()}")

# Create README first (before locking)
readme_path = f"{RAW_DIR}/README_DO_NOT_DELETE.txt"
with open(readme_path, 'w') as f:
    f.write(f"""
================================================================================
⚠️  DO NOT DELETE THIS FOLDER ⚠️
================================================================================

Raw LiTS17 dataset with HU values.

Downloaded: {time.strftime('%Y-%m-%d %H:%M:%S')}
Size: ~50 GB
Source: kaggle.com/datasets/javariatahir/litstrain-val

STATUS: READ-ONLY

Contents:
- LiTS(train_test)/train_CT/    (111 CT volumes, raw HU)
- LiTS(train_test)/train_mask/  (111 segmentation masks)
- LiTS(train_test)/test_CT/     (20 CT volumes, raw HU)
- LiTS(train_test)/test_mask/   (20 segmentation masks)

WHY IT MATTERS:
- Used for vessel extraction (novel VasTA-Net method)
- Source of truth for all preprocessing
- Cannot easily re-download (Kaggle CLI single-file bug)
- Paper reproducibility requires original HU values

TO MODIFY (only if absolutely needed):
   chmod -R u+w {RAW_DIR}
================================================================================
""")

print(f"📝 README created")

# Make read-only recursively
print(f"\n🔒 Setting read-only permissions...")
subprocess.run(['chmod', '-R', 'a-w', RAW_DIR], capture_output=True)
print(f"   ✅ Read-only applied")

# Verify
import stat
mode = os.stat(RAW_DIR).st_mode
is_writable = bool(mode & stat.S_IWUSR)
print(f"   Directory writable? {is_writable}")

print("\n" + "=" * 80)
print("✅ PROTECTED - data is safe from accidental deletion")
print("=" * 80)

🔒 PROTECTING RAW DATA

📊 Size: 51G	/workspace/liver_tumor_project/data/lits17_raw
📝 README created

🔒 Setting read-only permissions...
   ✅ Read-only applied
   Directory writable? False

✅ PROTECTED - data is safe from accidental deletion


In [55]:
"""
=========================================================================
Vessel Extraction v4: On REAL HU data with proper CT windowing
=========================================================================
Test all 3 filters on 2 volumes with proper HU-based preprocessing
=========================================================================
"""
import os
import time
import numpy as np
import nibabel as nib
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from skimage.filters import frangi, sato, meijering
from scipy.ndimage import gaussian_filter

print("=" * 80)
print("🩸 VESSEL EXTRACTION v4: REAL HU DATA")
print("=" * 80)

# Paths
RAW_DIR = "/workspace/liver_tumor_project/data/lits17_raw/LiTS(train_test)"
# We have LABELS in preprocessed folder (matches original volumes)
LABEL_DIR_TR = "/workspace/liver_tumor_project/data/preprocessed/labelsTr"
LABEL_DIR_TS = "/workspace/liver_tumor_project/data/preprocessed/labelsTs"
OUT_DIR = "/workspace/liver_tumor_project/novel_method/phase1_hu_vessels"
os.makedirs(OUT_DIR, exist_ok=True)

# Test volumes - use raw HU CT + raw labels from raw folder (matching resolution)
# The raw masks are in raw_dir - but sizes may differ from preprocessed CT
# Use RAW mask to match RAW CT resolution
test_volumes = [
    ("train_CT", "volume-79.nii", "train_mask", "segmentation-79.nii"),
    ("test_CT", "volume-111.nii", "test_mask", "segmentation-111.nii"),
]

# Standard liver window for HU CT
# HU: air=-1000, water=0, liver tissue=~50-70, vessels with contrast=~100-200
LIVER_WINDOW_MIN = -100
LIVER_WINDOW_MAX = 200

for i, (ct_dir, ct_file, lbl_dir, lbl_file) in enumerate(test_volumes, 1):
    print(f"\n{'='*80}")
    print(f"📊 Volume {i}: {ct_file}")
    print(f"{'='*80}")
    
    # Load raw CT (HU values)
    ct_path = f"{RAW_DIR}/{ct_dir}/{ct_file}"
    lbl_path = f"{RAW_DIR}/{lbl_dir}/{lbl_file}"
    
    print(f"   Loading CT: {ct_path}")
    ct_nii = nib.load(ct_path)
    ct = ct_nii.get_fdata().astype(np.float32)
    print(f"   CT shape: {ct.shape}")
    print(f"   CT range: [{ct.min():.1f}, {ct.max():.1f}] HU")
    print(f"   CT spacing: {ct_nii.header.get_zooms()}")
    
    print(f"\n   Loading labels: {lbl_path}")
    lbl_nii = nib.load(lbl_path)
    lbl = lbl_nii.get_fdata().astype(np.uint8)
    print(f"   Labels shape: {lbl.shape}")
    print(f"   Labels unique: {np.unique(lbl)}")
    
    # Ensure shapes match
    if ct.shape != lbl.shape:
        print(f"   ⚠️ Shape mismatch! CT {ct.shape} vs Label {lbl.shape}")
        continue
    
    # Liver mask
    liver_mask = (lbl >= 1)
    tumor_mask = (lbl == 2)
    print(f"   Liver voxels: {liver_mask.sum():,}")
    print(f"   Tumor voxels: {tumor_mask.sum():,}")
    
    # ============================================================
    # Preprocessing: Apply liver window + normalize to [0,1]
    # ============================================================
    print(f"\n   🎨 Applying liver window [{LIVER_WINDOW_MIN}, {LIVER_WINDOW_MAX}]...")
    ct_windowed = np.clip(ct, LIVER_WINDOW_MIN, LIVER_WINDOW_MAX)
    ct_norm = (ct_windowed - LIVER_WINDOW_MIN) / (LIVER_WINDOW_MAX - LIVER_WINDOW_MIN)
    ct_liver_only = ct_norm * liver_mask
    
    # Light smoothing to reduce noise
    ct_smooth = gaussian_filter(ct_liver_only, sigma=0.5)
    
    # Sample within liver to see contrast
    liver_intensity = ct_norm[liver_mask]
    print(f"   Within-liver intensity: mean={liver_intensity.mean():.3f}, std={liver_intensity.std():.3f}")
    print(f"   Percentiles: 5%={np.percentile(liver_intensity, 5):.3f}, "
          f"95%={np.percentile(liver_intensity, 95):.3f}")
    
    # ============================================================
    # Run 3 filters
    # ============================================================
    filter_results = {}
    
    for filter_name, filter_fn in [
        ('frangi', lambda x: frangi(x, sigmas=np.arange(1.0, 3.5, 0.5), 
                                    alpha=0.5, beta=0.5, gamma=None, black_ridges=False)),
        ('sato', lambda x: sato(x, sigmas=[1.0, 1.5, 2.0, 2.5, 3.0], black_ridges=False)),
        ('meijering', lambda x: meijering(x, sigmas=[1.0, 1.5, 2.0, 2.5, 3.0], black_ridges=False)),
    ]:
        print(f"\n   🩸 Running {filter_name}...")
        start = time.time()
        response = filter_fn(ct_smooth) * liver_mask
        elapsed = time.time() - start
        
        # Threshold at 85th percentile of positive responses within liver
        r = response[liver_mask]
        r_pos = r[r > 0]
        if len(r_pos) > 0:
            thresh = np.percentile(r_pos, 85)
            vessel_mask = ((response > thresh) & liver_mask).astype(np.uint8)
        else:
            thresh = 0
            vessel_mask = np.zeros_like(ct, dtype=np.uint8)
        
        n_vessel = vessel_mask.sum()
        vessel_pct = 100 * n_vessel / liver_mask.sum()
        
        # Vessel-tumor overlap
        if tumor_mask.sum() > 0:
            vessel_in_tumor = ((vessel_mask == 1) & tumor_mask).sum()
            tumor_overlap = 100 * vessel_in_tumor / tumor_mask.sum()
        else:
            tumor_overlap = 0
        
        print(f"      Time: {elapsed:.1f}s")
        print(f"      Response max: {response.max():.4f}")
        print(f"      Threshold: {thresh:.4f}")
        print(f"      Vessel voxels: {n_vessel:,} ({vessel_pct:.1f}% of liver)")
        print(f"      Vessel-tumor overlap: {tumor_overlap:.1f}%")
        
        filter_results[filter_name] = {
            'response': response,
            'mask': vessel_mask,
            'pct': vessel_pct,
            'overlap': tumor_overlap,
            'time': elapsed
        }
    
    # ============================================================
    # Visualization: All 3 filters side-by-side
    # ============================================================
    print(f"\n   🖼️ Creating visualization...")
    
    liver_z = np.where(np.any(liver_mask, axis=(0, 1)))[0]
    mid_z = liver_z[len(liver_z) // 2]
    
    fig, axes = plt.subplots(2, 4, figsize=(24, 12))
    
    # Row 1: Original + labels + filter responses
    axes[0, 0].imshow(ct[:, :, mid_z], cmap='gray', vmin=LIVER_WINDOW_MIN, vmax=LIVER_WINDOW_MAX)
    axes[0, 0].set_title(f'CT slice z={mid_z} (windowed [-100, 200] HU)')
    axes[0, 0].axis('off')
    
    for col, name in enumerate(['frangi', 'sato', 'meijering'], 1):
        r = filter_results[name]['response']
        axes[0, col].imshow(ct[:, :, mid_z], cmap='gray', 
                          vmin=LIVER_WINDOW_MIN, vmax=LIVER_WINDOW_MAX, alpha=0.5)
        r_slice = r[:, :, mid_z]
        if r_slice.max() > 0:
            axes[0, col].imshow(np.ma.masked_where(r_slice < r_slice.max()*0.1, r_slice), 
                              cmap='hot', alpha=0.7)
        axes[0, col].set_title(f'{name} response (max={r_slice.max():.3f})')
        axes[0, col].axis('off')
    
    # Row 2: Labels + binary vessel masks with tumor
    axes[1, 0].imshow(ct[:, :, mid_z], cmap='gray', 
                     vmin=LIVER_WINDOW_MIN, vmax=LIVER_WINDOW_MAX, alpha=0.5)
    lm = liver_mask[:, :, mid_z].astype(float)
    tm = tumor_mask[:, :, mid_z].astype(float)
    axes[1, 0].imshow(np.ma.masked_where(lm == 0, lm), cmap='Greens', alpha=0.3)
    axes[1, 0].imshow(np.ma.masked_where(tm == 0, tm), cmap='Reds', alpha=0.8)
    axes[1, 0].set_title('Liver (green) + Tumor (red)')
    axes[1, 0].axis('off')
    
    for col, name in enumerate(['frangi', 'sato', 'meijering'], 1):
        vm = filter_results[name]['mask'][:, :, mid_z].astype(float)
        pct = filter_results[name]['pct']
        overlap = filter_results[name]['overlap']
        axes[1, col].imshow(ct[:, :, mid_z], cmap='gray', 
                          vmin=LIVER_WINDOW_MIN, vmax=LIVER_WINDOW_MAX, alpha=0.5)
        axes[1, col].imshow(np.ma.masked_where(vm == 0, vm), cmap='cool', alpha=0.85)
        axes[1, col].imshow(np.ma.masked_where(tm == 0, tm), cmap='Reds', alpha=0.6)
        axes[1, col].set_title(f'{name}: vessels {pct:.1f}%, tumor overlap {overlap:.1f}%')
        axes[1, col].axis('off')
    
    plt.suptitle(f'{ct_file} - HU data vessel extraction comparison', fontsize=14)
    plt.tight_layout()
    
    idx = ct_file.replace('volume-', '').replace('.nii', '')
    viz_path = f"{OUT_DIR}/viz_hu_{idx}.png"
    plt.savefig(viz_path, dpi=80, bbox_inches='tight')
    plt.close()
    print(f"   💾 Saved: {viz_path}")

print("\n" + "=" * 80)
print("✅ v4 COMPLETE - View visualizations")
print("=" * 80)
print(f"""
Display in Jupyter:

from IPython.display import Image, display
display(Image('{OUT_DIR}/viz_hu_79.png'))
display(Image('{OUT_DIR}/viz_hu_111.png'))

Look for TRUE vessel branching structures now!
""")

🩸 VESSEL EXTRACTION v4: REAL HU DATA

📊 Volume 1: volume-79.nii
   Loading CT: /workspace/liver_tumor_project/data/lits17_raw/LiTS(train_test)/train_CT/volume-79.nii
   CT shape: (512, 512, 147)
   CT range: [-1013.0, 1367.0] HU
   CT spacing: (0.796875, 0.796875, 3.0)

   Loading labels: /workspace/liver_tumor_project/data/lits17_raw/LiTS(train_test)/train_mask/segmentation-79.nii
   Labels shape: (512, 512, 147)
   Labels unique: [0 1 2]
   Liver voxels: 660,094
   Tumor voxels: 9,085

   🎨 Applying liver window [-100, 200]...
   Within-liver intensity: mean=0.639, std=0.096
   Percentiles: 5%=0.470, 95%=0.767

   🩸 Running frangi...
      Time: 239.2s
      Response max: 0.4781
      Threshold: 0.0104
      Vessel voxels: 65,170 (9.9% of liver)
      Vessel-tumor overlap: 6.1%

   🩸 Running sato...
      Time: 232.1s
      Response max: 0.1643
      Threshold: 0.0475
      Vessel voxels: 88,638 (13.4% of liver)
      Vessel-tumor overlap: 10.6%

   🩸 Running meijering...
      Time:

In [ ]:
from IPython.display import Image, display

print("Volume 79:")
display(Image('/workspace/liver_tumor_project/novel_method/phase1_hu_vessels/viz_hu_79.png'))

print("\nVolume 111 (test set):")
display(Image('/workspace/liver_tumor_project/novel_method/phase1_hu_vessels/viz_hu_111.png'))

Volume 79:
